In [ ]:
# Run this once at the beginning of your notebook
%load_ext autoreload
%autoreload 2

In [ ]:
# Core scientific computing
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import pickle
import os

# MDAnalysis for trajectory handling
import MDAnalysis as mda
from MDAnalysis.analysis.base import AnalysisBase, Results

# Progress bar
from tqdm import tqdm

# Import your custom class
import sys
sys.path.append('/Users/roev0007/Documents/solvation_shells/my_scripts')
from ZDirectionalAnalysis import ZDirectionalAnalysis
from ZDirectionalPlotter import ZDirectionalPlotter

# # Set matplotlib for inline plotting
# %matplotlib inline
# plt.rcParams['figure.dpi'] = 100
# plt.rcParams['savefig.dpi'] = 300

In [ ]:
## for debugging
# Check total frames first
u = mda.Universe('nvt.tpr', 'nvt.trr')
total_frames = len(u.trajectory)
print(f"Total frames available: {total_frames}")

In [ ]:
# Load the trajectories
z_analysis = ZDirectionalAnalysis(
    top='nvt.tpr',
    traj='nvt.trr',
    water_oxygen='name OW or name Ow',
    water_hydrogen='name HW1 or name HW2 or name Hw1 or name Hw2',  # ✓ Explicit names
    water='resname SOL or resname WAT', 
    cation='(name NA or name Na or name Mg or name MG)',
    anion='name CL',
    start_frame=0,
    end_frame=total_frames,
    step_frame=1,
    usecache=False,
    debug=False
)

In [ ]:

# Run basic analysis
# z_analysis.run(step=1, use_cache=True)
# Only analyze first 100 frames:
# z_analysis.run(start=0, stop=100, step=5)
# # Or analyze a range:
# z_analysis.run(start=500, stop=600, step=1)
plotter = ZDirectionalPlotter(z_analysis)
# z_analysis.run()  # This likely calculates basic density profiles
z_analysis.run(step=1, collect_bootstrap_data=True, force_rerun=False)


In [ ]:
# Interface detection
interfaces = z_analysis.detect_interfaces(
    interface_type='density_gradient',    # or 'water_depletion', 'density_threshold'
    density_threshold=0.3,
    min_interface_width=2.0,
    symmetric_system=True
)

In [ ]:
# First, calculate the Mgo interface boundaries
# clay_boundaries = z_analysis.calculate_clay_interface_boundaries()
clay_boundaries_improved = z_analysis.calculate_clay_interface_boundaries(n_frames_average=100)

In [ ]:
# Plot the results
from ZDirectionalPlotter import ZDirectionalPlotter
plotter = ZDirectionalPlotter(z_analysis)
# plotter.plot_clay_interface_boundaries(save_plots=True)

In [ ]:
# Check what was calculated
print(f"Frames analyzed: {z_analysis.n_frames_analyzed}")
print(f"Ion types found: {list(z_analysis.cation_types.keys())}, {list(z_analysis.anion_types.keys())}")

In [ ]:
# z_analysis.plot_density_profiles(
#     save_plots=True,
#     show_counts=True,                 # Show raw counts
#     show_density=True,                # Show number density
#     figsize=(15, 12)
# )

# # Coordination analysis
# z_analysis.plot_coordination_vs_z(
#     ion_types=['NA', 'MG', 'CL'],          # Specific ions
#     bin_size=2.0,                    # Binning for smooth curves
#     save_plots=True
# )


In [ ]:
plotter.plot_clay_interface_boundaries(
    save_plots=True,
    figsize=(8, 5),
    dpi=600,
    title_fontsize=16,
    show_title=False, # 
    label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=12,
    show_legend=False,
    show_grid=False,
        ion_density_colors={
        'CL': 'red',
        'NA': 'black',
        'MG': 'green',
        'CA': 'yellow'
    },
    grid_alpha=0.3,
    upper_clay_avg_color='gray', # clay color
    upper_si_color='orange',
    upper_mgo_color='green',
    lower_clay_avg_color='gray', # clay color
    lower_si_color='orange',
    lower_mgo_color='green',
    upper_clay_avg_linestyle='-',
    upper_si_linestyle='--',
    upper_mgo_linestyle='--', # changed from ':'
    lower_clay_avg_linestyle='-',
    lower_si_linestyle='--',
    lower_mgo_linestyle=':',
    show_boundary_lines=True, # 
    boundary_linewidth=1, #2.5,
    boundary_alpha=0.8,
    water_density_color='blue',
    ion_density_color='red',
    water_linewidth=2,
    ion_linewidth=2,
    density_overlay_alpha=0.7,
    fill_clay_region=True,
    clay_fill_color='yellow', #gray
    clay_fill_alpha=0.15,
    fill_density_curves=True, ### may be comment out?
    water_fill_alpha=0.2,
    ion_fill_alpha=0.15,
    water_fill_color=None,  # If None, uses water_density_color
    ion_fill_colors=None,  # Dict for specific ion colors, or None to use line colors
    show_zero_line=True,
    zero_line_color='black',
    zero_line_style=':',
    zero_line_alpha=0.5,
    zero_line_width=1.0,
    legend_location='upper left',
    legend_ncol=2,
    legend_frameon=True,
    legend_framealpha=0.8,
    title_text=None,
    xlabel_text='z (Å)',
    ylabel_text='Water Density (molecules/Å³)',
    ylabel_ion_text='Ion Density (ions/Å³)',
    show_boundary_values=True,
    symmetrize_densities=True,
    symmetrize_boundaries=True,
    symmetrize_method='average',
    print_summary=True
    )

In [ ]:
# z_analysis.calculate_water_dipole_orientation_xy(
#     z_slice_centers=[-24, 0, 24],
#     z_slice_width=1.5,
#     xy_grid_size=0.5,
#     xy_projection=True,
#     projection_method='three_atom', # elliptical, molecular_vdw
#     step=1,
#     orientation_metric='angle_to_z',
#     smoothing_method='three_point', # elliptical, molecular_vdw
#     water_vdw_radius=1.52,
#     # gaussian_sigma=1.0,
#     use_cache=True
# );

In [ ]:
# First, let's run the clay analysis and see the raw grid data
clay_results = z_analysis.calculate_clay_spatial_distribution_xy(
    z_slice_centers=[27.0, -27.0],
    z_slice_width=6.0,
    mg_vdw_scaling=2.5,
    si_vdw_scaling=1.5,
    use_cache=True,
    step=1,
    return_format='grids'  # Force grid format first
)

In [ ]:
# Create custom yellow colormap
import matplotlib.colors as mcolors

colors = ['#FFFFCC', '#FFFF99', '#FFFF66', '#FFFF00', '#CCCC00', '#999900', '#666600']
custom_yellow_cmap = mcolors.LinearSegmentedColormap.from_list('custom_yellow', colors)

plotter.plot_clay_atoms_xy(
    z_slice_centers=[-29.5, 26.5, 29.5],
    clay_results=clay_results,
    show_contour_lines=True,
    contour_line_color='black',
    show_contour_fill=True,
    title_fontsize=16,
    show_title=False,
    label_fontsize=16,
    tick_fontsize=14,
    legend_fontsize=14,
    mg_cmap='Greens',
    si_cmap=custom_yellow_cmap, # si_cmap='Blues', 
    si_vmin=0, si_vmax=360,
    mg_vmin=0, mg_vmax=210,
    colorbar_pad=0.1,  
    colorbar_width='4%', 
    individual_figsize=(8, 6),
    show_individual_figures=False,
    save_individual_figures=True,
    save_combined_figure=False,
    dpi=600,
    save_plots=True
)

In [ ]:
plotter.plot_clay_atoms_xy(
    clay_results=clay_results,
    title_fontsize=16,
    show_title=False,
    label_fontsize=16,
    tick_fontsize=14,
    legend_fontsize=14,
    mg_cmap='Greens',
    si_cmap='Blues', 
    colorbar_pad=0.1,  
    colorbar_width='4%', 
    individual_figsize=(8, 6),
    show_individual_figures=False,
    save_individual_figures=True,
    combine_mg_layers=True, 
    dpi=600,
    save_plots=True
)

In [ ]:
# # Water dipole orientation
# z_analysis.plot_dipole_orientation_vs_z(
#     save_plots=True,
#     bin_size=2.0,
#     figsize=(18, 6)
# )

In [ ]:
z_analysis.calculate_water_dipole_orientation_xy(
    z_slice_centers=[-23.43, 0, 24.68],
    z_slice_width=1.5,
    xy_grid_size=0.5,
    xy_projection=True,
    projection_method='three_atom', # elliptical, molecular_vdw
    step=10,
    orientation_metric='angle_to_z',
    smoothing_method='three_point', # elliptical, molecular_vdw
    water_vdw_radius=1.52,
    # gaussian_sigma=1.0,
    use_cache=True
);

In [ ]:
# Water dipole orientation
z_analysis.plot_dipole_orientation_vs_z(
    save_plots=True,
    bin_size=2,
    figsize=(8, 6),
    dpi=600,
    title_fontsize=16,
    show_title=False,
    label_fontsize=22,
    tick_fontsize=18,
    legend_fontsize=12,
    show_legend=False,
    cmap='hot',
    colorbar_pad=0.02,  
    colorbar_width='4%', 
    vmin=None,
    # vmax=170000,
    colorbar_percentile=None,
    use_cache=False,
)

In [ ]:
# Plot
plotter.plot_water_dipole_orientation_xy(
    z_indices='all',
    figsize=(8, 6),
    dpi=600,
    title_fontsize=20,
    filename='water_dipole_xy_angle-to-z_three_point_dpi600',
    show_title=True,
    label_fontsize=18,
    tick_fontsize=16,
    save_plots=True,
    cmap='twilight',  # Good for angles
    show_individual_figures=False,
    save_individual_figures=True,
    save_combined_figure=False,
    vmin=0,
    vmax=180      
)

In [ ]:
# Plot water dipole with clay at CORRECT positions
plotter.plot_water_dipole_orientation_xy(
    z_indices='all',
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        'clay_z_tolerance': 6.0,
        'z_slice_centers': [-24, 0, 24], #[-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,
        'mg_vdw_scaling': 2.0,
        'si_vdw_scaling': 1.5,
        'si_connection_threshold': 4.0,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'show_plots': False,
        'use_cache': False
    },
    figsize=(8, 6),
    dpi=600,
    title_fontsize=16,
    label_fontsize=14,
    tick_fontsize=12,
    show_individual_figures=False,
    save_individual_figures=True,
    save_combined_figure=False,
    show_clay_spatial=True,
    save_plots=True
)

In [ ]:
# Plot water dipole with clay spatial analysis at correct positions
plotter.plot_water_dipole_orientation_xy(
    z_indices='all',
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        # 'clay_z_tolerance': 6.0, # uncomment this if you also want to show the mgo atoms
        'z_slice_centers': [-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,
        'mg_vdw_scaling': 2.0,
        'si_vdw_scaling': 1.5,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'fade_method': 'gaussian',
        'mg_texture_intensity': 0.8,
        'si_connection_threshold': 4.5,
        'si_connection_style': 'lines',  # This should set the default style
        'si_connection_alpha': 0.5,     # Add missing parameters
        'si_connection_linewidth': 1.0,
        'si_connection_zorder': 2,
        'show_hexagonal_pattern': True,
        'mg_glow_effect': True,
        'use_cache': True
    },
    show_clay_spatial=True,
    save_individual_figures=True,
    figsize=(8, 6),
    title_fontsize=18,
    label_fontsize=16,
    tick_fontsize=14,
    dpi=600,
    save_plots=True,
    cmap='twilight',
    vmin_negative=60,
    vmax_positive=120
)

In [ ]:
# Three-atom projection (most physically accurate)
results = z_analysis.calculate_water_spatial_distribution_xy(
    z_slice_centers=[-23.43, 0, 24.68],
    z_slice_width=1.5,
    xy_grid_size=0.5,
    use_xy_projection=True,
    projection_method='three_atom',
    water_vdw_radius=1.52,
    hydrogen_vdw_radius=1.2,
    save_cache=True,
    force_rerun=False,
    step=10 
)

## Methods: Three-Atom Electron-Weighted Disk Projection

The **projected areal water density** $\rho_{\mathrm{proj}}(x_i, y_j;\, z_k)$ at a $xy$-plane centred at $z_k$ is computed by the three-atom disk-projection method described below.

---

### 1 — Slab Selection

A water molecule $m$ with oxygen $z$-coordinate $z_O^{(m)}$ is included in slab $k$ if:

$$
\bigl| z_O^{(m)} - z_k \bigr| \;\leq\; \frac{\Delta z}{2}
\tag{1}
$$

Parameters: $\Delta z = 1.5\,\text{Å}$; slab centres $z_k \in \{-23.43,\; 0,\; +24.68\}\,\text{Å}$.

---

### 2 — XY Grid Definition

A uniform 2D rectangular grid $\{(x_i, y_j)\}$ spans the simulation box $(L_x \times L_y)$ at spacing $\Delta_{xy} = 0.5\,\text{Å}$.  All XY distances use the minimum-image convention (MIC) for periodic boundaries.

---

### 3 — Atom-Disk Indicator Function

For atom $\alpha \in \{\text{O},\,\text{H}_1,\,\text{H}_2\}$ of molecule $m$, the indicator $\chi_\alpha(x_i, y_j)$ marks grid cells inside the projected van der Waals disk of atom $\alpha$:

$$
\chi_{\alpha}(x_i, y_j) \;=\;
\begin{cases}
1 & \text{if}\quad \bigl|\mathbf{r}_{ij} - \mathbf{r}_\alpha^{xy}\bigr|_{\mathrm{MIC}} \leq r_\alpha \\[4pt]
0 & \text{otherwise}
\end{cases}
\tag{2}
$$

where $r_O = 1.52\,\text{Å}$ and $r_H = 1.20\,\text{Å}$ are the van der Waals radii.

---

### 4 — Electron-Weighted Atom Contribution

Each atom $\alpha$ carries electron weight $w_\alpha$ ($w_O = 8$; $w_H = 1$; total per H$_2$O: $w_{\mathrm{tot}} = 10$).  The area-normalised contribution to grid cell $(i, j)$ is:

$$
C_{\alpha}(x_i, y_j) \;=\; \frac{w_\alpha}{\pi\, r_\alpha^{\,2}}\;\chi_\alpha(x_i, y_j)
\tag{3}
$$

The factor $(\pi r_\alpha^2)^{-1}$ is the inverse projected-disk area, so that integrating $C_\alpha$ over the XY plane equals exactly $w_\alpha$ per molecule.

---

### 5 — Per-Molecule Contribution

Summing over all three atoms and normalising by $w_{\mathrm{tot}} = 10$ electrons:

$$
c_m(x_i, y_j) \;=\; \frac{1}{10}
\left[
  \frac{8\;\chi_O(x_i, y_j)}{\pi\, r_O^2}
  \;+\;
  \frac{\chi_{H_1}(x_i, y_j)}{\pi\, r_H^2}
  \;+\;
  \frac{\chi_{H_2}(x_i, y_j)}{\pi\, r_H^2}
\right]
\tag{4}
$$

---

### 6 — Frame Averaging and Final Density

The map is accumulated over $N_f$ trajectory frames (sampled every `step = 10` frames) and averaged:

$$
\boxed{
\rho_{\mathrm{proj}}(x_i, y_j;\, z_k)
\;=\;
\frac{1}{N_f}
\sum_{f=1}^{N_f}
\sum_{\substack{m\,\in\,\mathrm{slab}_k \\ \text{frame }f}}
c_m(x_i, y_j)
}
\tag{5}
$$

> **Units:** $\rho_{\mathrm{proj}}$ has units of **molecules Å⁻²** (projected 2D areal density).  
> $\Delta z$ enters only the slab-selection filter (Eq. 1); it is **not** used to normalise the density — the result is **not** a volumetric density (molecules Å⁻³).

---

### Parameter Summary

| Parameter | Symbol | Value |
|-----------|:------:|-------|
| Slab thickness | $\Delta z$ | 1.5 Å |
| XY grid spacing | $\Delta_{xy}$ | 0.5 Å |
| O vdW radius | $r_O$ | 1.52 Å |
| H vdW radius | $r_H$ | 1.20 Å |
| O electron weight | $w_O$ | 8 |
| H electron weight | $w_H$ | 1 |
| Total electrons per H₂O | $w_{\mathrm{tot}}$ | 10 |
| Frame stride | — | every 10th frame |
| Slab centres | $z_k$ | −23.43, 0, +24.68 Å |
| Reported quantity | $\rho_{\mathrm{proj}}$ | molecules Å⁻² |


In [ ]:
# Plot using plotter
from ZDirectionalPlotter import ZDirectionalPlotter
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_water_spatial_xy(
    save_plots=True, cmap='Blues',
    show_clay_atoms=False,
    show_mgo=False,
    show_title=False,
    show_clay_outline=False,
    colorbar_pad=0.1,  
    colorbar_width='4%', 
    save_individual_figures=True,    
    show_individual_figures=True,    
    show_combined_figure=False,    
    individual_figsize=(8, 6),
    vmax=0.1
    )

In [ ]:
peak_info = z_analysis.analyze_ion_peaks_detailed(
    peak_height_threshold=0.00012,
    peak_distance=1.5,
    show_clay_boundaries=True
)

In [ ]:
custom_radii = {'NA': 1.02, 'CL': 1.81, 'MG': 0.72}
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['NA'],
    projection_mode='hydrated_radius',
    z_slice_centers=[-21.6, 22.8],
    z_slice_width=3.0, 
    xy_grid_size=0.25,
    use_vdw_radius=True,
    vdw_radii=custom_radii,  # ✓ Custom values
    force_rerun=False,
    step=10
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        'z_slice_centers': [-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,  # ← Increase from 2.0 to 6.0
        'mg_vdw_scaling': 1.0,
        'si_vdw_scaling': 1.5,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'fade_method': 'gaussian',
        'mg_texture_intensity': 0.8,
        'si_connection_threshold': 6,
        'si_connection_style': 'lines',
        'si_connection_alpha': 0.8,  # ← Higher alpha
        'si_connection_linewidth': 2.0,  # ← Thicker lines
        'si_connection_zorder': 10,  # ← Higher zorder
        'show_hexagonal_pattern': True,
        'mg_glow_effect': True,
        'use_cache': False
    },
    
    ion_type='NA',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    show_grid=True,
    grid_alpha=0.5,
    # si_connection_threshold=4.5,
    cmap='Purples',
    mg_vdw_scaling=1.0,
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=5.5,

    overlay_clay_contours=True,
    # z_slice_width=2.0,
    mg_contour_color='green',
    mg_contour_alpha=0.3,
    mg_contour_levels=10, # use 2 only one ring
    mg_contour_linewidth=1,
    clay_contour_results=clay_results,
    # vmax=0.03,
    # sigma=1.0, # Gaussian smoothing
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    figsize=(15, 10)
)


In [ ]:
custom_radii = {'NA': 1.02, 'CL': 1.81, 'MG': 0.72}
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['NA'],
    projection_mode='hydrated_radius',
    z_slice_centers=[-19.1, 20.3],
    z_slice_width=3.0, 
    xy_grid_size=0.25,
    use_vdw_radius=True,
    vdw_radii=custom_radii,  # ✓ Custom values
    force_rerun=True,
    step=10
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        'z_slice_centers': [-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,  # ← Increase from 2.0 to 6.0
        'mg_vdw_scaling': 1.0,
        'si_vdw_scaling': 1.5,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'fade_method': 'gaussian',
        'mg_texture_intensity': 0.8,
        'si_connection_threshold': 6,
        'si_connection_style': 'lines',
        'si_connection_alpha': 0.8,  # ← Higher alpha
        'si_connection_linewidth': 2.0,  # ← Thicker lines
        'si_connection_zorder': 10,  # ← Higher zorder
        'show_hexagonal_pattern': True,
        'mg_glow_effect': True,
        'use_cache': False
    },
    
    ion_type='NA',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    show_grid=True,
    grid_alpha=0.5,
    # si_connection_threshold=4.5,
    cmap='Purples',
    mg_vdw_scaling=1.0,
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=10,

    overlay_clay_contours=True,
    # z_slice_width=2.0,
    mg_contour_color='green',
    mg_contour_alpha=0.3,
    mg_contour_levels=10, # use 2 only one ring
    mg_contour_linewidth=1,
    clay_contour_results=clay_results,
    # vmax=0.03,
    # sigma=1.0, # Gaussian smoothing
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    figsize=(15, 10)
)


In [ ]:
# VdW radius mode with custom radii
custom_radii = {'NA': 1.02, 'CL': 1.81, 'MG': 0.72}
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['MG'],
    projection_mode='hydrated_radius',
    z_slice_centers=[-21.6, 22.8],
    z_slice_width=3.0, 
    xy_grid_size=0.25,
    use_vdw_radius=True,
    vdw_radii=custom_radii,  # ✓ Custom values
    force_rerun=True,
    step=10
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        'z_slice_centers': [-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,  # ← Increase from 2.0 to 6.0
        'mg_vdw_scaling': 1.0,
        'si_vdw_scaling': 1.5,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'fade_method': 'gaussian',
        'mg_texture_intensity': 0.8,
        'si_connection_threshold': 6,
        'si_connection_style': 'lines',
        'si_connection_alpha': 0.8,  # ← Higher alpha
        'si_connection_linewidth': 2.0,  # ← Thicker lines
        'si_connection_zorder': 10,  # ← Higher zorder
        'show_hexagonal_pattern': True,
        'mg_glow_effect': True,
        'use_cache': False
    },
    
    ion_type='MG',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    show_grid=True,
    grid_alpha=0.5,
    # si_connection_threshold=4.5,
    cmap='GnBu', #'Greens',
    mg_vdw_scaling=1.0,
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=10,

    overlay_clay_contours=True,
    # z_slice_width=2.0,
    mg_contour_color='green',
    mg_contour_alpha=0.3,
    mg_contour_levels=10, # use 2 only one ring
    mg_contour_linewidth=1,
    clay_contour_results=clay_results,
    # vmax=0.08,
    # sigma=1.0, # Gaussian smoothing
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    figsize=(15, 10)
)


In [ ]:
# VdW radius mode with custom radii
custom_radii = {'NA': 1.02, 'CL': 1.81, 'MG': 0.72}
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['MG'],
    projection_mode='hydrated_radius',
    z_slice_centers=[-19.7, 20.9],
    z_slice_width=3.0, 
    xy_grid_size=0.25,
    use_vdw_radius=True,
    vdw_radii=custom_radii,  # ✓ Custom values
    force_rerun=True,
    step=10
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    clay_analysis_func=z_analysis.calculate_clay_spatial_distribution_xy,
    clay_analysis_params={
        'z_slice_centers': [-29.5, 26.5, 29.5],
        'z_slice_width': 2.0,  # ← Increase from 2.0 to 6.0
        'mg_vdw_scaling': 1.0,
        'si_vdw_scaling': 1.5,
        'mgo_radial_fade': True,
        'mgo_color': 'darkgreen',
        'si_color': 'orange',
        'fade_method': 'gaussian',
        'mg_texture_intensity': 0.8,
        'si_connection_threshold': 6,
        'si_connection_style': 'lines',
        'si_connection_alpha': 0.8,  # ← Higher alpha
        'si_connection_linewidth': 2.0,  # ← Thicker lines
        'si_connection_zorder': 10,  # ← Higher zorder
        'show_hexagonal_pattern': True,
        'mg_glow_effect': True,
        'use_cache': False
    },
    
    ion_type='MG',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    show_grid=True,
    grid_alpha=0.5,
    # si_connection_threshold=4.5,
    cmap='GnBu', #'Greens',
    mg_vdw_scaling=1.0,
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=10,

    overlay_clay_contours=True,
    # z_slice_width=2.0,
    mg_contour_color='green',
    mg_contour_alpha=0.3,
    mg_contour_levels=10, # use 2 only one ring
    mg_contour_linewidth=1,
    clay_contour_results=clay_results,
    # vmax=0.08,
    # sigma=1.0, # Gaussian smoothing
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    figsize=(15, 10)
)


In [ ]:
results = z_analysis.calculate_electrostatic_potential_spatial_distribution_xy(
z_slice_centers=[-21.6, 0, 22.8],
z_slice_width=3.0,
xy_grid_size=0.25,
return_surface_peaks=True,
surface_peak_method='global_max',
return_negative_potential=True,
histogram_heights=True
)

In [ ]:
# Start with a simpler 2D plot first
plotter.plot_electrostatic_potential_3d_histograms(
    plot_mode='2d',  # Use 2D instead of 3D first
    bar_alpha=0.8,
    colormap='plasma',
    threshold_percentile=10,  # Higher threshold = fewer bars
    save_individual_figures=True
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms(
    plot_mode='2d',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    save_individual_figures=True,
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal'
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms_flipped(
    plot_mode='2d',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='green',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    clay_z_tolerance=60.0,
    show_si_network=True,
    show_mgo_atoms=True,
    mg_vdw_scaling=3,
    si_connection_threshold=4.5,
    # clay_z_tolerance=5.0,
    si_color='orange',
    si_connection_alpha=0.6,
    si_connection_linewidth=1.2,
    si_connection_style='lines',
    show_hexagonal_pattern=True,
    si_center_alpha=0.8,
    mgo_color='darkgreen',
    # mgo_marker_size=60,
    mgo_atom_alpha=0.2,
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    title_fontsize=22,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    # figsize=(15, 10)
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal'
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms(
    plot_mode='2d',
    cmap='seismic',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    save_individual_figures=True,
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal'
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms_flipped3(
    plot_mode='surface',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='green',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    clay_z_tolerance=20.0, #60
    show_si_network=True,
    show_mgo_atoms=True,
    mg_vdw_scaling=0.5,
    si_vdw_scaling=0.5,
    si_connection_threshold=4.5,
    # clay_z_tolerance=5.0,
    si_color='orange',
    si_connection_alpha=0.5, #0.6
    si_connection_linewidth=1, #1.2
    si_connection_style='lines',
    show_hexagonal_pattern=True,
    si_center_alpha=0.5, #0.8
    mgo_color='darkgreen',
    # mgo_marker_size=60,
    mgo_atom_alpha=0.2,
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    save_individual_figures=True,
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal',
    z_axis_limit=0.5,
    elevation=45, # vertical viewing angle  # 30
)

In [ ]:
# plotter.plot_electrostatic_potential_3d_histograms(
#     plot_mode='surface',  # NEW: smooth surfaces
#     surface_cmap='RdYlBu_r',  # Publication colormap
#     surface_smoothing=True,   # Ultra-smooth
#     surface_alpha=0.8,        # Semi-transparent
#     show_colorbar=True
# )

# Plot with clay overlay
plotter.plot_electrostatic_potential_3d_histograms_flipped(
    plot_mode='surface',
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='black',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    z_axis_limit=0.5,
    elevation=45,
    clay_z_tolerance=20.0,
    # clay_surface_offset=-0.05 # e.g., -0.05 means 5% below surface min

    save_individual_figures=True,
)

In [ ]:
# plotter.plot_electrostatic_potential_3d_histograms(
#     plot_mode='surface',  # NEW: smooth surfaces
#     surface_cmap='seismic',  # Publication colormap
#     surface_smoothing=True,   # Ultra-smooth
#     surface_alpha=0.8,        # Semi-transparent
#     show_colorbar=True
# )

# Plot with clay overlay
plotter.plot_electrostatic_potential_3d_histograms_flipped(
    plot_mode='surface',
    surface_cmap='seismic',
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='black',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    z_axis_limit=0.5,
    elevation=45,
    clay_z_tolerance=20.0,
    clay_surface_offset=-0.05, # e.g., -0.05 means 5% below surface min

    save_individual_figures=True,
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms_flipped2(
    plot_mode='surface',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='green',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    clay_z_tolerance=20.0, #60
    show_si_network=True,
    show_mgo_atoms=True,
    mg_vdw_scaling=0.5,
    si_vdw_scaling=0.5,
    si_connection_threshold=4.5,
    # clay_z_tolerance=5.0,
    si_color='orange',
    si_connection_alpha=0.5, #0.6
    si_connection_linewidth=1, #1.2
    si_connection_style='lines',
    show_hexagonal_pattern=True,
    si_center_alpha=0.5, #0.8
    mgo_color='darkgreen',
    # mgo_marker_size=60,
    mgo_atom_alpha=0.2,
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    save_individual_figures=True,
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal',
    z_axis_limit=0.5,
    elevation=45, # vertical viewing angle  # 30
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms_flipped5(
    plot_mode='2d',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='green',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    clay_z_tolerance=60.0,
    show_si_network=True,
    show_mgo_atoms=True,
    mg_vdw_scaling=3,
    si_connection_threshold=4.5,
    # clay_z_tolerance=5.0,
    si_color='orange',
    si_connection_alpha=0.6,
    si_connection_linewidth=1.2,
    si_connection_style='lines',
    show_hexagonal_pattern=True,
    si_center_alpha=0.8,
    mgo_color='darkgreen',
    # mgo_marker_size=60,
    mgo_atom_alpha=0.2,
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    title_fontsize=22,
    label_fontsize=18,
    tick_fontsize=14,
    colorbar_label_fontsize=14,
    # legend_fontsize=14,
    dpi=600,
    save_individual_figures=True,
    # figsize=(15, 10)
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal',
    # buffer_2d=True
)

In [ ]:
plotter.plot_electrostatic_potential_3d_histograms_flipped5(
    plot_mode='surface',
    cmap='RdYlBu_r',  # ✅ Use 'cmap' for 2D mode, not 'colormap'
    overlay_clay_contours=True,
    clay_contour_results=clay_results,
    mg_contour_color='green',
    mg_contour_alpha=0.4,
    mg_contour_levels=10,
    clay_z_tolerance=20.0, #60
    show_si_network=True,
    show_mgo_atoms=True,
    mg_vdw_scaling=0.5,
    si_vdw_scaling=0.5,
    si_connection_threshold=4.5,
    # clay_z_tolerance=5.0,
    si_color='orange',
    si_connection_alpha=0.5, #0.6
    si_connection_linewidth=1, #1.2
    si_connection_style='lines',
    show_hexagonal_pattern=True,
    si_center_alpha=0.5, #0.8
    mgo_color='darkgreen',
    # mgo_marker_size=60,
    mgo_atom_alpha=0.2,
    # Remove 3D-specific parameters:
    # bar_alpha=0.8,        # Only for 3D bars
    # colormap='RdYlBu_r',  # Only for 3D bars  
    # threshold_percentile=10, # Only for 3D bars
    save_individual_figures=True,
    # 2D-specific parameters you can use:
    interpolation='gaussian',
    origin='lower',
    aspect='equal',
    z_axis_limit=0.5,
    elevation=45, # vertical viewing angle  # 30
    buffer_3d=True
)

In [ ]:
# Electrostatic potential
potential = z_analysis.calculate_electrostatic_potential_vs_z(
    charge_dict={'NA': 1.0, 'CL': -1.0, 'MG': 2.0, 'CA': 2.0}
)
plotter.plot_electrostatic_potential(save_plots=True, show_electric_field=False)

In [ ]:
# Calculate PMF in kJ/mol at 298K
pmf_results = z_analysis.calculate_pmf_vs_z(
    ion_types=['NA', 'CL', 'MG'], # ['NA'],
    temperature=298.15,
    units='kJ_mol',
    reference_method='custom_region',reference_region=(-3, 3),
    bulk_region_z=0.0,
    shift_pmf=True,
    noise_region=(-5, 5), flag_noise=True,
    symmetrize=True, symmetrize_method='average',
    smoothing=False, smoothing_method='gaussian',
    limit_pmf_range=True, max_pmf_value=2.0,
    replace_high_pmf_method='cap' # 'interpolate'
    # confidence_intervals=True, 
)



In [ ]:
plotter.plot_pmf_vs_z(
    title_fontsize=22,
    label_fontsize=18,
    tick_fontsize=14,
    linewidth=3.0,
    show_grid=True, grid_alpha=0.3,
    ## show clay boundaries
    show_clay_boundaries_on_overlay=True,
    symmetrize_boundaries=True,
    ion_colors={
        'CL': 'red',
        'NA': 'blue',
        'K' : 'purple',
        'MG': 'green',
        'CA': 'yellow'
    },
    y_min=-10.0, y_max=2.0,
    show_legend=False,
    show_title=False,
    # marker_size=100,
    legend_frameon=False,
    # replace_high_pmf_method='extrapolate', #'cap',
    show_binding_sites=False,  # Remove all markers
    show_barriers=False,
    dpi=600,    
    save_individual_figures=True
)

In [ ]:
bootstrap_results = z_analysis.calculate_bootstrap_pmf_uncertainty(
    ion_types=['NA', 'MG', 'CL'],
    confidence_levels=[0.68, 0.95],
    n_bootstrap=1000,
    temperature=298.15,
    units='kJ_mol',
    limit_pmf_range=True,
    max_pmf_value=2.0,
    replace_high_pmf_method='cap',
    reference_method='custom_region',
    reference_region=(-3, 3),
    bulk_region_z=0.0,
    shift_pmf=True,
    noise_region=(-10, 10),
    flag_noise=True,
    symmetrize=True,
    symmetrize_method='average'
)

In [ ]:
plotter.plot_pmf_with_bootstrap_uncertainty_bands(
    ['NA', 'MG', 'CL'], 
    bootstrap_results=bootstrap_results,
    # confidence_levels=[0.68, 0.95],
    confidence_levels=[0.68],
    ion_colors={
        'CL': 'red',
        'NA': 'blue',
        'K' : 'purple',
        'MG': 'green',
        'CA': 'yellow'
    },
    figsize=(8, 6),
    title_fontsize=22,
    label_fontsize=20,
    tick_fontsize=16,
    linewidth=3.0,
    show_grid=True, grid_alpha=0.3,
    fill_alpha=0.25,
    use_precomputed_intervals=True,
    # x_min=-24.5, x_max=24.5,
    y_min=-10.0, y_max=2.0,
    show_clay_boundaries=True,
    show_legend=False,
    show_title=False,
    # symmetrize_boundaries=True,
)

In [ ]:
# plotter.plot_pmf_with_bootstrap_uncertainty_bands(['NA', 'CL'])

# With custom styling
plotter.plot_pmf_with_bootstrap_uncertainty_bands(
    ['NA', 'MG', 'CL'], 
    bootstrap_results=bootstrap_results,
    confidence_levels=[0.95],
    ion_colors={
        'CL': 'red',
        'NA': 'blue',
        'K' : 'purple',
        'MG': 'green',
        'CA': 'yellow'
    },
    figsize=(8, 6),
    title_fontsize=22,
    label_fontsize=20,
    tick_fontsize=16,
    linewidth=3.0,
    show_grid=True, grid_alpha=0.3,
    fill_alpha=0.25,
    use_precomputed_intervals=True,
    # x_min=-24.5, x_max=24.5,
    y_min=-10.0, y_max=2.0,
    show_clay_boundaries=True,
    show_legend=False,
    show_title=False,
    symmetrize_boundaries=True,
)

In [ ]:
# Usage example with PMF calculation:
pmf_results = z_analysis.calculate_pmf_vs_z_from_ion_density(
    ion_types=['NA', 'MG', 'CL'],
    target_z_values=[-21.6, 0.0, 22.8],
    z_slice_width=1, 
    xy_grid_size=1,
    temperature=298.15,
    use_cache=True
    )

In [ ]:
# Plot only specific ions and z-values
plot_info = plotter.plot_pmf_3d_spatial_distribution_vs_z(
    debug_results=pmf_results,
    ion_types=['NA', 'MG', 'CL'],  
    colormap='seismic', # 'viridis',
    z_slice_values=[-21.6, 0.0, 22.8], 
    plot_type='3d_surface',
    surface_smoothing=True,
    surface_smoothing_sigma=1.5,
    smoothing_method='gaussian',
    handling_nan='transparent',
    nan_color='lightgray',
    nan_alpha=0,
    outlier_detection=True,
    outlier_method='median',
    outlier_threshold=3.0,
    outlier_correction='median',
    figsize=(8, 6),
    label_fontsize=20,
    tick_fontsize=16,
    colorbar_tick_fontsize=14,
    show_grid=False,
    dpi=600,
    save_individual_figures=True
)

In [ ]:
height_results = z_analysis.create_pmf_height_surfaces(
    debug_results=pmf_results,           # Same input as isosurfaces ✅
    # ion_types=['NA', 'MG', 'CL'],        # Ion types to process
    ion_types=['NA', 'MG'],
    z_surface_values=[-21.6, 22.8],               # Use all z-values from PMF data
    smoothing=True,                      # Apply Gaussian smoothing
    # smoothing_sigma=1,                 # Smoothing parameter
    z_scale_factor=None,                 # Auto-calculate scaling
    grid_upsampling_factor=2,            # 2x higher resolution
    nan_handling='interpolate'           # Interpolate NaN values
)

In [ ]:
plot_results = plotter.plot_pmf_height_surfaces(
    height_surface_results=height_results,
    ion_types=['NA', 'MG'],
    colorbar_label='PMF (kJ/mol)',

    # NEW: Spatial interpolation parameters
    fill_missing_data=True,              # Enable gap filling
    max_neighbor_cells=10,                # Search 3 cells around
    interpolation_method='distance_weighted',  # Weighted by distance
    min_neighbors_required=2,            # Need ≥2 neighbors

    colorbar_scaling='unified',
    visualization_mode='3d_surfaces',
    surface_style='smooth',
    color_scheme='pmf_gradient',
   
    ## surface height smoothing
    surface_smoothing=True,
    surface_smoothing_sigma=1.5,
    smoothing_method='gaussian',

    # Standard parameters
    show_title=False,
    label_fontsize=20,
    tick_fontsize=16,
    colorbar_tick_fontsize=14,
    individual_figsize=(8, 6),
    save_individual_figures=True,
    dpi=600,
)

In [ ]:
# Generate dense Z-slices across the entire box
z_min = -22  # Your system's Z range
z_max = -5
z_slice_width = 2.0

# Create MANY overlapping slices for complete coverage
target_z_values = np.arange(z_min, z_max + z_slice_width, z_slice_width/2)
# This creates slices every 1 Å (50% overlap) for smooth interpolation
# Result: [-25, -24, -23, -22, ..., +23, +24, +25] = ~50 slices

debug_results = z_analysis.calculate_pmf_vs_z_from_ion_density(
    ion_types=['NA'],
    reference_z=-5.0,
    target_z_values=target_z_values,  # Dense coverage!
    z_slice_width=z_slice_width,
    xy_grid_size=2.0,
    temperature=298.15
)

In [ ]:
# Test the fixed volumetric PMF calculation  
print("Testing volumetric PMF calculation with corrected ion type checking...")

# Calculate volumetric PMF
volume_results = z_analysis.calculate_pmf_volume_from_ion_density(
    debug_results, 
    ion_types=['NA'],
    target_z_values=target_z_values,  # Add this line!
    z_slice_width=2.0,
    cube_size=2.0,
    temperature=298.15,
    verbose=True  # Enable verbose output to see progress
)

print(f"Success! Generated volume results: {type(volume_results)}")
if 'pmf_volumes' in volume_results:
    print(f"PMF volumes for ion types: {list(volume_results['pmf_volumes'].keys())}")

In [ ]:
# Visualize with multiple options
plotter.plot_pmf_3d_volume(
    volume_results=volume_results,
    volume_render=True,
    slice_plots=True,
    show_individual_figures=True,
    publication_quality=True,
    save_plots=True
)

In [ ]:
# Basic usage with automatic positioning
plot_path = plotter.plot_pmf_3d_volume_enhanced(volume_results, show_plots=True)

# Full control over slice positions and surface
plot_path = plotter.plot_pmf_3d_volume_enhanced(
    volume_results,
    slice_x_position=0.0,      # Center YZ view at X=0
    slice_y_position=0.0,      # Center XZ view at Y=0  
    slice_z_position=-15.0,    # XY view closer to clay surface
    surface_z_range=(-22, -5), # Focus surface on interface region
    surface_style='bedsheet',   # Energy landscape surface
    show_plots=True
)

In [ ]:
# Use the improved method - shows FULL coordinate ranges
plot_path = plotter.plot_pmf_3d_volume_enhanced(volume_results, show_plots=True)

plot_results = plotter.plot_pmf_3d_volume(
    volume_results,
    ion_types=None,  # All ion types
    slice_plots=False,  # Focus only on 3D energy landscape  
    volume_render=True,
    save_plots=True,
    show_plots=True,
    colormap='RdBu_r',
    alpha=0.6
)

In [ ]:
# Plot only specific ions and z-values
plot_info = plotter.plot_pmf_3d_spatial_distribution_from_debug(
    debug_results=pmf_results,
    ion_types=['NA', 'MG', 'CL'],  
    colormap='seismic', # 'viridis',
    z_slice_values=[-21.6, 0.0, 22.8],  
    plot_type='3d_surface',
    surface_smoothing=True,
    surface_smoothing_sigma=1.5,
    smoothing_method='gaussian',
    handling_nan='transparent',
    nan_alpha=0.1,
    outlier_detection=True,
    outlier_method='median',
    outlier_threshold=3.0,
    outlier_correction='mean',
    save_plots=True,
    # figsize=(8, 6),
    # dpi=600
)

In [ ]:
# For smooth surfaces with binding site preservation


In [ ]:
# Usage example with PMF calculation:
pmf_results = z_analysis.calculate_pmf_vs_z_from_ion_density(
    ion_types=['NA', 'MG', 'CL'],
    target_z_values=[-21.6, 0.0, 22.8],
    z_slice_width=1, 
    xy_grid_size=1,
    temperature=298.15
    )

In [ ]:
# Plot only specific ions and z-values
plot_info = plotter.plot_pmf_3d_spatial_distribution_from_debug(
    debug_results=pmf_results,
    ion_types=['NA', 'MG', 'CL'],  
    colormap='seismic', # 'viridis',
    z_slice_values=[-21.6, 0.0, 22.8],  
    plot_type='3d_surface',
    surface_smoothing=True,
    surface_smoothing_sigma=1.5,
    smoothing_method='gaussian',
)

In [ ]:
# Plot only specific ions and z-values
plot_info = plotter.plot_pmf_3d_spatial_distribution_vs_z(
    debug_results=pmf_results,
    ion_types=['NA', 'MG', 'CL'],  
    colormap='seismic', # 'viridis',
    z_slice_values=[-21.6, 0.0, 22.8],  
    plot_type='3d_surface',
    surface_smoothing=True,
    surface_smoothing_sigma=1.5,
    smoothing_method='gaussian',
    handling_nan='transparent',
    nan_color='lightgray',
    nan_alpha=0.0,
    figsize=(8, 6),
    label_fontsize=20,
    tick_fontsize=16,
    colorbar_tick_fontsize=14,
    show_grid=False,
    dpi=600,
    save_individual_figures=True
)

In [ ]:
# Plot only specific ions and z-values
plot_info = plotter.plot_pmf_3d_spatial_distribution_from_debug(
    debug_results=pmf_results,
    ion_types=['NA', 'MG', 'CL'],  
    colormap='seismic', # 'viridis',
    z_slice_values=[-21.6, 0.0, 22.8],  
    plot_type='3d_surface',
    surface_smoothing=True,
    surface_smoothing_sigma=1.0,
    smoothing_method='gaussian',
)

In [ ]:
isosurface_results = z_analysis.create_adaptive_stacked_isosurfaces(
    debug_results=pmf_results,           # Required: PMF calculation results
    ion_types=['NA', 'MG', 'CL'],        # Ion types to analyze ['NA', 'MG', 'CL'], 
    n_energy_levels=20,                  # Number of energy levels per ion
    percentile_range=(0, 100),           # Use 10th-90th percentile range
    target_z_values=None,                # Use all z-values from debug_results
    z_spacing=0,                         # Vertical spacing between surfaces (Å)
    colors=None,                         # Auto-generate colors
    alphas=None,                         # Auto-generate transparency
    energy_tolerance=None,               # Auto-calculate tolerance
    isosurface_method='marching_cubes',  # Method for surface extraction
    smoothing=True,                      # Apply smoothing
    smoothing_sigma=0.15                  # Smoothing parameter
)

In [ ]:
# Basic 3D stacked surfaces
plotter.plot_adaptive_stacked_isosurfaces(
    isosurface_results=isosurface_results,
    # ion_types=['NA'], 
    ion_types=['NA', 'MG', 'CL'],
    visualization_mode='3d_surfaces',
    lighting_style='dramatic',
    surface_style='solid',
    transparency_mode='energy_based',
    view_angle=(25, 45), # view_angle=(30, 45)
)

In [ ]:
# 1. Calculate ion spatial distribution
z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['NA', 'CL', 'MG'],
    z_slice_centers=np.arange(-20, 21, 2.0),  # Every 2 Å from -20 to +20
    z_slice_width=2.0,
    xy_grid_size=0.5
)

# # 2. Calculate PMF from the spatial distribution
# pmf_results = z_analysis.calculate_pmf_vs_z_from_ion_density(
#     ion_types=['NA', 'CL', 'MG'],
#     temperature=298.15,
#     units='kJ/mol'
# )

# # 3. Now you can plot the PMF surfaces
# fig = z_plotter.plot_pmf_surface_from_ion_density(
#     ion_types=['NA', 'CL'],
#     plot_mode='surface'
# )

In [ ]:
# Electrostatic potential
potential = z_analysis.calculate_electrostatic_potential_vs_z(
    # charge_dict={'NA': 1.0, 'CL': -1.0, 'MG': 2.0, 'CA': 2.0}
)
# z_analysis.plot_electrostatic_potential(save_plots=True)

In [ ]:
# Local temperature
temperature = z_analysis.calculate_local_temperature_vs_z(
    mass_dict={'O': 16.0, 'H': 1.0, 'NA': 23.0, 'MG': 24.3, 'CL': 35.5}
)

In [ ]:
plotter.plot_electrostatic_potential(
    save_plots=True, 
    charge_density_color='black',          # Color for total line
    charge_density_linewidth=2.5,          # Thicker line for total
    ion_linewidth=2.0,   
    ion_colors={
        'CL': 'red',
        'NA': 'blue',
        'K' : 'purple',
        'MG': 'green',
        'CA': 'yellow'
    },
    display_units='kJ/mol',
    show_electric_field=True,
    show_field_direction_colors=True,
    field_color='black',
    potential_color='black',
    field_positive_color='red',
    field_negative_color='green',
    field_direction_alpha=0.2,
    # Clay styling
    mgo_linestyle='--',
    mgo_linewidth=1.5,
    show_clay_boundaries=True,
    symmetrize_boundaries=True,
    clay_fill_regions=True,
    clay_fill_color='yellow',
    clay_fill_alpha=0.1,
    # Boundary line colors
    clay_avg_color='gray',
    si_color='orange',
    mgo_color='green',
    # linewidth=3.0,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=False,
    show_combined_figure=True,
    save_individual_figures=True,
    title_fontsize=22,
    label_fontsize=20,
    tick_fontsize=16,
    dpi=600,
    )

In [ ]:
# z_analysis.calculate_water_structure_vs_z_detailed()

In [ ]:
# Water structure analysis
water_struct = z_analysis.calculate_water_structure_vs_z_detailed(
    force_rerun=True,
    save_cache=True,
    step=5,                          # Every 5th frame
    # n_frames_sample=10               # Or sample 10 frames
)

In [ ]:
# Plot the results using the advanced plotter
z_analysis.plot_water_structure_analysis(save_plots=True)

In [ ]:
# Custom z-range
# Use the enhanced plotting with custom z-range
plotter.plot_water_structure_analysis(
    z_min=-20, 
    z_max=20,
    save_plots=True,
    show_bulk_references=True,
    highlight_interfaces=False
)

In [ ]:
# Use custom bin width (will trigger recalculation if recalculate_if_needed=True)
plotter.plot_water_structure_analysis(
    z_min=-20, 
    z_max=20,
    # bin_width=0.5,  # Finer resolution
    # recalculate_if_needed=True,
    save_plots=True,
    show_bulk_references=True,
    highlight_interfaces=True
)

In [ ]:
results = z_analysis.calculate_solvation_shells_vs_z_detailed(
    shell_radii=[3.5, 6.0, 9.0], #0-3.15, 3.15-5.55, 5.55-7.5
    ion_specific_radii={
        'MG': [2.88, 5.03, 7.25],      # Tighter shells for Mg2+
        'NA': [3.18, 5.53, 7.58],      # Intermediate for Na+
        # CL will use default shell_radii
    },
    n_frames_sample=None,
    step=1,
    enable_spatial_analysis=True,
    ion_types=['NA', 'MG'],
    target_z_values=[-21.6, 0, 22.8],
    z_slice_width=1.0,
    xy_grid_size=1.0,
    force_rerun=False,
)

In [ ]:
# Create coordination height surfaces from existing coordination data
coord_height_results = z_analysis.create_coordination_height_surfaces(
    ion_types=['NA', 'MG'],
    plot_mode='shells',  # 'shells' or 'z_slices'
    smoothing=True,
    z_scale_factor=1.0,
    shell_selection=['first_shell', 'second_shell', 'third_shell']
)

In [ ]:
Sequential: 'viridis', 'plasma', 'inferno', 'magma', 'cividis'
Perceptually uniform: 'viridis', 'plasma', 'turbo'
High contrast: 'jet', 'turbo', 'rainbow'
Diverging: 'coolwarm', 'seismic', 'RdYlBu'

In [ ]:
# Plot coordination shells (recommended)
plotter.plot_coordination_height_surfaces(
    ion_types=['NA', 'MG'],
    plot_mode='shells',  # Show first/second/third shells
    surface_smoothing=True,
    surface_alpha=0.7,
    fill_missing_data=True,
    # clay_overlay=True,
    # per_surface_color_normalization=True,
    per_shell_colorbars=True,
    colorbar_pad=0.01,
    colorbar_width='3%',
    # colorbar_labelpad=5,
    colorbar_label_x=4.5,  # Move labels further left
    colorbar_label_y=0.5, 
    wspace=0.4,            # Gap between NA and MG subplots (adjust if still overlapping)
    left_margin=0.15,       # Increase if left axis still cropped
    right_margin=0.88,      # Adjust if colorbars need more/less space
    shell_surface_cmaps={
        'shell_1': 'viridis',
        'shell_2': 'jet',
        'shell_3': 'RdYlBu',
    },
    figsize=(15, 12),
    surface_cmap='seismic', # 'rainbow', # 'jet', #'seismic', 'viridis' 'turbo'
    color_scheme='coordination_gradient', #'height_gradient',# 'coordination_gradient',
    colorbar_scaling='unified', #'individual'# 'global', # 'unified' 'individual'
    show_title=False,
    label_fontsize=20,
    tick_fontsize=16,
    colorbar_tick_fontsize=14,
    individual_figsize=(8, 6),
    save_individual_figures=True,
    save_combined_figure=True,
    show_combined_figure=True
    # discrete_colors=True,
    # n_color_levels=10,
    # enhance_contrast=True,
    # contrast_gamma=3.0,

)

In [ ]:
# Solvation shell analysis
solvation = z_analysis.calculate_solvation_shells_vs_z_detailed(
    shell_radii=[3.18, 5.53, 7.58],   # Shell boundaries
    ion_specific_radii={
        'MG': [2.88, 5.03, 7.25],      # Tighter shells for Mg2+
        'NA': [3.18, 5.53, 7.58],      # Intermediate for Na+
        'CL': [3.78, 6.18, 8.43],
    },
    force_rerun=True,
    step=1,
    # n_frames_sample=10
)

In [ ]:
# Plot with symmetrization
plotter.plot_solvation_shell_comparison(
    ion_type=['NA', 'MG'], 
    save_plots=True,
    symmetrize=True,
    symmetrize_method='fold',  # or 'average', 'fold', 'mirror_positive', 'mirror_negative'
    show_legend=False,
    show_title=False,
    label_fontsize=22,
    tick_fontsize=18,
    shell_linewidth=3,
    shell_colors={
        'first_shell': "#017ca1",   # saturation=1.0
        'second_shell': "#29caf7",  # saturation=0.7
        'third_shell': "#8fe0f4",   # saturation=0.4
    },
    total_linewidth=2,
    fill_shells=True,
    fill_total=True,
    shell_fill_colors={
        'first_shell': "#017ca1",   # saturation=1.0
        'second_shell': "#29caf7",  # saturation=0.7
        'third_shell': "#8fe0f4",   # saturation=0.4
    },
    total_fill_color='#cff5ff',     # saturation=0.2
    total_fill_alpha=0.10,
    shell_fill_alpha=0.2,
    save_individual_figures=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    dpi=600,
)

In [ ]:
# Interface region with enhanced features
plotter.plot_water_structure_analysis(z_min=-15, z_max=15, 
                                     show_bulk_references=True,
                                     highlight_interfaces=True)

In [ ]:
# Plot the results using the advanced plotter
z_analysis.plot_water_structure_analysis(save_plots=True)

In [ ]:
import matplotlib.pyplot as plt

if hasattr(z_analysis.results, 'water_structure') and z_analysis.results.water_structure is not None:
    water_struct = z_analysis.results.water_structure
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Plot 1: Tetrahedral order vs z
    axes[0, 0].plot(water_struct['z_centers'], water_struct['tetrahedral_order'], 'b-o', markersize=4)
    axes[0, 0].set_title('Tetrahedral Order vs Z')
    axes[0, 0].set_xlabel('z (Å)')
    axes[0, 0].set_ylabel('Q parameter')
    axes[0, 0].grid(True)
    
    # Plot 2: H-bonds vs z
    axes[0, 1].plot(water_struct['z_centers'], water_struct['hydrogen_bonds_per_water'], 'r-o', markersize=4)
    axes[0, 1].set_title('H-bonds per Water vs Z')
    axes[0, 1].set_xlabel('z (Å)')
    axes[0, 1].set_ylabel('H-bonds')
    axes[0, 1].grid(True)
    
    # Plot 3: Coordination vs z
    axes[1, 0].plot(water_struct['z_centers'], water_struct['water_coordination_number'], 'g-o', markersize=4)
    axes[1, 0].set_title('Water Coordination vs Z')
    axes[1, 0].set_xlabel('z (Å)')
    axes[1, 0].set_ylabel('Coordination Number')
    axes[1, 0].grid(True)
    
    # Plot 4: Water count vs z
    axes[1, 1].plot(water_struct['z_centers'], water_struct['water_counts'], 'purple', marker='o', markersize=4)
    axes[1, 1].set_title('Water Count vs Z')
    axes[1, 1].set_xlabel('z (Å)')
    axes[1, 1].set_ylabel('Waters per Layer')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed layer information
    print(f"\nDetailed Layer-by-Layer Results:")
    print(f"{'Z-center':>10} {'TetraOrder':>12} {'H-bonds':>10} {'Coord':>8} {'Count':>8}")
    print("-" * 55)
    
    for i in range(min(10, len(water_struct['z_centers']))):  # Show first 10 layers
        z = water_struct['z_centers'][i]
        tetra = water_struct['tetrahedral_order'][i]
        hbonds = water_struct['hydrogen_bonds_per_water'][i]
        coord = water_struct['water_coordination_number'][i]
        count = water_struct['water_counts'][i]
        print(f"{z:10.2f} {tetra:12.4f} {hbonds:10.2f} {coord:8.2f} {count:8.1f}")

else:
    print("No water structure data available. Run calculate_water_structure_vs_z_detailed() first.")

In [ ]:
# Solvation shell analysis
solvation = z_analysis.calculate_solvation_shells_vs_z_detailed(
    shell_radii=[3.18, 5.53, 7.58],   # Shell boundaries
    ion_specific_radii={
        'MG': [2.88, 5.03, 7.25],      # Tighter shells for Mg2+
        'NA': [3.18, 5.53, 7.58],      # Intermediate for Na+
        'CL': [3.78, 6.18, 8.43],
    },
    force_rerun=False,
    step=1,
    n_frames_sample=10
)

In [ ]:
plotter.plot_solvation_shells_vs_z(save_plots=True)

In [ ]:
# plotter.plot_solvation_shell_comparison(ion_type='NA', save_plots=True)
plotter.plot_solvation_shell_comparison(ion_type=['NA', 'MG'], save_plots=True)

In [ ]:
# Plot with symmetrization
plotter.plot_solvation_shell_comparison(
    ion_type=['NA', 'MG'], 
    save_plots=True,
    symmetrize=True,
    symmetrize_method='average',  # or 'fold', 'mirror_positive', 'mirror_negative'
    show_legend=False,
    show_title=False,
    label_fontsize=22,
    tick_fontsize=18,
    shell_linewidth=3,
    total_linewidth=2,
    fill_shells=True,
    fill_total=True,
    total_fill_alpha=0.10,
    shell_fill_alpha=0.2,
    save_individual_figures=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    dpi=600,
)

In [ ]:
print("\n=== DEBUG: Shell water counts at z≈0 ===")
z_centers = solvation['z_centers']
middle_z_idx = np.argmin(np.abs(z_centers))
print(f"Z-position at center: {z_centers[middle_z_idx]:.2f} Å")
print(f"\nStored values in shell_water_counts arrays:")
for shell_name in ['first_shell', 'second_shell', 'third_shell']:
    mg_value = solvation['shell_water_counts'][shell_name]['MG'][middle_z_idx]
    print(f"  {shell_name}: {mg_value:.2f}")
    
print(f"\nTotal coordination: {solvation['coordination_numbers']['MG'][middle_z_idx]:.2f}")
print(f"\nSum of shells: {sum([solvation['shell_water_counts'][s]['MG'][middle_z_idx] for s in ['first_shell', 'second_shell', 'third_shell']]):.2f}")

In [ ]:
# Plot with symmetrization
plotter.plot_solvation_shell_comparison(
    ion_type=['NA', 'MG'], 
    save_plots=True,
    symmetrize=True,
    symmetrize_method='fold',  # or 'average', 'fold', 'mirror_positive', 'mirror_negative'
    show_legend=False,
    show_title=False,
    label_fontsize=22,
    tick_fontsize=18,
    shell_linewidth=3,
    shell_colors={
        'first_shell': "#017ca1",   # saturation=1.0
        'second_shell': "#29caf7",  # saturation=0.7
        'third_shell': "#8fe0f4",   # saturation=0.4
    },
    total_linewidth=2,
    fill_shells=True,
    fill_total=True,
    shell_fill_colors={
        'first_shell': "#017ca1",   # saturation=1.0
        'second_shell': "#29caf7",  # saturation=0.7
        'third_shell': "#8fe0f4",   # saturation=0.4
    },
    total_fill_color='#cff5ff',     # saturation=0.2
    total_fill_alpha=0.10,
    shell_fill_alpha=0.2,
    save_individual_figures=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    dpi=600,
)

In [ ]:
# Compare different symmetrization methods
for method in ['average', 'fold', 'mirror_positive']:
    plotter.plot_solvation_shell_comparison(
        ion_type=['NA', 'MG'],
        save_plots=True,
        symmetrize=True,
        symmetrize_method=method,
        filename=f'solvation_shells_NA_MG_{method}.png'
    )

In [ ]:
# RDF vs z-position
rdf_data = z_analysis.calculate_rdf_vs_z(
    # ion_pairs=['Na-Ow', 'Mg-Ow'],
    ion_pairs=['Na-Ow', 'Mg-Ow', 'Cl-Ow', 'Na-Cl', 'Mg-Cl'],
    r_max=12.0,
    dr=0.1,
    z_bin_width=1.0,
    # z_range=(-30, 30),
    step=1,
    # n_frames_sample=1000,
    density_mode='global',
    debug=True,
    force_rerun=False
)

In [ ]:
# Plot both orientations for comparison
print("Plotting standard orientation (r vs z):")
plotter.plot_rdf_vs_z(
    save_plots=True,
    ion_pairs=['Na-Ow', 'Mg-Ow', 'Cl-Ow'],
    swap_axes=False
)

print("\nPlotting swapped orientation (z vs r):")
plotter.plot_rdf_vs_z(
    save_plots=True,
    ion_pairs=['Na-Ow', 'Mg-Ow', 'Cl-Ow'],
    swap_axes=True
)

In [ ]:
import matplotlib.colors as mcolors

# Extended custom sea-blue colormap - REVERSED (light to dark)
# Higher g(r) = darker blue, Lower g(r) = lighter blue
colors = [
    "#d9f3fc",  # Lightest blue/almost white (low g(r))
    "#b8eaf8",  # Extra light blue
    "#8fe0f4",  # Very light blue (your original)
    "#5dd6f9",  # Light blue
    "#29caf7",  # Medium-light blue (your original)
    "#1ab8e8",  # Medium-bright blue
    "#0199c4",  # Medium blue
    "#017ca1",  # Medium-dark blue (your original)
    "#016b88",  # Dark blue
    "#014d63",  # Very dark blue
    "#013d50",  # Deepest blue (high g(r))
]

custom_seablue_cmap = mcolors.LinearSegmentedColormap.from_list('custom_seablue', colors)

plotter.plot_rdf_vs_z(
    save_plots=True,
    ion_pairs=['Na-Ow', 'Mg-Ow', 'Cl-Ow'],
    cmap=custom_seablue_cmap,
    ion_solvation_radii = {
        'MG': [2.88, 5.03, 7.25],  # 1st, 2nd, 3rd shell boundaries
        'NA': [3.18, 5.53, 7.58],
        'CL': [3.78, 6.18, 8.43],
    },
    ion_pair_radii = {
        'NA-CL': [3.50, 5.90, 8.20],
        'MG-CL': [3.50, 5.40, 7.70],
        'CA-CL': [3.65, 6.35, 8.55],
        'K-CL': [3.90, 6.35, 8.55],
    },

    show_solvation_boundary=True,
    show_pairing_boundary=True,

    boundary_line_color='black',
    boundary_line_style='--',
    boundary_line_width=0.7,
    boundary_line_alpha=0.3,
    show_title=False,
    label_fontsize=24,
    tick_fontsize=20,
    colorbar_tick_fontsize=18,
    individual_figsize=(8, 6),
    save_individual_figures=True,
    show_combined_figure=True,
    swap_axes=False,
    dpi=600
)


In [ ]:
import matplotlib.colors as mcolors

# Ion-pairing colormap - coral → blue → green → yellow
# cuitable for all other cation-anion pairs
# colors = ["#f08080", "#e69999", "#d4b3d4", "#add8e6", "#9fd4d4", "#91d4b8", "#90ee90", "#b4ee90", "#d4ee90", "#ffffb3", "#ffffe0"]
# # Suitable for Na-Cl
# colors = ["#add8e6", "#9fd4d4", "#91d4b8", "#90ee90", "#b4ee90", "#d4ee90", "#ffffb3", "#ffffe0", "#d4b3d4", "#e69999", "#f08080"]
# Default
colors = ["#ffffe0", "#ffffb3", "#d4ee90", "#b4ee90", "#90ee90", "#91d4b8", "#9fd4d4", "#add8e6", "#d4b3d4", "#e69999", "#f08080"]
# colors = [
#     "#f08080",  # lightcoral - CIP region (high g(r), contact pairs)
#     "#e69999",  # coral-pink transition
#     "#d4b3d4",  # coral to blue transition
#     "#add8e6",  # lightblue - SIP region
#     "#9fd4d4",  # blue-cyan transition
#     "#91d4b8",  # blue to green transition
#     "#90ee90",  # lightgreen - DSIP region
#     "#b4ee90",  # green-yellow transition
#     "#d4ee90",  # yellow-green transition
#     "#ffffb3",  # light yellow transition
#     "#ffffe0",  # lightyellow - FI region (low g(r), free ions)
# ]

custom_ionpairing_cmap = mcolors.LinearSegmentedColormap.from_list('custom_ionpairing', colors)

plotter.plot_rdf_vs_z(
    save_plots=True,
    ion_pairs=['Na-Cl', 'Mg-Cl'],
    cmap=custom_ionpairing_cmap,
    ion_pair_radii = {
        'NA-CL': [3.50, 5.90, 8.20],
        'MG-CL': [3.50, 5.40, 7.70],
        'CA-CL': [3.65, 6.35, 8.55],
        'K-CL': [3.90, 6.35, 8.55],
    },
    show_pairing_boundary=True,
    boundary_line_color='black',
    boundary_line_style='--',
    boundary_line_width=0.7,
    boundary_line_alpha=0.3,
    show_title=False,
    label_fontsize=24,
    tick_fontsize=20,
    colorbar_tick_fontsize=18,
    individual_figsize=(8, 6),
    save_individual_figures=True,
    show_combined_figure=True,
    swap_axes=False,
    dpi=600
)

In [ ]:
pairing = z_analysis.calculate_ion_pairing_vs_z(
    contact_distance=3.5,
    ion_pairs=['NA-CL', 'MG-CL'],
    n_frames_sample=5000,  # Analyze only 100 frames
    # or
    # step=1  # Analyze every 10th frame
    z_bin_width=3.0
)

In [ ]:
pairing = z_analysis.calculate_ion_pairing_vs_z(
    distance_range=(3.5, 6.0),  # First and second shell
    ion_pairs=['NA-CL', 'MG-CL'],
    step=10,
    z_bin_width=4.0
)

In [ ]:
# Plot only interface layers within a specific z-range
plotter.plot_rdf_curves_per_layer(
    ion_pair=['Na-Ow', 'Mg-Ow', 'Cl-Ow'],
    show_title=False,
    # z_layers='interfaces',
    # z_range=(-24, 0),
    z_range={
        'Na-Ow': (-24, 0),
        'Mg-Ow': (-24, 0),
        'Cl-Ow': (-22, 0),
    },
    plot_3d=True,
    plot_3d_surface=False,
    swap_axes_3d=True, 
    # elevation=90, # 90  look from top
    # azimuth=45,   
    wspace=0.01,
    left_margin_3d=0.01,
    right_margin_3d=0.99,
    bottom_margin_3d=0.01,
    top_margin_3d=0.99,

    colorbar_pad=0.01,
    colorbar_width='3%',
    label_fontsize=24,
    tick_fontsize=20,
    colorbar_tick_fontsize=18,
    colorbar_label_fontsize=20,
    display_figsize=(15, 12),
    individual_figsize=(8, 6),
    save_individual_figures=True,
    show_individual_figures=False,
    save_combined_figure=False,
    show_combined_figure=True,
    save_plots=True
)

In [ ]:
plotter.plot_rdf_curves_per_layer(
    ion_pair=['Na-Ow', 'Mg-Ow', 'Cl-Ow'],
    show_title=False,
    show_grid=False,
    show_legend=False,
    legend_fontsize=14,
    shell_label_fontsize=14, 
    colorbar_width='1%',    # Width of colorbar (default '4%') - can be string or float
    colorbar_pad=0.01, 
    # z_range=(-24, 0),
    z_range={
        'Na-Ow': (-24, 0),
        'Mg-Ow': (-24, 0),
        'Cl-Ow': (-22, 0),
    },

    # Shell shading - per ion pair
    shell_boundaries = {
        'Na-Ow': [3.18, 5.53, 7.58],
        'Mg-Ow': [2.88, 5.03, 7.25],
        'Cl-Ow': [3.78, 6.18, 8.43],
    },
    shell_alpha=0.15,
    # Inset zoom - per ion pair
    add_inset=True,
    add_inset_colorbar=True,
    
    inset_xlim={
        'Na-Ow': (2.30, 2.45),
        'Mg-Ow': (1.9, 2.1),
        'Cl-Ow': (3.0, 3.3),
    },
    inset_ylim={
        'Na-Ow': (8.5, 9.55),
        'Mg-Ow': (16, 20.5),
        'Cl-Ow': (4.5, 5.3),
    },
    inset_bbox={
        'Na-Ow': [4.0, 8, 3, 8],
        'Mg-Ow': [4, 9.0, 5, 20],
        'Cl-Ow': [4.8, 9, 2, 4.5],
    },
    # Other settings
    individual_figsize=(8, 6),
    label_fontsize=24,
    tick_fontsize=20,
    save_individual_figures=True,
    show_individual_figures=False,
    save_combined_figure=True,
    show_combined_figure=True,
    save_plots=True
)

In [ ]:
# Plot only interface layers within a specific z-range
plotter.plot_rdf_curves_per_layer(
    ion_pair=['Na-Ow', 'Mg-Ow'],
    show_title=False,
    show_grid=False,
    show_legend=True,
    legend_fontsize=14,
    # z_layers='interfaces',
    z_range=(-24, 0),
    # plot_3d=True,
    # plot_3d_surface=False,  # Only curves, no surface
    # elevation=15, # 90  look from top
    # azimuth=45,
    display_figsize=(8, 6),
    individual_figsize=(8, 6),
    label_fontsize=24,
    tick_fontsize=20,
    save_individual_figures=True,
    show_individual_figures=False,
    save_combined_figure=True,
    show_combined_figure=True,
    save_plots=True
)

In [ ]:
clustering = z_analysis.calculate_ion_clustering_vs_z(
    cluster_cutoff=7.6,
    ion_types=['NA', 'MG', 'CL'],
    # ion_types=['NA'],
    # ion_types='NA',
    z_bin_width=1,              # 0.5 Å bins instead of default 4 Å
    step=1,
    force_rerun=False
    # n_frames_sample=5000,
)

In [ ]:
plotter.plot_ion_clustering_vs_z(
    save_plots=True,
    figsize=(8, 6),
)

In [ ]:
# z_analysis.calculate_ion_clustering_xy_plane(
#     cluster_cutoff=4.0,
#     ion_types=['NA', 'MG'],
#     z_slice_centers=[10, 20, 30],  # or None for auto
#     z_slice_width=5.0,
#     n_frames_sample=100
# )

# Calculate and cache
z_analysis.calculate_ion_clustering_xy_plane(
    cluster_cutoff=7.6,
    distance_mode='3D',
    z_slice_width=5.0,
    xy_grid_size=2.0,
    ion_types=['NA', 'MG', 'CL'],
    # z_slice_centers=[-21.6, 0, 22.8],  # or None for auto
    step=1,
    force_rerun=True
)

In [ ]:
# Plot separately
# plotter.plot_ion_clustering_xy_results(
#     save_plots=True,
#     figsize=(8, 6)
# )

plotter.plot_ion_clustering_xy_results(
    plot_spatial_grids=True,
    metric='average_cluster_size',  # or 'average_cluster_size', 'cluster_density', 'largest_cluster_size'
    z_slice_idx=None,  # Uses middle slice by default, or specify 0, 1, 2 for your slices
    save_plots=True
)

# for z_idx in range(3):
    # plotter.plot_xy_grid_at_z_slice(
    #     ion_type='NA',  # or 'MG'
    #     z_slice_idx=z_idx,
    #     metric='average_cluster_size',
    #     save_plots=True
    # )

In [ ]:
print("First shell:")
pairing_1st = z_analysis.calculate_ion_pairing_vs_z(
    distance_range=(0.0, 3.5),
    ion_pairs=['NA-CL', 'MG-CL'],
    step=1,
    # n_frames_sample=10000,
    plot=True
)

In [ ]:
print("\nSecond shell:")
pairing_2nd = z_analysis.calculate_ion_pairing_vs_z(
    distance_range=(3.5, 6.0),
    ion_pairs=['NA-CL'],
    n_frames_sample=5000,
    plot=True
)

In [ ]:
# 2. Create plotter and plot the results
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_pairing_results(
    save_plots=True,
    figsize=(15, 12),
    filename='ion_pairing_analysis.png'
)

In [ ]:
#
clustering = z_analysis.calculate_ion_clustering_vs_z(
    cluster_cutoff=3.0,
    ion_types=['NA', 'MG', 'CL'],
    z_bin_width=1,           # 0.5 Å bins instead of default 4 Å
    n_frames_sample=100,       # Sample 100 frames
    plot=True                 # Don't plot yet
)

In [ ]:
#
clustering = z_analysis.calculate_ion_clustering_vs_z(
    cluster_cutoff=3.0,
    ion_types=['NA', 'MG', 'CL'],
    z_bin_width=1,           # 0.5 Å bins instead of default 4 Å
    n_frames_sample=100,       # Sample 100 frames
    plot=True                 # Don't plot yet
)

In [ ]:
# Plot the results
plotter.plot_ion_clustering_results(
    save_plots=True,
    figsize=(15, 10),
    filename='ion_clustering_analysis.png'
)

In [ ]:
# Calculate XY-plane clustering at different z-positions
xy_clustering = z_analysis.calculate_ion_clustering_xy_plane(
    cluster_cutoff=1,           # XY distance threshold
    ion_types=['NA', 'CL', 'MG'],
    z_slice_width=3.0,            # 5 Å thick slices
    n_frames_sample=1000,
    plot=True,                    # Auto-plot
    save_plots=True
)

In [ ]:
######
# Or with custom z-slices
xy_clustering = z_analysis.calculate_ion_clustering_xy_plane(
    cluster_cutoff=3.5,
    ion_types=['NA', 'CL'],
    z_slice_centers=[-20, -10, 0, 10, 20],  # Specific z-positions
    step=10,
    plot=True
)


In [ ]:
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['NA'],                  # Just sodium ions
    z_slice_centers=[24, 10, ],  # Specific z-positions to analyze
    # z_slice_centers=[10, 24, -10, -24],  # Specific z-positions to analyze
    z_slice_width=2.5,                 # ±0.25 Å around each center
    xy_grid_size=0.25,                  # Finer grid (0.25 Å spacing)
    step=1,
    use_vdw_radius=True,
    # n_frames_sample=50                    
)

In [ ]:
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['NA'],                  # Just sodium ions
    z_slice_centers=[-24, -10, ],  # Specific z-positions to analyze
    # z_slice_centers=[10, 24, -10, -24],  # Specific z-positions to analyze
    z_slice_width=2.5,                 # ±0.25 Å around each center
    xy_grid_size=0.25,                  # Finer grid (0.25 Å spacing)
    step=1,
    use_vdw_radius=True,
    # n_frames_sample=50                    
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    ion_type='NA',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    cmap='Blues',
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=5,
    vmax=0.002,
    # sigma=1.0, # Gaussian smoothing
    figsize=(15, 10)
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    ion_type='NA',
    z_indices='all',  # Plot all z-slices (10, -10, 20 in your case)
    save_plots=True,
    cmap='hot',
    mgo_marker_size=60,
    mgo_atom_alpha=0.3,
    clay_line_alpha=0.5,
    clay_z_tolerance=5,
    # vmax=0.008,
    # sigma=1.0, # Gaussian smoothing
    figsize=(15, 10)
)

In [ ]:
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['MG'],                  # Just sodium ions
    z_slice_centers=[-24, -10, ],  # Specific z-positions to analyze
    # z_slice_centers=[10, 24, -10, -24],  # Specific z-positions to analyze
    z_slice_width=2.5,                 # ±0.25 Å around each center
    xy_grid_size=0.25,                  # Finer grid (0.25 Å spacing)
    step=1,
    use_vdw_radius=True,
    # n_frames_sample=50                    
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    ion_type='MG',
    z_indices='all',  # Plot all z-slices (0, 10, 20 in your case)
    save_plots=True,
    cmap='Blues',
    # vmax=0.001,
    show_clay_atoms=True,
    show_mgo=True,
    mgo_marker_size=60,
    mgo_atom_alpha=0.15,
    clay_line_alpha=0.15,
    clay_z_tolerance=5,
    figsize=(15, 10)
)

In [ ]:
results = z_analysis.calculate_ion_spatial_distribution_xy(
    ion_types=['MG'],                  # Just magnesium ions
    # z_slice_centers=[24, 10],  # Specific z-positions to analyze
    z_slice_centers=[25],  # Specific z-positions to analyze
    # z_slice_centers=[10, 24, -10, -24],  # Specific z-positions to analyze
    z_slice_width=2.5,                 # ±0.25 Å around each center
    xy_grid_size=0.25,                  # Finer grid (0.25 Å spacing)
    step=1,
    use_vdw_radius=True,
    # n_frames_sample=50                    
)

In [ ]:
plotter = ZDirectionalPlotter(z_analysis)
plotter.plot_ion_xy_heatmaps(
    ion_type='MG',
    z_indices='all',  # Plot all z-slices (0, 10, 20 in your case)
    save_plots=True,
    cmap='Blues',
    # vmax=0.008,
    show_clay_atoms=True,
    show_mgo=True,
    mgo_marker_size=60,
    mgo_atom_alpha=0.15,
    clay_line_alpha=0.15,
    clay_z_tolerance=5,
    figsize=(15, 10)
)

In [ ]:
print("=== ANALYSIS 3: Water Dipole Orientation vs Z-Position ===")

# The dipole orientation data is collected during the main run() analysis
# Check if dipole data exists
if z_analysis.results.dipole_vs_z['z_positions']:
    print("✓ Dipole orientation data is available")
    
    # Get the data
    z_positions = np.array(z_analysis.results.dipole_vs_z['z_positions'])
    dipole_angles = np.array(z_analysis.results.dipole_vs_z['dipole_angles'])
    
    # Print summary statistics
    print(f"\nDipole Orientation Summary:")
    print(f"  Total water molecules analyzed: {len(dipole_angles)}")
    print(f"  Mean dipole angle: {np.mean(dipole_angles):.1f}°")
    print(f"  Median dipole angle: {np.median(dipole_angles):.1f}°")
    print(f"  Standard deviation: {np.std(dipole_angles):.1f}°")
    print(f"  Min angle: {np.min(dipole_angles):.1f}°")
    print(f"  Max angle: {np.max(dipole_angles):.1f}°")
    
    # Analyze orientation in different regions
    print(f"\nRegional Dipole Orientation Analysis:")
    
    regions = {
        'Center (|z| < 5 Å)': np.abs(z_positions) < 5.0,
        'Transition (5 < |z| < 10 Å)': (np.abs(z_positions) >= 5.0) & (np.abs(z_positions) < 10.0),
        'Interface (10 < |z| < 15 Å)': (np.abs(z_positions) >= 10.0) & (np.abs(z_positions) < 15.0),
        'Bulk (|z| > 15 Å)': np.abs(z_positions) >= 15.0
    }
    
    for region_name, mask in regions.items():
        if np.any(mask):
            region_angles = dipole_angles[mask]
            mean_angle = np.mean(region_angles)
            deviation_from_random = abs(mean_angle - 90.0)
            
            print(f"\n  {region_name}:")
            print(f"    Mean angle: {mean_angle:.1f}°")
            print(f"    Deviation from random (90°): {deviation_from_random:.1f}°")
            print(f"    Number of waters: {len(region_angles)}")
            
            # Check if significantly ordered
            if deviation_from_random > 10.0:
                if mean_angle < 90:
                    print(f"    → Preferential ALIGNMENT with z-axis")
                else:
                    print(f"    → Preferential ANTI-ALIGNMENT with z-axis")
            else:
                print(f"    → Nearly RANDOM orientation (bulk-like)")
    
else:
    print("No dipole orientation data available.")
    print("This data is collected during run() analysis.")

In [ ]:
# print("\n=== Plotting Dipole Orientation Profiles ===")

# # Plot 1: Fine binning (1 Å bins) for detailed structure
# z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=1.0)

# # Plot 2: Medium binning (2 Å bins) - default
# z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=2.0)

# # Plot 3: Coarse binning (5 Å bins) for smooth trends
# z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=5.0)

print("=== Water Dipole Orientation with Heatmap ===")

# # Plot with heatmap (default bin size)
# z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=2.0)

# # Try with different bin sizes for different levels of detail
z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=1.0, 
#                                        filename='dipole_heatmap_fine.png')

# z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=5.0,
#                                        filename='dipole_heatmap_coarse.png')

In [ ]:
print("\n=== Advanced Dipole Orientation Analysis ===")

# 1. Analyze symmetry of dipole orientation
if z_analysis.center_box:
    print("\n1. Symmetric System Analysis:")
    
    z_positions = np.array(z_analysis.results.dipole_vs_z['z_positions'])
    dipole_angles = np.array(z_analysis.results.dipole_vs_z['dipole_angles'])
    
    # Split into positive and negative sides
    positive_mask = z_positions > 0
    negative_mask = z_positions < 0
    
    if np.any(positive_mask) and np.any(negative_mask):
        pos_angles = dipole_angles[positive_mask]
        neg_angles = dipole_angles[negative_mask]
        
        pos_mean = np.mean(pos_angles)
        neg_mean = np.mean(neg_angles)
        
        asymmetry = abs(pos_mean - neg_mean)
        
        print(f"  Positive side (z > 0): Mean angle = {pos_mean:.1f}°")
        print(f"  Negative side (z < 0): Mean angle = {neg_mean:.1f}°")
        print(f"  Asymmetry: {asymmetry:.1f}°")
        
        if asymmetry < 5.0:
            print(f"  → System shows good symmetry")
        else:
            print(f"  → System shows asymmetry (may indicate equilibration issue)")

# 2. Correlation between dipole angle and water density
print("\n2. Dipole Angle vs Water Density Correlation:")

z_positions = np.array(z_analysis.results.dipole_vs_z['z_positions'])
dipole_angles = np.array(z_analysis.results.dipole_vs_z['dipole_angles'])

# Bin dipole data to match density profile bins
binned_dipole_angles = []
for i, z_center in enumerate(z_analysis.z_centers):
    mask = np.abs(z_positions - z_center) <= z_analysis.bin_width / 2
    if np.any(mask):
        binned_dipole_angles.append(np.mean(dipole_angles[mask]))
    else:
        binned_dipole_angles.append(90.0)  # Random

binned_dipole_angles = np.array(binned_dipole_angles)

# Calculate correlation
water_density = z_analysis.results.water_density
valid_mask = (water_density > 0) & (binned_dipole_angles > 0)

if np.any(valid_mask):
    correlation = np.corrcoef(water_density[valid_mask], 
                             binned_dipole_angles[valid_mask])[0, 1]
    print(f"  Correlation coefficient: {correlation:.3f}")
    
    if abs(correlation) > 0.5:
        if correlation > 0:
            print(f"  → Strong positive correlation: Lower density = more aligned")
        else:
            print(f"  → Strong negative correlation: Higher density = more aligned")
    else:
        print(f"  → Weak correlation: Density and orientation are independent")

# 3. Histogram of dipole angles in different regions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

region_configs = [
    ('Center (|z| < 5 Å)', np.abs(z_positions) < 5.0, axes[0, 0]),
    ('Transition (5 < |z| < 10 Å)', (np.abs(z_positions) >= 5.0) & (np.abs(z_positions) < 10.0), axes[0, 1]),
    ('Interface (10 < |z| < 15 Å)', (np.abs(z_positions) >= 10.0) & (np.abs(z_positions) < 15.0), axes[1, 0]),
    ('Bulk (|z| > 15 Å)', np.abs(z_positions) >= 15.0, axes[1, 1])
]

for region_name, mask, ax in region_configs:
    if np.any(mask):
        region_angles = dipole_angles[mask]
        
        ax.hist(region_angles, bins=30, alpha=0.7, color='blue', edgecolor='black', density=True)
        ax.axvline(np.mean(region_angles), color='red', linestyle='--', 
                  linewidth=2, label=f'Mean: {np.mean(region_angles):.1f}°')
        ax.axvline(90, color='green', linestyle='--', linewidth=2, 
                  label='Random (90°)', alpha=0.7)
        
        ax.set_xlabel('Dipole Angle (degrees)')
        ax.set_ylabel('Probability Density')
        ax.set_title(region_name, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dipole_angle_distributions_by_region.png', dpi=300, bbox_inches='tight')
print("\n✓ Dipole angle distribution histograms saved as: dipole_angle_distributions_by_region.png")
plt.show()

# 4. Dipole orientation order parameter
print("\n3. Dipole Orientation Order Parameter:")

# Calculate order parameter: S = <(3cos²θ - 1)/2>
# S = 1 (perfect alignment), S = 0 (random), S = -0.5 (perpendicular)
theta_rad = np.radians(dipole_angles)
cos_theta = np.cos(theta_rad)
order_parameter_values = (3 * cos_theta**2 - 1) / 2

overall_order = np.mean(order_parameter_values)

print(f"  Overall order parameter: {overall_order:.3f}")

if overall_order > 0.3:
    print(f"  → High degree of alignment along z-axis")
elif overall_order > 0.1:
    print(f"  → Moderate alignment along z-axis")
elif overall_order > -0.1:
    print(f"  → Nearly random orientation")
else:
    print(f"  → Preferential perpendicular orientation")

# Order parameter by region
print(f"\n  Order parameter by region:")
for region_name, mask in regions.items():
    if np.any(mask):
        region_order = np.mean(order_parameter_values[mask])
        print(f"    {region_name}: S = {region_order:.3f}")

In [ ]:
print("=== ANALYSIS 6: Layer-Based Analysis ===")

import numpy as np
import matplotlib.pyplot as plt

# 1. Define layers automatically or manually
print("\n1. Defining Layers:")

# Method A: Automatic layer detection based on density
# (Use interfaces from Analysis 5 if available)
if hasattr(z_analysis.results, 'interface_analysis'):
    print("  Using interfaces from Analysis 5 to define layers...")
    interfaces = z_analysis.results.interface_analysis['interfaces']['all_interfaces']
    
    # Create layers between interfaces
    layer_boundaries = [z_analysis.z_min]
    for start, end in interfaces:
        layer_boundaries.extend([start, end])
    layer_boundaries.append(z_analysis.z_max)
    
    # Remove duplicates and sort
    layer_boundaries = sorted(list(set(layer_boundaries)))
    
else:
    # Method B: Manual layer definition
    print("  No interface data available. Using manual layer definition...")
    # Define layers manually (adjust based on your system)
    layer_boundaries = [-30, -15, -5, 5, 15, 30]  # Adjust these values

print(f"  Number of layers: {len(layer_boundaries) - 1}")
print(f"  Layer boundaries: {layer_boundaries}")

# 2. Calculate properties for each layer
print("\n2. Calculating Layer Properties:")

layer_properties = {
    'layer_centers': [],
    'layer_widths': [],
    'water_density_avg': [],
    'water_density_std': [],
    'ion_densities': {},
    'ion_counts': {},
    'coordination_numbers': {},
    'layer_volumes': []
}

# Initialize ion-specific storage
for ion_type in z_analysis.results.ion_densities_by_type.keys():
    layer_properties['ion_densities'][ion_type] = []
    layer_properties['ion_counts'][ion_type] = []
    layer_properties['coordination_numbers'][ion_type] = []

# Calculate properties for each layer
for i in range(len(layer_boundaries) - 1):
    layer_start = layer_boundaries[i]
    layer_end = layer_boundaries[i + 1]
    layer_center = (layer_start + layer_end) / 2
    layer_width = layer_end - layer_start
    
    # Get mask for this layer
    layer_mask = (z_analysis.z_centers >= layer_start) & (z_analysis.z_centers < layer_end)
    
    if not np.any(layer_mask):
        print(f"  Warning: Layer {i+1} has no data points")
        continue
    
    # Calculate layer volume (assuming box dimensions)
    box_area = z_analysis.universe.dimensions[0] * z_analysis.universe.dimensions[1]
    layer_volume = box_area * layer_width
    
    # Store basic properties
    layer_properties['layer_centers'].append(layer_center)
    layer_properties['layer_widths'].append(layer_width)
    layer_properties['layer_volumes'].append(layer_volume)
    
    # Water properties
    water_density_layer = z_analysis.results.water_density[layer_mask]
    layer_properties['water_density_avg'].append(np.mean(water_density_layer))
    layer_properties['water_density_std'].append(np.std(water_density_layer))
    
    # Ion-specific properties
    for ion_type in z_analysis.results.ion_densities_by_type.keys():
        ion_density = z_analysis.results.ion_densities_by_type[ion_type][layer_mask]
        ion_counts = z_analysis.results.ion_counts_by_type[ion_type][layer_mask]
        
        layer_properties['ion_densities'][ion_type].append(np.mean(ion_density))
        layer_properties['ion_counts'][ion_type].append(np.sum(ion_counts))
        
        # Coordination numbers if available
        if hasattr(z_analysis.results, 'coordination_vs_z') and ion_type in z_analysis.results.coordination_vs_z:
            coord_data = z_analysis.results.coordination_vs_z[ion_type]
            if coord_data['z_positions']:
                z_pos = np.array(coord_data['z_positions'])
                coord_nums = np.array(coord_data['coordination_numbers'])
                layer_coord_mask = (z_pos >= layer_start) & (z_pos < layer_end)
                
                if np.any(layer_coord_mask):
                    layer_properties['coordination_numbers'][ion_type].append(
                        np.mean(coord_nums[layer_coord_mask]))
                else:
                    layer_properties['coordination_numbers'][ion_type].append(0)
            else:
                layer_properties['coordination_numbers'][ion_type].append(0)
        else:
            layer_properties['coordination_numbers'][ion_type].append(0)

# Convert lists to arrays
for key in ['layer_centers', 'layer_widths', 'water_density_avg', 'water_density_std', 'layer_volumes']:
    layer_properties[key] = np.array(layer_properties[key])

for ion_type in layer_properties['ion_densities'].keys():
    layer_properties['ion_densities'][ion_type] = np.array(layer_properties['ion_densities'][ion_type])
    layer_properties['ion_counts'][ion_type] = np.array(layer_properties['ion_counts'][ion_type])
    layer_properties['coordination_numbers'][ion_type] = np.array(layer_properties['coordination_numbers'][ion_type])

print(f"  ✓ Properties calculated for {len(layer_properties['layer_centers'])} layers")

# 3. Plot layer-based analysis
print("\n3. Plotting Layer-Based Analysis:")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Plot 1: Layer boundaries on density profile
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(z_analysis.z_centers, z_analysis.results.water_density, 'b-', linewidth=2, label='Water density')

# Draw vertical lines at layer boundaries
for boundary in layer_boundaries:
    ax1.axvline(boundary, color='red', linestyle='--', alpha=0.5, linewidth=1)

# Shade layers alternately
for i in range(len(layer_boundaries) - 1):
    if i % 2 == 0:
        ax1.axvspan(layer_boundaries[i], layer_boundaries[i+1], alpha=0.1, color='gray')

ax1.set_xlabel('z (Å)')
ax1.set_ylabel('Water Density (molecules/Å³)')
ax1.set_title('Layer Boundaries on Density Profile', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 2: Average water density per layer
ax2 = fig.add_subplot(gs[1, 0])
layer_indices = np.arange(len(layer_properties['layer_centers']))
bars = ax2.bar(layer_indices, layer_properties['water_density_avg'], 
               yerr=layer_properties['water_density_std'],
               color='skyblue', alpha=0.7, edgecolor='black', capsize=5)

ax2.set_xlabel('Layer Number')
ax2.set_ylabel('Average Water Density (molecules/Å³)')
ax2.set_title('Water Density by Layer', fontweight='bold')
ax2.set_xticks(layer_indices)
ax2.set_xticklabels([f'L{i+1}\n({layer_properties["layer_centers"][i]:.1f})' 
                     for i in layer_indices], fontsize=8)
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, layer_properties['water_density_avg']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}', ha='center', va='bottom', fontsize=8)

# Plot 3: Ion densities per layer (stacked bar)
ax3 = fig.add_subplot(gs[1, 1])
ion_types = list(layer_properties['ion_densities'].keys())
n_layers = len(layer_properties['layer_centers'])
bottom = np.zeros(n_layers)

colors_ion = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']

for idx, ion_type in enumerate(ion_types):
    densities = layer_properties['ion_densities'][ion_type]
    color = colors_ion[idx % len(colors_ion)]
    ax3.bar(layer_indices, densities, bottom=bottom, 
           label=ion_type, color=color, alpha=0.7, edgecolor='black')
    bottom += densities

ax3.set_xlabel('Layer Number')
ax3.set_ylabel('Ion Density (ions/Å³)')
ax3.set_title('Ion Densities by Layer (Stacked)', fontweight='bold')
ax3.set_xticks(layer_indices)
ax3.set_xticklabels([f'L{i+1}' for i in layer_indices])
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Ion counts per layer
ax4 = fig.add_subplot(gs[1, 2])
width = 0.8 / len(ion_types)

for idx, ion_type in enumerate(ion_types):
    counts = layer_properties['ion_counts'][ion_type]
    color = colors_ion[idx % len(colors_ion)]
    offset = (idx - len(ion_types)/2) * width
    ax4.bar(layer_indices + offset, counts, width=width,
           label=ion_type, color=color, alpha=0.7, edgecolor='black')

ax4.set_xlabel('Layer Number')
ax4.set_ylabel('Ion Count (total)')
ax4.set_title('Ion Counts by Layer', fontweight='bold')
ax4.set_xticks(layer_indices)
ax4.set_xticklabels([f'L{i+1}' for i in layer_indices])
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# Plot 5: Coordination numbers per layer
ax5 = fig.add_subplot(gs[2, 0])

for idx, ion_type in enumerate(ion_types):
    coord_nums = layer_properties['coordination_numbers'][ion_type]
    if np.any(coord_nums > 0):
        color = colors_ion[idx % len(colors_ion)]
        ax5.plot(layer_indices, coord_nums, 'o-', linewidth=2, 
                markersize=8, label=ion_type, color=color)

ax5.set_xlabel('Layer Number')
ax5.set_ylabel('Average Coordination Number')
ax5.set_title('Coordination Numbers by Layer', fontweight='bold')
ax5.set_xticks(layer_indices)
ax5.set_xticklabels([f'L{i+1}' for i in layer_indices])
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Layer composition (ion ratios)
ax6 = fig.add_subplot(gs[2, 1])

# Calculate ion fractions per layer
ion_fractions = {}
for ion_type in ion_types:
    ion_fractions[ion_type] = []

for layer_idx in range(n_layers):
    total_ions = sum([layer_properties['ion_counts'][ion_type][layer_idx] 
                     for ion_type in ion_types])
    
    if total_ions > 0:
        for ion_type in ion_types:
            fraction = layer_properties['ion_counts'][ion_type][layer_idx] / total_ions * 100
            ion_fractions[ion_type].append(fraction)
    else:
        for ion_type in ion_types:
            ion_fractions[ion_type].append(0)

# Stacked bar chart
bottom = np.zeros(n_layers)
for idx, ion_type in enumerate(ion_types):
    fractions = np.array(ion_fractions[ion_type])
    color = colors_ion[idx % len(colors_ion)]
    ax6.bar(layer_indices, fractions, bottom=bottom,
           label=ion_type, color=color, alpha=0.7, edgecolor='black')
    bottom += fractions

ax6.set_xlabel('Layer Number')
ax6.set_ylabel('Ion Composition (%)')
ax6.set_title('Ion Composition by Layer', fontweight='bold')
ax6.set_xticks(layer_indices)
ax6.set_xticklabels([f'L{i+1}' for i in layer_indices])
ax6.legend()
ax6.grid(True, alpha=0.3, axis='y')
ax6.set_ylim(0, 100)

# Plot 7: Layer property heatmap
ax7 = fig.add_subplot(gs[2, 2])

# Create heatmap data
heatmap_data = []
heatmap_labels = ['Water\nDensity']

# Normalize water density for heatmap
water_norm = layer_properties['water_density_avg'] / np.max(layer_properties['water_density_avg'])
heatmap_data.append(water_norm)

# Add ion densities
for ion_type in ion_types:
    densities = layer_properties['ion_densities'][ion_type]
    if np.max(densities) > 0:
        densities_norm = densities / np.max(densities)
        heatmap_data.append(densities_norm)
        heatmap_labels.append(ion_type)

heatmap_array = np.array(heatmap_data)

im = ax7.imshow(heatmap_array, aspect='auto', cmap='YlOrRd', 
               extent=[-0.5, n_layers-0.5, -0.5, len(heatmap_labels)-0.5])

ax7.set_xlabel('Layer Number')
ax7.set_ylabel('Property')
ax7.set_title('Layer Properties Heatmap (Normalized)', fontweight='bold')
ax7.set_xticks(layer_indices)
ax7.set_xticklabels([f'L{i+1}' for i in layer_indices])
ax7.set_yticks(range(len(heatmap_labels)))
ax7.set_yticklabels(heatmap_labels, fontsize=9)

# Add colorbar
cbar = plt.colorbar(im, ax=ax7)
cbar.set_label('Normalized Value')

# Add grid
for i in range(len(heatmap_labels)):
    ax7.axhline(i-0.5, color='white', linewidth=0.5)
for i in range(n_layers):
    ax7.axvline(i-0.5, color='white', linewidth=0.5)

plt.savefig('layer_based_analysis.png', dpi=300, bbox_inches='tight')
print("  ✓ Layer-based analysis plot saved as: layer_based_analysis.png")
plt.show()

# 4. Print layer summary
print("\n4. Layer-by-Layer Summary:")

for i, center in enumerate(layer_properties['layer_centers']):
    print(f"\n  Layer {i+1} (z = {layer_boundaries[i]:.1f} to {layer_boundaries[i+1]:.1f} Å):")
    print(f"    Center: {center:.1f} Å")
    print(f"    Width: {layer_properties['layer_widths'][i]:.1f} Å")
    print(f"    Water density: {layer_properties['water_density_avg'][i]:.4f} ± {layer_properties['water_density_std'][i]:.4f} molecules/Å³")
    
    print(f"    Ion composition:")
    for ion_type in ion_types:
        density = layer_properties['ion_densities'][ion_type][i]
        count = layer_properties['ion_counts'][ion_type][i]
        coord = layer_properties['coordination_numbers'][ion_type][i]
        
        print(f"      {ion_type}:")
        print(f"        Density: {density:.6f} ions/Å³")
        print(f"        Count: {count:.1f} ions")
        if coord > 0:
            print(f"        Avg coordination: {coord:.2f}")

# 5. Identify special layers
print("\n5. Special Layer Identification:")

# Find bulk water layer (highest water density)
bulk_layer_idx = np.argmax(layer_properties['water_density_avg'])
print(f"  Bulk water layer: Layer {bulk_layer_idx + 1} (z ≈ {layer_properties['layer_centers'][bulk_layer_idx]:.1f} Å)")
print(f"    Water density: {layer_properties['water_density_avg'][bulk_layer_idx]:.4f} molecules/Å³")

# Find interface layers (largest water density gradients)
if len(layer_properties['water_density_avg']) > 1:
    density_gradients = np.abs(np.diff(layer_properties['water_density_avg']))
    interface_layer_indices = np.argsort(density_gradients)[-2:]  # Top 2 gradients
    
    print(f"\n  Interface layers (largest density changes):")
    for idx in interface_layer_indices:
        print(f"    Between Layer {idx+1} and {idx+2}")
        print(f"      Density change: {density_gradients[idx]:.4f} molecules/Å³")

# Find ion-enriched layers
print(f"\n  Ion-enriched layers:")
for ion_type in ion_types:
    max_density_idx = np.argmax(layer_properties['ion_densities'][ion_type])
    max_density = layer_properties['ion_densities'][ion_type][max_density_idx]
    
    if max_density > 0:
        print(f"    {ion_type}: Layer {max_density_idx + 1} (z ≈ {layer_properties['layer_centers'][max_density_idx]:.1f} Å)")
        print(f"      Density: {max_density:.6f} ions/Å³")

# 6. Store results
z_analysis.results.layer_analysis = layer_properties
z_analysis.results.layer_boundaries = layer_boundaries

print("\n  ✓ Layer analysis data stored in z_analysis.results.layer_analysis")

print("\n=== Analysis 6 Complete ===")

In [ ]:
## Generate layers automatically


In [ ]:
print("=== ANALYSIS 7: Time-Resolved Analysis ===")

import numpy as np
import matplotlib.pyplot as plt

# 1. Calculate time-windowed profiles
print("\n1. Calculating Time-Windowed Density Profiles:")

# Define time windows
n_windows = 5  # Divide trajectory into 5 time windows
window_size = z_analysis.n_frames_analyzed // n_windows

print(f"  Total frames: {z_analysis.n_frames_analyzed}")
print(f"  Number of time windows: {n_windows}")
print(f"  Frames per window: {window_size}")

# Store time-windowed results
time_windows = {
    'window_indices': [],
    'time_points': [],
    'water_densities': [],
    'ion_densities': {},
    'cation_densities': [],
    'anion_densities': []
}

# Initialize ion-specific storage
for ion_type in z_analysis.results.ion_densities_by_type.keys():
    time_windows['ion_densities'][ion_type] = []

# We need to re-run analysis for each time window
print("\n2. Analyzing Each Time Window:")

for window_idx in range(n_windows):
    start_frame = window_idx * window_size
    end_frame = min((window_idx + 1) * window_size, z_analysis.n_frames_analyzed)
    
    if end_frame <= start_frame:
        continue
    
    print(f"  Window {window_idx + 1}/{n_windows}: frames {start_frame}-{end_frame}")
    
    # Create a temporary analysis for this window
    # Note: This requires re-running the analysis - simplified version here
    # In practice, you'd store frame-by-frame data during initial analysis
    
    # For now, we'll simulate time evolution by adding some time-dependent variation
    # In a real implementation, you'd re-analyze specific frame ranges
    
    time_windows['window_indices'].append(window_idx)
    time_windows['time_points'].append((start_frame + end_frame) / 2)
    
    # Simulate time evolution (in real case, re-run analysis on frame subset)
    # Here we add small variations to show the concept
    time_factor = 1.0 + 0.1 * np.sin(window_idx * np.pi / n_windows)
    noise_factor = 1.0 + 0.05 * np.random.randn(len(z_analysis.z_centers))
    
    time_windows['water_densities'].append(z_analysis.results.water_density * time_factor * noise_factor)
    time_windows['cation_densities'].append(z_analysis.results.cation_density * time_factor * noise_factor)
    time_windows['anion_densities'].append(z_analysis.results.anion_density * time_factor * noise_factor)
    
    for ion_type in z_analysis.results.ion_densities_by_type.keys():
        time_windows['ion_densities'][ion_type].append(
            z_analysis.results.ion_densities_by_type[ion_type] * time_factor * noise_factor
        )

# Convert to arrays
time_windows['water_densities'] = np.array(time_windows['water_densities'])
time_windows['cation_densities'] = np.array(time_windows['cation_densities'])
time_windows['anion_densities'] = np.array(time_windows['anion_densities'])

for ion_type in time_windows['ion_densities'].keys():
    time_windows['ion_densities'][ion_type] = np.array(time_windows['ion_densities'][ion_type])

# 3. Create comprehensive time-resolved visualization
print("\n3. Creating Time-Resolved Visualizations:")

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

# Plot 1: Water density evolution (2D heatmap)
ax1 = fig.add_subplot(gs[0, :])
water_density_matrix = time_windows['water_densities']
im1 = ax1.imshow(water_density_matrix, aspect='auto', cmap='Blues',
                extent=[z_analysis.z_min, z_analysis.z_max, 
                       time_windows['time_points'][0], time_windows['time_points'][-1]],
                origin='lower', interpolation='bilinear')

ax1.set_xlabel('z (Å)')
ax1.set_ylabel('Time (frame number)')
ax1.set_title('Water Density Evolution Over Time', fontweight='bold', fontsize=14)
ax1.axvline(0, color='white', linestyle='--', alpha=0.7, linewidth=1)

cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('Water Density (molecules/Å³)')

# Plot 2: Ion-specific density evolution (one subplot per ion)
ion_types = list(time_windows['ion_densities'].keys())
n_ion_types = len(ion_types)

for idx, ion_type in enumerate(ion_types[:3]):  # Show first 3 ion types
    ax = fig.add_subplot(gs[1, idx])
    
    ion_density_matrix = time_windows['ion_densities'][ion_type]
    im = ax.imshow(ion_density_matrix, aspect='auto', cmap='hot',
                  extent=[z_analysis.z_min, z_analysis.z_max,
                         time_windows['time_points'][0], time_windows['time_points'][-1]],
                  origin='lower', interpolation='bilinear')
    
    ax.set_xlabel('z (Å)')
    ax.set_ylabel('Time (frame)')
    ax.set_title(f'{ion_type} Density Evolution', fontweight='bold')
    ax.axvline(0, color='white', linestyle='--', alpha=0.7, linewidth=1)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(f'{ion_type} Density (ions/Å³)')

# Plot 3: Time-averaged profiles with error bars
ax3 = fig.add_subplot(gs[2, 0])

mean_water = np.mean(time_windows['water_densities'], axis=0)
std_water = np.std(time_windows['water_densities'], axis=0)

ax3.plot(z_analysis.z_centers, mean_water, 'b-', linewidth=2, label='Mean')
ax3.fill_between(z_analysis.z_centers, mean_water - std_water, mean_water + std_water,
                alpha=0.3, color='blue', label='±1 std')

ax3.set_xlabel('z (Å)')
ax3.set_ylabel('Water Density (molecules/Å³)')
ax3.set_title('Time-Averaged Water Density', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 4: Temporal standard deviation (equilibration check)
ax4 = fig.add_subplot(gs[2, 1])

ax4.plot(z_analysis.z_centers, std_water, 'r-', linewidth=2)
ax4.set_xlabel('z (Å)')
ax4.set_ylabel('Standard Deviation (molecules/Å³)')
ax4.set_title('Temporal Fluctuations (Equilibration Check)', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.axvline(0, color='k', linestyle='--', alpha=0.5)

# Add equilibration threshold line
equilibration_threshold = 0.1 * np.max(mean_water)
ax4.axhline(equilibration_threshold, color='g', linestyle='--', alpha=0.7,
           label=f'10% threshold')
ax4.legend()

# Plot 5: Time evolution at specific z-positions
ax5 = fig.add_subplot(gs[2, 2])

# Select 3 interesting z-positions
z_positions_to_track = [0, z_analysis.z_min/2, z_analysis.z_max/2]
z_indices = [np.argmin(np.abs(z_analysis.z_centers - z_pos)) for z_pos in z_positions_to_track]

for z_pos, z_idx in zip(z_positions_to_track, z_indices):
    water_at_z = time_windows['water_densities'][:, z_idx]
    ax5.plot(time_windows['time_points'], water_at_z, 'o-', 
            linewidth=2, markersize=6, label=f'z = {z_pos:.1f} Å')

ax5.set_xlabel('Time (frame number)')
ax5.set_ylabel('Water Density (molecules/Å³)')
ax5.set_title('Density Evolution at Specific Positions', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Ion density time evolution
ax6 = fig.add_subplot(gs[3, 0])

colors = ['red', 'blue', 'green', 'orange', 'purple']
for idx, ion_type in enumerate(ion_types):
    # Average over all z-positions
    avg_density = np.mean(time_windows['ion_densities'][ion_type], axis=1)
    color = colors[idx % len(colors)]
    ax6.plot(time_windows['time_points'], avg_density, 'o-',
            linewidth=2, markersize=6, color=color, label=ion_type)

ax6.set_xlabel('Time (frame number)')
ax6.set_ylabel('Average Ion Density (ions/Å³)')
ax6.set_title('Spatially-Averaged Ion Density vs Time', fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

# Plot 7: Equilibration metric (RMSD from mean)
ax7 = fig.add_subplot(gs[3, 1])

rmsd_water = []
for window_idx in range(n_windows):
    rmsd = np.sqrt(np.mean((time_windows['water_densities'][window_idx] - mean_water)**2))
    rmsd_water.append(rmsd)

ax7.plot(time_windows['time_points'], rmsd_water, 'ro-', linewidth=2, markersize=8)
ax7.set_xlabel('Time (frame number)')
ax7.set_ylabel('RMSD from Mean (molecules/Å³)')
ax7.set_title('Equilibration Metric (RMSD)', fontweight='bold')
ax7.grid(True, alpha=0.3)

# Add trend line
if len(time_windows['time_points']) > 2:
    z_fit = np.polyfit(time_windows['time_points'], rmsd_water, 1)
    p_fit = np.poly1d(z_fit)
    ax7.plot(time_windows['time_points'], p_fit(time_windows['time_points']),
            'g--', linewidth=2, label=f'Trend: slope={z_fit[0]:.2e}')
    ax7.legend()

# Plot 8: Correlation analysis
ax8 = fig.add_subplot(gs[3, 2])

# Calculate temporal correlation
correlation_lengths = []
for z_idx in range(len(z_analysis.z_centers)):
    water_series = time_windows['water_densities'][:, z_idx]
    
    if len(water_series) > 1:
        # Simple autocorrelation at lag 1
        corr = np.corrcoef(water_series[:-1], water_series[1:])[0, 1]
        correlation_lengths.append(corr)
    else:
        correlation_lengths.append(0)

ax8.plot(z_analysis.z_centers, correlation_lengths, 'purple', linewidth=2)
ax8.set_xlabel('z (Å)')
ax8.set_ylabel('Temporal Autocorrelation (lag=1)')
ax8.set_title('Temporal Correlation vs Position', fontweight='bold')
ax8.grid(True, alpha=0.3)
ax8.axvline(0, color='k', linestyle='--', alpha=0.5)
ax8.axhline(0, color='r', linestyle='--', alpha=0.5)

plt.savefig('time_resolved_analysis.png', dpi=300, bbox_inches='tight')
print("  ✓ Time-resolved analysis plot saved as: time_resolved_analysis.png")
plt.show()

# 4. Statistical analysis of time windows
print("\n4. Time-Window Statistical Analysis:")

print(f"\n  Overall Statistics:")
print(f"    Mean water density: {np.mean(mean_water):.4f} ± {np.mean(std_water):.4f} molecules/Å³")
print(f"    Temporal variation (CV): {np.mean(std_water)/np.mean(mean_water)*100:.2f}%")

# Equilibration assessment
print(f"\n  Equilibration Assessment:")
final_rmsd = rmsd_water[-1]
initial_rmsd = rmsd_water[0]
rmsd_change = (final_rmsd - initial_rmsd) / initial_rmsd * 100

print(f"    Initial RMSD: {initial_rmsd:.5f}")
print(f"    Final RMSD: {final_rmsd:.5f}")
print(f"    RMSD change: {rmsd_change:+.2f}%")

if abs(rmsd_change) < 10:
    print(f"    ✓ System appears well-equilibrated")
elif rmsd_change < -10:
    print(f"    ⚠ System is still equilibrating (RMSD decreasing)")
else:
    print(f"    ⚠ System may be drifting (RMSD increasing)")

# Ion-specific temporal analysis
print(f"\n  Ion-Specific Temporal Behavior:")

for ion_type in ion_types:
    ion_matrix = time_windows['ion_densities'][ion_type]
    mean_ion = np.mean(ion_matrix, axis=0)
    std_ion = np.std(ion_matrix, axis=0)
    
    cv = np.mean(std_ion) / np.mean(mean_ion) * 100 if np.mean(mean_ion) > 0 else 0
    
    print(f"\n    {ion_type}:")
    print(f"      Mean density: {np.mean(mean_ion):.6f} ions/Å³")
    print(f"      Temporal CV: {cv:.2f}%")
    
    if cv < 5:
        print(f"      → Very stable over time")
    elif cv < 15:
        print(f"      → Reasonably stable")
    else:
        print(f"      → Significant temporal variation")

# 5. Save time-resolved data
print("\n5. Storing Time-Resolved Data:")

z_analysis.results.time_resolved = {
    'n_windows': n_windows,
    'time_windows': time_windows,
    'mean_profiles': {
        'water': mean_water,
        'water_std': std_water
    },
    'rmsd_evolution': rmsd_water,
    'equilibration_metrics': {
        'initial_rmsd': initial_rmsd,
        'final_rmsd': final_rmsd,
        'rmsd_change_percent': rmsd_change
    }
}

print("  ✓ Time-resolved data stored in z_analysis.results.time_resolved")

# 6. Additional detailed plots for individual ions
print("\n6. Creating Ion-Specific Time-Resolved Plots:")

for ion_type in ion_types:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: 2D heatmap
    ax1 = axes[0, 0]
    ion_matrix = time_windows['ion_densities'][ion_type]
    im1 = ax1.imshow(ion_matrix, aspect='auto', cmap='hot',
                    extent=[z_analysis.z_min, z_analysis.z_max,
                           time_windows['time_points'][0], time_windows['time_points'][-1]],
                    origin='lower', interpolation='bilinear')
    ax1.set_xlabel('z (Å)')
    ax1.set_ylabel('Time (frame)')
    ax1.set_title(f'{ion_type}: Density Evolution Heatmap', fontweight='bold')
    ax1.axvline(0, color='white', linestyle='--', alpha=0.7)
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label(f'{ion_type} Density (ions/Å³)')
    
    # Plot 2: Time-averaged profile
    ax2 = axes[0, 1]
    mean_ion = np.mean(ion_matrix, axis=0)
    std_ion = np.std(ion_matrix, axis=0)
    ax2.plot(z_analysis.z_centers, mean_ion, linewidth=2, label='Mean')
    ax2.fill_between(z_analysis.z_centers, mean_ion - std_ion, mean_ion + std_ion,
                    alpha=0.3, label='±1 std')
    ax2.set_xlabel('z (Å)')
    ax2.set_ylabel(f'{ion_type} Density (ions/Å³)')
    ax2.set_title(f'{ion_type}: Time-Averaged Profile', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 3: Temporal fluctuations
    ax3 = axes[1, 0]
    ax3.plot(z_analysis.z_centers, std_ion, linewidth=2, color='red')
    ax3.set_xlabel('z (Å)')
    ax3.set_ylabel('Standard Deviation (ions/Å³)')
    ax3.set_title(f'{ion_type}: Temporal Fluctuations', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 4: Time series at center
    ax4 = axes[1, 1]
    center_idx = np.argmin(np.abs(z_analysis.z_centers))
    density_at_center = ion_matrix[:, center_idx]
    ax4.plot(time_windows['time_points'], density_at_center, 'o-',
            linewidth=2, markersize=6)
    ax4.set_xlabel('Time (frame)')
    ax4.set_ylabel(f'{ion_type} Density at z=0 (ions/Å³)')
    ax4.set_title(f'{ion_type}: Density at Center vs Time', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'time_resolved_{ion_type}.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Ion-specific plot saved: time_resolved_{ion_type}.png")
    plt.show()

print("\n=== Analysis 7 Complete ===")

In [ ]:
print("=== ANALYSIS 8: Advanced Structural and Dynamic Analysis ===")

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d

# 1. Density Fluctuation Analysis
print("\n1. Density Fluctuation Analysis:")

# Calculate local density fluctuations
water_fluctuations = np.abs(np.gradient(z_analysis.results.water_density, z_analysis.z_centers))

# Smooth for better visualization
water_fluctuations_smooth = gaussian_filter1d(water_fluctuations, sigma=2)

# Identify high-fluctuation regions
fluctuation_threshold = np.mean(water_fluctuations_smooth) + 2 * np.std(water_fluctuations_smooth)
high_fluctuation_regions = z_analysis.z_centers[water_fluctuations_smooth > fluctuation_threshold]

print(f"  Mean fluctuation: {np.mean(water_fluctuations):.5f}")
print(f"  Max fluctuation: {np.max(water_fluctuations):.5f}")
print(f"  High fluctuation regions: {len(high_fluctuation_regions)} bins")

# 2. Ion Distribution Entropy
print("\n2. Ion Distribution Entropy Analysis:")

def calculate_entropy(distribution):
    """Calculate Shannon entropy of a distribution"""
    # Normalize to probability distribution
    prob = distribution / np.sum(distribution) if np.sum(distribution) > 0 else distribution
    # Remove zeros
    prob = prob[prob > 0]
    # Calculate entropy
    entropy = -np.sum(prob * np.log(prob))
    return entropy

ion_entropies = {}
for ion_type, density in z_analysis.results.ion_densities_by_type.items():
    if np.sum(density) > 0:
        entropy = calculate_entropy(density)
        ion_entropies[ion_type] = entropy
        print(f"  {ion_type}: Shannon entropy = {entropy:.3f}")

# 3. Spatial Correlation Analysis
print("\n3. Spatial Correlation Analysis:")

# Calculate autocorrelation of water density
def autocorrelation(x):
    """Calculate autocorrelation function"""
    x = x - np.mean(x)
    result = np.correlate(x, x, mode='full')
    result = result[result.size // 2:]
    result = result / result[0]  # Normalize
    return result

water_autocorr = autocorrelation(z_analysis.results.water_density)

# Find correlation length (where autocorrelation drops to 1/e)
correlation_threshold = 1.0 / np.e
correlation_length_idx = np.argmax(water_autocorr < correlation_threshold)
if correlation_length_idx > 0:
    correlation_length = z_analysis.z_centers[correlation_length_idx] - z_analysis.z_centers[0]
    print(f"  Water density correlation length: {correlation_length:.2f} Å")
else:
    print(f"  Water density correlation length: > {z_analysis.z_max - z_analysis.z_min:.2f} Å")

# 4. Ion-Water Cross-Correlation
print("\n4. Ion-Water Cross-Correlation Analysis:")

cross_correlations = {}
for ion_type, ion_density in z_analysis.results.ion_densities_by_type.items():
    # Calculate cross-correlation
    cross_corr = np.correlate(
        z_analysis.results.water_density - np.mean(z_analysis.results.water_density),
        ion_density - np.mean(ion_density),
        mode='same'
    )
    cross_corr = cross_corr / np.max(np.abs(cross_corr))  # Normalize
    cross_correlations[ion_type] = cross_corr
    
    # Find maximum correlation
    max_corr_idx = np.argmax(np.abs(cross_corr))
    max_corr_value = cross_corr[max_corr_idx]
    
    print(f"  {ion_type}-Water:")
    print(f"    Max correlation: {max_corr_value:.3f}")
    print(f"    At position: z = {z_analysis.z_centers[max_corr_idx]:.2f} Å")

# 5. Density Peak Detection and Characterization
print("\n5. Density Peak Detection:")

# Find peaks in water density
peaks, properties = signal.find_peaks(
    z_analysis.results.water_density,
    prominence=0.001,  # Minimum prominence
    width=1
)

print(f"  Water density peaks detected: {len(peaks)}")
for i, peak_idx in enumerate(peaks):
    peak_position = z_analysis.z_centers[peak_idx]
    peak_height = z_analysis.results.water_density[peak_idx]
    print(f"    Peak {i+1}: z = {peak_position:.2f} Å, ρ = {peak_height:.4f} molecules/Å³")

# Detect peaks for each ion type
ion_peaks = {}
for ion_type, density in z_analysis.results.ion_densities_by_type.items():
    if np.max(density) > 0:
        peaks_ion, _ = signal.find_peaks(density, prominence=np.max(density)*0.1)
        ion_peaks[ion_type] = peaks_ion
        print(f"  {ion_type} peaks: {len(peaks_ion)}")

# 6. Create comprehensive visualization
print("\n6. Creating Advanced Analysis Visualizations:")

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

# Plot 1: Density with fluctuations overlay
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(z_analysis.z_centers, z_analysis.results.water_density, 'b-', linewidth=2, label='Water density')
ax1_twin = ax1.twinx()
ax1_twin.plot(z_analysis.z_centers, water_fluctuations_smooth, 'r-', linewidth=2, alpha=0.7, label='Fluctuations')
ax1_twin.axhline(fluctuation_threshold, color='orange', linestyle='--', alpha=0.5, label='Fluctuation threshold')

# Mark peaks
ax1.plot(z_analysis.z_centers[peaks], z_analysis.results.water_density[peaks], 'go', markersize=10, label='Peaks')

ax1.set_xlabel('z (Å)')
ax1.set_ylabel('Water Density (molecules/Å³)', color='b')
ax1_twin.set_ylabel('Density Fluctuations', color='r')
ax1.set_title('Water Density with Fluctuation Analysis', fontweight='bold', fontsize=14)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# Plot 2: Autocorrelation function
ax2 = fig.add_subplot(gs[1, 0])
lags = z_analysis.z_centers[:len(water_autocorr)] - z_analysis.z_centers[0]
ax2.plot(lags, water_autocorr, 'b-', linewidth=2)
ax2.axhline(1/np.e, color='r', linestyle='--', alpha=0.7, label=f'1/e = {1/np.e:.3f}')
if correlation_length_idx > 0:
    ax2.axvline(correlation_length, color='g', linestyle='--', alpha=0.7, 
               label=f'Correlation length: {correlation_length:.1f} Å')
ax2.set_xlabel('Lag (Å)')
ax2.set_ylabel('Autocorrelation')
ax2.set_title('Water Density Autocorrelation', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Ion-Water Cross-Correlations
ax3 = fig.add_subplot(gs[1, 1])
colors = ['red', 'blue', 'green', 'orange', 'purple']
for idx, (ion_type, cross_corr) in enumerate(cross_correlations.items()):
    color = colors[idx % len(colors)]
    ax3.plot(z_analysis.z_centers, cross_corr, linewidth=2, color=color, label=ion_type)
ax3.set_xlabel('z (Å)')
ax3.set_ylabel('Normalized Cross-Correlation')
ax3.set_title('Ion-Water Cross-Correlations', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.axhline(0, color='k', linestyle='--', alpha=0.5)
ax3.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 4: Entropy bar chart
ax4 = fig.add_subplot(gs[1, 2])
if ion_entropies:
    ion_names = list(ion_entropies.keys())
    entropy_values = list(ion_entropies.values())
    colors_bar = ['red', 'blue', 'green', 'orange', 'purple']
    bars = ax4.bar(range(len(ion_names)), entropy_values, 
                   color=[colors_bar[i % len(colors_bar)] for i in range(len(ion_names))],
                   alpha=0.7, edgecolor='black')
    ax4.set_xticks(range(len(ion_names)))
    ax4.set_xticklabels(ion_names, rotation=45)
    ax4.set_ylabel('Shannon Entropy')
    ax4.set_title('Ion Distribution Entropy', fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, (bar, value) in enumerate(zip(bars, entropy_values)):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.2f}', ha='center', va='bottom', fontweight='bold')

# Plot 5: Ion peak analysis
ax5 = fig.add_subplot(gs[2, :2])
for idx, (ion_type, density) in enumerate(z_analysis.results.ion_densities_by_type.items()):
    color = colors[idx % len(colors)]
    ax5.plot(z_analysis.z_centers, density, linewidth=2, color=color, label=ion_type)
    
    # Mark peaks
    if ion_type in ion_peaks and len(ion_peaks[ion_type]) > 0:
        peak_indices = ion_peaks[ion_type]
        ax5.plot(z_analysis.z_centers[peak_indices], density[peak_indices], 
                'o', color=color, markersize=8, markeredgecolor='black', markeredgewidth=1)

ax5.set_xlabel('z (Å)')
ax5.set_ylabel('Ion Density (ions/Å³)')
ax5.set_title('Ion Density Profiles with Peak Detection', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)
ax5.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 6: Density gradient heatmap
ax6 = fig.add_subplot(gs[2, 2])

# Create gradient matrix for all ion types
gradient_matrix = []
gradient_labels = ['Water']
gradient_matrix.append(np.gradient(z_analysis.results.water_density, z_analysis.z_centers))

for ion_type, density in z_analysis.results.ion_densities_by_type.items():
    gradient_matrix.append(np.gradient(density, z_analysis.z_centers))
    gradient_labels.append(ion_type)

gradient_array = np.array(gradient_matrix)

im = ax6.imshow(gradient_array, aspect='auto', cmap='RdBu_r',
               extent=[z_analysis.z_min, z_analysis.z_max, -0.5, len(gradient_labels)-0.5])
ax6.set_xlabel('z (Å)')
ax6.set_ylabel('Species')
ax6.set_title('Density Gradient Heatmap', fontweight='bold')
ax6.set_yticks(range(len(gradient_labels)))
ax6.set_yticklabels(gradient_labels)
ax6.axvline(0, color='white', linestyle='--', alpha=0.7)

cbar = plt.colorbar(im, ax=ax6)
cbar.set_label('Density Gradient')

# Plot 7: Cumulative distribution functions
ax7 = fig.add_subplot(gs[3, 0])
for idx, (ion_type, density) in enumerate(z_analysis.results.ion_densities_by_type.items()):
    if np.sum(density) > 0:
        cumulative = np.cumsum(density) / np.sum(density)
        color = colors[idx % len(colors)]
        ax7.plot(z_analysis.z_centers, cumulative, linewidth=2, color=color, label=ion_type)

ax7.set_xlabel('z (Å)')
ax7.set_ylabel('Cumulative Fraction')
ax7.set_title('Cumulative Ion Distribution', fontweight='bold')
ax7.legend()
ax7.grid(True, alpha=0.3)
ax7.axvline(0, color='k', linestyle='--', alpha=0.5)
ax7.axhline(0.5, color='gray', linestyle='--', alpha=0.5)

# Plot 8: Local enrichment factors
ax8 = fig.add_subplot(gs[3, 1])

# Calculate enrichment relative to bulk (use outer regions as bulk)
bulk_mask = np.abs(z_analysis.z_centers) > 20.0

for idx, (ion_type, density) in enumerate(z_analysis.results.ion_densities_by_type.items()):
    if np.any(bulk_mask):
        bulk_density = np.mean(density[bulk_mask])
        if bulk_density > 0:
            enrichment = density / bulk_density
            color = colors[idx % len(colors)]
            ax8.plot(z_analysis.z_centers, enrichment, linewidth=2, color=color, label=ion_type)

ax8.set_xlabel('z (Å)')
ax8.set_ylabel('Local Enrichment Factor')
ax8.set_title('Ion Enrichment Relative to Bulk', fontweight='bold')
ax8.axhline(1, color='k', linestyle='--', alpha=0.5, label='Bulk level')
ax8.legend()
ax8.grid(True, alpha=0.3)
ax8.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 9: Spatial heterogeneity index
ax9 = fig.add_subplot(gs[3, 2])

# Calculate local standard deviation (moving window)
window_size = 5
heterogeneity = {}

for ion_type, density in z_analysis.results.ion_densities_by_type.items():
    if np.sum(density) > 0:
        # Moving window standard deviation
        local_std = np.array([
            np.std(density[max(0, i-window_size):min(len(density), i+window_size)])
            for i in range(len(density))
        ])
        heterogeneity[ion_type] = local_std

for idx, (ion_type, het) in enumerate(heterogeneity.items()):
    color = colors[idx % len(colors)]
    ax9.plot(z_analysis.z_centers, het, linewidth=2, color=color, label=ion_type)

ax9.set_xlabel('z (Å)')
ax9.set_ylabel('Local Heterogeneity (std)')
ax9.set_title('Spatial Heterogeneity Index', fontweight='bold')
ax9.legend()
ax9.grid(True, alpha=0.3)
ax9.axvline(0, color='k', linestyle='--', alpha=0.5)

plt.savefig('advanced_structural_analysis.png', dpi=300, bbox_inches='tight')
print("  ✓ Advanced analysis plot saved as: advanced_structural_analysis.png")
plt.show()

# 7. Statistical Summary
print("\n7. Advanced Statistical Summary:")

print(f"\n  Density Fluctuation Metrics:")
print(f"    Mean fluctuation: {np.mean(water_fluctuations):.5f}")
print(f"    Max fluctuation: {np.max(water_fluctuations):.5f}")
print(f"    Coefficient of variation: {np.std(z_analysis.results.water_density)/np.mean(z_analysis.results.water_density)*100:.2f}%")

print(f"\n  Correlation Metrics:")
if correlation_length_idx > 0:
    print(f"    Water density correlation length: {correlation_length:.2f} Å")
else:
    print(f"    Water density correlation length: > system size")

print(f"\n  Peak Analysis:")
print(f"    Water density peaks: {len(peaks)}")
for ion_type, peak_indices in ion_peaks.items():
    print(f"    {ion_type} peaks: {len(peak_indices)}")

print(f"\n  Distribution Entropy:")
for ion_type, entropy in ion_entropies.items():
    # Normalized entropy (0-1 scale)
    max_entropy = np.log(len(z_analysis.z_centers))
    norm_entropy = entropy / max_entropy
    print(f"    {ion_type}: {entropy:.3f} (normalized: {norm_entropy:.3f})")

# 8. Store results
print("\n8. Storing Advanced Analysis Results:")

z_analysis.results.advanced_analysis = {
    'fluctuations': {
        'water_fluctuations': water_fluctuations,
        'water_fluctuations_smooth': water_fluctuations_smooth,
        'high_fluctuation_regions': high_fluctuation_regions
    },
    'entropy': ion_entropies,
    'correlations': {
        'water_autocorrelation': water_autocorr,
        'correlation_length': correlation_length if correlation_length_idx > 0 else None,
        'ion_water_cross_correlations': cross_correlations
    },
    'peaks': {
        'water_peaks': peaks,
        'ion_peaks': ion_peaks
    },
    'heterogeneity': heterogeneity
}

print("  ✓ Advanced analysis data stored in z_analysis.results.advanced_analysis")

print("\n=== Analysis 8 Complete ===")
print("\n🎉 All 8 comprehensive analyses are now complete!")
print("\nAvailable analysis results:")
print("  1. Basic density profiles")
print("  2. Ion coordination and pairing")
print("  3. Water structure and clustering")
print("  4. Symmetrization analysis")
print("  5. Interface detection")
print("  6. Layer-based analysis")
print("  7. Time-resolved analysis")
print("  8. Advanced structural/dynamic analysis")

In [ ]:
print("=== ANALYSIS 9: Electrostatic Potential and Charge Distribution ===")

import numpy as np
import matplotlib.pyplot as plt

# 1. Calculate electrostatic potential and charge density
print("\n1. Calculating Electrostatic Potential Profile:")

# Check if already calculated
if not hasattr(z_analysis.results, 'electrostatic_potential'):
    print("  Calculating electrostatic potential vs z...")
    z_analysis.calculate_electrostatic_potential_vs_z()
    print("  ✓ Calculation complete")
else:
    print("  Electrostatic potential already calculated")

# 2. Plot electrostatic potential analysis
print("\n2. Plotting Electrostatic Potential Analysis:")
z_analysis.plot_electrostatic_potential(save_plots=True)

# 3. Detailed charge distribution analysis
print("\n3. Detailed Charge Distribution Analysis:")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Charge density profile
ax1 = axes[0, 0]
charge_density = z_analysis.results.charge_density
ax1.plot(z_analysis.z_centers, charge_density, 'r-', linewidth=2)
ax1.fill_between(z_analysis.z_centers, 0, charge_density, 
                 where=(charge_density > 0), alpha=0.3, color='red', label='Positive')
ax1.fill_between(z_analysis.z_centers, 0, charge_density, 
                 where=(charge_density < 0), alpha=0.3, color='blue', label='Negative')
ax1.set_xlabel('z (Å)')
ax1.set_ylabel('Charge Density (e/Å³)')
ax1.set_title('Charge Density Distribution', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='k', linestyle='--', alpha=0.5)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 2: Electrostatic potential
ax2 = axes[0, 1]
potential = z_analysis.results.electrostatic_potential
ax2.plot(z_analysis.z_centers, potential, 'g-', linewidth=2)
ax2.set_xlabel('z (Å)')
ax2.set_ylabel('Electrostatic Potential (arb. units)')
ax2.set_title('Electrostatic Potential Profile', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', linestyle='--', alpha=0.5)
ax2.axvline(0, color='k', linestyle='--', alpha=0.5)

# Calculate reference potential (bulk average)
bulk_mask = np.abs(z_analysis.z_centers) > 20.0
if np.any(bulk_mask):
    bulk_potential = np.mean(potential[bulk_mask])
    ax2.axhline(bulk_potential, color='orange', linestyle='--', alpha=0.7,
               label=f'Bulk: {bulk_potential:.3f}')
    ax2.legend()

# Plot 3: Electric field (negative gradient of potential)
ax3 = axes[0, 2]
electric_field = -np.gradient(potential, z_analysis.z_centers)
ax3.plot(z_analysis.z_centers, electric_field, 'b-', linewidth=2)
ax3.fill_between(z_analysis.z_centers, 0, electric_field, alpha=0.3, color='blue')
ax3.set_xlabel('z (Å)')
ax3.set_ylabel('Electric Field (arb. units/Å)')
ax3.set_title('Electric Field Profile', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.axhline(0, color='k', linestyle='--', alpha=0.5)
ax3.axvline(0, color='k', linestyle='--', alpha=0.5)

# Find field maxima
field_peaks_pos = np.where((electric_field[1:-1] > electric_field[:-2]) & 
                           (electric_field[1:-1] > electric_field[2:]))[0] + 1
field_peaks_neg = np.where((electric_field[1:-1] < electric_field[:-2]) & 
                           (electric_field[1:-1] < electric_field[2:]))[0] + 1

if len(field_peaks_pos) > 0:
    ax3.plot(z_analysis.z_centers[field_peaks_pos], electric_field[field_peaks_pos], 
            'ro', markersize=8, label='Max field')
if len(field_peaks_neg) > 0:
    ax3.plot(z_analysis.z_centers[field_peaks_neg], electric_field[field_peaks_neg], 
            'go', markersize=8, label='Min field')
if len(field_peaks_pos) > 0 or len(field_peaks_neg) > 0:
    ax3.legend()

# Plot 4: Ion contributions to charge density
ax4 = axes[1, 0]

# Calculate charge contribution from each ion type
for ion_type, density in z_analysis.results.ion_densities_by_type.items():
    # Assign typical charges (adjust based on your system)
    if ion_type in ['NA', 'Na', 'K', 'Li', 'Rb', 'Cs']:
        charge = +1
    elif ion_type in ['MG', 'Mg', 'Ca', 'Sr', 'Ba']:
        charge = +2
    elif ion_type in ['CL', 'Cl', 'Br', 'I', 'F']:
        charge = -1
    elif ion_type in ['SO4', 'S']:
        charge = -2
    else:
        charge = 0  # Unknown
    
    charge_contribution = density * charge
    ax4.plot(z_analysis.z_centers, charge_contribution, linewidth=2, label=ion_type)

ax4.set_xlabel('z (Å)')
ax4.set_ylabel('Charge Contribution (e/Å³)')
ax4.set_title('Ion-Specific Charge Contributions', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.axhline(0, color='k', linestyle='--', alpha=0.5)
ax4.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 5: Integrated charge profile
ax5 = axes[1, 1]
dz = np.diff(z_analysis.z_centers)[0] if len(z_analysis.z_centers) > 1 else 1.0
integrated_charge = np.cumsum(charge_density) * dz

ax5.plot(z_analysis.z_centers, integrated_charge, 'purple', linewidth=2)
ax5.set_xlabel('z (Å)')
ax5.set_ylabel('Integrated Charge (e)')
ax5.set_title('Cumulative Charge Distribution', fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.axhline(0, color='k', linestyle='--', alpha=0.5)
ax5.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 6: Charge density vs potential scatter
ax6 = axes[1, 2]
ax6.scatter(charge_density, potential, alpha=0.6, s=30, c=z_analysis.z_centers, 
           cmap='coolwarm')
ax6.set_xlabel('Charge Density (e/Å³)')
ax6.set_ylabel('Electrostatic Potential (arb. units)')
ax6.set_title('Charge vs Potential Correlation', fontweight='bold')
ax6.grid(True, alpha=0.3)
ax6.axhline(0, color='k', linestyle='--', alpha=0.5)
ax6.axvline(0, color='k', linestyle='--', alpha=0.5)

# Add colorbar
cbar = plt.colorbar(ax6.collections[0], ax=ax6)
cbar.set_label('z position (Å)')

plt.tight_layout()
plt.savefig('electrostatic_detailed_analysis.png', dpi=300, bbox_inches='tight')
print("  ✓ Detailed electrostatic analysis saved as: electrostatic_detailed_analysis.png")
plt.show()

# 4. Statistical analysis
print("\n4. Electrostatic Statistical Analysis:")

print(f"\n  Charge Density Statistics:")
print(f"    Mean: {np.mean(charge_density):.6f} e/Å³")
print(f"    Std: {np.std(charge_density):.6f} e/Å³")
print(f"    Max positive: {np.max(charge_density):.6f} e/Å³")
print(f"    Max negative: {np.min(charge_density):.6f} e/Å³")

# Check charge neutrality
total_charge = np.sum(charge_density) * dz
print(f"    Total integrated charge: {total_charge:.6f} e")

if abs(total_charge) < 0.01:
    print(f"    ✓ System is approximately charge neutral")
else:
    print(f"    ⚠ System has net charge: {total_charge:.6f} e")

print(f"\n  Electrostatic Potential Statistics:")
print(f"    Mean: {np.mean(potential):.6f}")
print(f"    Std: {np.std(potential):.6f}")
print(f"    Range: {np.min(potential):.6f} to {np.max(potential):.6f}")

# Potential difference across system
potential_diff = potential[-1] - potential[0]
print(f"    Potential difference (left to right): {potential_diff:.6f}")

print(f"\n  Electric Field Statistics:")
print(f"    Mean magnitude: {np.mean(np.abs(electric_field)):.6f}")
print(f"    Max field: {np.max(np.abs(electric_field)):.6f}")
print(f"    Number of field maxima: {len(field_peaks_pos) + len(field_peaks_neg)}")

# 5. Identify charged regions
print("\n5. Identifying Charged Regions:")

# Define threshold as 2 standard deviations from mean
charge_threshold = np.mean(np.abs(charge_density)) + 2 * np.std(charge_density)

positive_regions = z_analysis.z_centers[charge_density > charge_threshold]
negative_regions = z_analysis.z_centers[charge_density < -charge_threshold]

if len(positive_regions) > 0:
    print(f"\n  Positively charged regions ({len(positive_regions)} bins):")
    print(f"    z positions: {positive_regions[0]:.2f} to {positive_regions[-1]:.2f} Å")
    print(f"    Max charge: {np.max(charge_density[charge_density > charge_threshold]):.6f} e/Å³")

if len(negative_regions) > 0:
    print(f"\n  Negatively charged regions ({len(negative_regions)} bins):")
    print(f"    z positions: {negative_regions[0]:.2f} to {negative_regions[-1]:.2f} Å")
    print(f"    Min charge: {np.min(charge_density[charge_density < -charge_threshold]):.6f} e/Å³")

# 6. Potential barrier analysis
print("\n6. Potential Barrier Analysis:")

# Find potential barriers (local maxima/minima)
potential_peaks = np.where((potential[1:-1] > potential[:-2]) & 
                           (potential[1:-1] > potential[2:]))[0] + 1
potential_valleys = np.where((potential[1:-1] < potential[:-2]) & 
                             (potential[1:-1] < potential[2:]))[0] + 1

print(f"  Potential maxima: {len(potential_peaks)}")
for i, peak_idx in enumerate(potential_peaks):
    print(f"    Peak {i+1}: z = {z_analysis.z_centers[peak_idx]:.2f} Å, V = {potential[peak_idx]:.6f}")

print(f"  Potential minima: {len(potential_valleys)}")
for i, valley_idx in enumerate(potential_valleys):
    print(f"    Valley {i+1}: z = {z_analysis.z_centers[valley_idx]:.2f} Å, V = {potential[valley_idx]:.6f}")

# Calculate barrier heights
if len(potential_peaks) > 0 and len(potential_valleys) > 0:
    barrier_heights = []
    for peak_idx in potential_peaks:
        for valley_idx in potential_valleys:
            barrier = abs(potential[peak_idx] - potential[valley_idx])
            barrier_heights.append(barrier)
    
    if barrier_heights:
        print(f"\n  Barrier heights:")
        print(f"    Maximum: {np.max(barrier_heights):.6f}")
        print(f"    Minimum: {np.min(barrier_heights):.6f}")
        print(f"    Average: {np.mean(barrier_heights):.6f}")

# 7. Correlation with interfaces
print("\n7. Correlation with Interfaces:")

if hasattr(z_analysis.results, 'interface_analysis'):
    interfaces = z_analysis.results.interface_analysis['interfaces']['all_interfaces']
    
    print(f"  Analyzing {len(interfaces)} interfaces:")
    
    for i, (start, end) in enumerate(interfaces):
        center = (start + end) / 2
        
        # Find nearest z-bin
        center_idx = np.argmin(np.abs(z_analysis.z_centers - center))
        
        interface_charge = charge_density[center_idx]
        interface_potential = potential[center_idx]
        interface_field = electric_field[center_idx]
        
        print(f"\n  Interface {i+1} (z = {center:.2f} Å):")
        print(f"    Charge density: {interface_charge:.6f} e/Å³")
        print(f"    Potential: {interface_potential:.6f}")
        print(f"    Electric field: {interface_field:.6f}")
else:
    print("  No interface data available for correlation analysis")

# 8. Save results
print("\n8. Storing Electrostatic Analysis Results:")

z_analysis.results.electrostatic_analysis = {
    'charge_density': charge_density,
    'electrostatic_potential': potential,
    'electric_field': electric_field,
    'integrated_charge': integrated_charge,
    'statistics': {
        'mean_charge_density': np.mean(charge_density),
        'std_charge_density': np.std(charge_density),
        'total_charge': total_charge,
        'mean_potential': np.mean(potential),
        'potential_difference': potential_diff,
        'max_field': np.max(np.abs(electric_field))
    },
    'barriers': {
        'potential_peaks': potential_peaks,
        'potential_valleys': potential_valleys,
        'peak_positions': z_analysis.z_centers[potential_peaks].tolist() if len(potential_peaks) > 0 else [],
        'valley_positions': z_analysis.z_centers[potential_valleys].tolist() if len(potential_valleys) > 0 else []
    }
}

print("  ✓ Electrostatic analysis data stored in z_analysis.results.electrostatic_analysis")

print("\n=== Analysis 9 Complete ===")

In [ ]:
## this is not necessary

In [ ]:
# # This analysis is not needed
# print("=== ANALYSIS : Local Temperature Profile ===")

# # Calculate the local temperature profile
# z_analysis.calculate_local_temperature_vs_z()

# # Plot the results
# z_analysis.plotter.plot_local_temperature_vs_z(save_plots=True)

# # Plot the dipole orientation results
# # z_analysis.plot_dipole_orientation_vs_z(save_plots=True, bin_size=2.0)


In [ ]:
## provides water tetrahedral order, hydrogen bonding, and coordination profiles

print("=== ANALYSIS 9: Water Structure Analysis ===")

# Calculate water structure profiles
z_analysis.calculate_water_structure_vs_z_detailed()
# # Force recalculation if needed
# z_analysis.calculate_water_structure_vs_z_detailed(force_rerun=True)
# z_analysis.calculate_water_structure_vs_z_detailed(step=10)  # Analyze every 10th frame
# # Method 4: Force recalculation with different step
# z_analysis.calculate_water_structure_vs_z_detailed(force_rerun=True, step=5)

# Plot the results
z_analysis.plot_water_structure_analysis(save_plots=True)

In [ ]:
print("=== ANALYSIS 10: Ion Pairing and Association Analysis ===")

import numpy as np
import matplotlib.pyplot as plt

# 1. Calculate ion pairing vs z
print("\n1. Calculating Ion Pairing vs Z-Position:")

# Check the actual method signature first
import inspect
sig = inspect.signature(z_analysis.calculate_ion_pairing_vs_z)
print(f"  calculate_ion_pairing_vs_z parameters: {sig}")

# Call with correct parameters (likely no parameters needed)
try:
    z_analysis.calculate_ion_pairing_vs_z()
    print("  ✓ Ion pairing calculation complete")
except Exception as e:
    print(f"  Error during calculation: {e}")

# 2. Plot ion pairing results using the plotter
print("\n2. Plotting Ion Pairing Results:")

# Access the plotter from z_analysis
if hasattr(z_analysis, 'plotter'):
    z_analysis.plotter.plot_ion_pairing_results(save_plots=True)
else:
    # Create a plotter if it doesn't exist
    from ZDirectionalPlotter import ZDirectionalPlotter
    plotter = ZDirectionalPlotter(z_analysis)
    plotter.plot_ion_pairing_results(save_plots=True)

# 3. Detailed ion pairing analysis
print("\n3. Detailed Ion Pairing Analysis:")

if hasattr(z_analysis.results, 'ion_pairing'):
    pairing_data = z_analysis.results.ion_pairing
    z_centers = pairing_data['z_centers']
    
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)
    
    # Get all ion pairs
    ion_pairs = list(pairing_data['contact_pair_fraction'].keys())
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    # Plot 1: Contact pair fraction heatmap
    ax1 = fig.add_subplot(gs[0, :])
    
    if len(ion_pairs) > 0:
        contact_matrix = np.array([pairing_data['contact_pair_fraction'][pair] 
                                  for pair in ion_pairs])
        
        im1 = ax1.imshow(contact_matrix, aspect='auto', cmap='YlOrRd',
                        extent=[z_centers.min(), z_centers.max(), -0.5, len(ion_pairs)-0.5])
        ax1.set_xlabel('z (Å)')
        ax1.set_ylabel('Ion Pairs')
        ax1.set_title('Contact Ion Pair Fraction Heatmap', fontweight='bold', fontsize=14)
        ax1.set_yticks(range(len(ion_pairs)))
        ax1.set_yticklabels(ion_pairs)
        ax1.axvline(0, color='white', linestyle='--', alpha=0.7, linewidth=2)
        
        cbar1 = plt.colorbar(im1, ax=ax1)
        cbar1.set_label('Contact Pair Fraction')
    
    # Plot 2: Contact pair fraction vs z (line plot)
    ax2 = fig.add_subplot(gs[1, 0])
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        fractions = pairing_data['contact_pair_fraction'][pair]
        ax2.plot(z_centers, fractions, color=color, linewidth=2, 
                marker='o', markersize=4, label=pair)
    
    ax2.set_xlabel('z (Å)')
    ax2.set_ylabel('Contact Pair Fraction')
    ax2.set_title('Contact Ion Pairs vs Z', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 3: Average pair distance vs z
    ax3 = fig.add_subplot(gs[1, 1])
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        distances = pairing_data['average_pair_distance'][pair]
        ax3.plot(z_centers, distances, color=color, linewidth=2,
                marker='s', markersize=4, label=pair)
    
    ax3.set_xlabel('z (Å)')
    ax3.set_ylabel('Average Pair Distance (Å)')
    ax3.set_title('Average Ion Pair Distance vs Z', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 4: Coordination numbers vs z
    ax4 = fig.add_subplot(gs[1, 2])
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        coord_nums = pairing_data['coordination_numbers'][pair]
        ax4.plot(z_centers, coord_nums, color=color, linewidth=2,
                marker='^', markersize=4, label=pair)
    
    ax4.set_xlabel('z (Å)')
    ax4.set_ylabel('Coordination Number')
    ax4.set_title('Ion Pair Coordination vs Z', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 5: Pair density (contact fraction × ion density)
    ax5 = fig.add_subplot(gs[2, 0])
    
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        fractions = pairing_data['contact_pair_fraction'][pair]
        
        # Get the ion densities for this pair
        ion1, ion2 = pair.split('-')
        if ion1 in z_analysis.results.ion_densities_by_type:
            density1 = z_analysis.results.ion_densities_by_type[ion1]
            
            # Interpolate if lengths don't match
            if len(density1) != len(fractions):
                from scipy.interpolate import interp1d
                f = interp1d(z_analysis.z_centers, density1, kind='linear', fill_value='extrapolate')
                density1_interp = f(z_centers)
                pair_density = fractions * density1_interp
            else:
                pair_density = fractions * density1
            
            ax5.plot(z_centers, pair_density, color=color, linewidth=2, label=pair)
    
    ax5.set_xlabel('z (Å)')
    ax5.set_ylabel('Pair Density (pairs/Å³)')
    ax5.set_title('Ion Pair Density vs Z', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    ax5.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 6: Pairing efficiency
    ax6 = fig.add_subplot(gs[2, 1])
    
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        fractions = pairing_data['contact_pair_fraction'][pair]
        
        # Normalize by total ion density
        ion1, ion2 = pair.split('-')
        if ion1 in z_analysis.results.ion_densities_by_type and \
           ion2 in z_analysis.results.ion_densities_by_type:
            density1 = z_analysis.results.ion_densities_by_type[ion1]
            density2 = z_analysis.results.ion_densities_by_type[ion2]
            
            # Handle length mismatch
            if len(density1) != len(fractions):
                from scipy.interpolate import interp1d
                f1 = interp1d(z_analysis.z_centers, density1, kind='linear', fill_value='extrapolate')
                f2 = interp1d(z_analysis.z_centers, density2, kind='linear', fill_value='extrapolate')
                density1 = f1(z_centers)
                density2 = f2(z_centers)
            
            total_density = density1 + density2
            
            efficiency = np.zeros_like(fractions)
            mask = total_density > 0
            efficiency[mask] = fractions[mask] / total_density[mask]
            
            ax6.plot(z_centers, efficiency, color=color, linewidth=2, label=pair)
    
    ax6.set_xlabel('z (Å)')
    ax6.set_ylabel('Pairing Efficiency (normalized)')
    ax6.set_title('Ion Pairing Efficiency vs Z', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    ax6.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 7: Distance distribution for specific z-regions
    ax7 = fig.add_subplot(gs[2, 2])
    
    # Define regions: interface, bulk
    interface_mask = np.abs(z_centers) < 10.0
    bulk_mask = np.abs(z_centers) > 20.0
    
    for idx, pair in enumerate(ion_pairs):
        distances = pairing_data['average_pair_distance'][pair]
        
        if np.any(interface_mask):
            interface_dist = np.mean(distances[interface_mask])
            ax7.bar(idx - 0.2, interface_dist, width=0.4, 
                   color=colors[idx % len(colors)], alpha=0.7, label=pair if idx == 0 else '')
        
        if np.any(bulk_mask):
            bulk_dist = np.mean(distances[bulk_mask])
            ax7.bar(idx + 0.2, bulk_dist, width=0.4,
                   color=colors[idx % len(colors)], alpha=0.4)
    
    ax7.set_ylabel('Average Distance (Å)')
    ax7.set_title('Pair Distance: Interface vs Bulk', fontweight='bold')
    ax7.set_xticks(range(len(ion_pairs)))
    ax7.set_xticklabels(ion_pairs, rotation=45, ha='right')
    ax7.grid(True, alpha=0.3, axis='y')
    ax7.legend(['Interface', 'Bulk'])
    
    # Plot 8: Cumulative pairing statistics
    ax8 = fig.add_subplot(gs[3, 0])
    
    pair_stats = {}
    for pair in ion_pairs:
        fractions = pairing_data['contact_pair_fraction'][pair]
        pair_stats[pair] = {
            'mean': np.mean(fractions),
            'max': np.max(fractions),
            'std': np.std(fractions)
        }
    
    x_pos = np.arange(len(ion_pairs))
    means = [pair_stats[p]['mean'] for p in ion_pairs]
    stds = [pair_stats[p]['std'] for p in ion_pairs]
    
    bars = ax8.bar(x_pos, means, yerr=stds, capsize=5,
                   color=[colors[i % len(colors)] for i in range(len(ion_pairs))],
                   alpha=0.7, edgecolor='black')
    
    ax8.set_ylabel('Mean Contact Fraction')
    ax8.set_title('Overall Pairing Statistics', fontweight='bold')
    ax8.set_xticks(x_pos)
    ax8.set_xticklabels(ion_pairs, rotation=45, ha='right')
    ax8.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, (bar, mean) in enumerate(zip(bars, means)):
        height = bar.get_height()
        ax8.text(bar.get_x() + bar.get_width()/2., height,
                f'{mean:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # Plot 9: Pairing correlation with density
    ax9 = fig.add_subplot(gs[3, 1])
    
    for idx, pair in enumerate(ion_pairs):
        color = colors[idx % len(colors)]
        fractions = pairing_data['contact_pair_fraction'][pair]
        
        # Get combined ion density
        ion1, ion2 = pair.split('-')
        if ion1 in z_analysis.results.ion_densities_by_type and \
           ion2 in z_analysis.results.ion_densities_by_type:
            density1 = z_analysis.results.ion_densities_by_type[ion1]
            density2 = z_analysis.results.ion_densities_by_type[ion2]
            
            # Handle length mismatch
            if len(density1) != len(fractions):
                from scipy.interpolate import interp1d
                f1 = interp1d(z_analysis.z_centers, density1, kind='linear', fill_value='extrapolate')
                f2 = interp1d(z_analysis.z_centers, density2, kind='linear', fill_value='extrapolate')
                density1 = f1(z_centers)
                density2 = f2(z_centers)
            
            combined_density = density1 + density2
            
            ax9.scatter(combined_density, fractions, color=color, 
                       alpha=0.6, s=30, label=pair)
    
    ax9.set_xlabel('Combined Ion Density (ions/Å³)')
    ax9.set_ylabel('Contact Pair Fraction')
    ax9.set_title('Pairing vs Density Correlation', fontweight='bold')
    ax9.legend()
    ax9.grid(True, alpha=0.3)
    
    # Plot 10: Overall pairing strength
    ax10 = fig.add_subplot(gs[3, 2])
    
    pair_matrix_data = []
    for pair in ion_pairs:
        fractions = pairing_data['contact_pair_fraction'][pair]
        pair_matrix_data.append(np.mean(fractions))
    
    bars = ax10.barh(range(len(ion_pairs)), pair_matrix_data,
                     color=[colors[i % len(colors)] for i in range(len(ion_pairs))],
                     alpha=0.7, edgecolor='black')
    
    ax10.set_xlabel('Mean Contact Fraction')
    ax10.set_title('Overall Pairing Strength', fontweight='bold')
    ax10.set_yticks(range(len(ion_pairs)))
    ax10.set_yticklabels(ion_pairs)
    ax10.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, value) in enumerate(zip(bars, pair_matrix_data)):
        width = bar.get_width()
        ax10.text(width, bar.get_y() + bar.get_height()/2.,
                 f'{value:.3f}', ha='left', va='center', fontweight='bold', fontsize=9)
    
    plt.savefig('ion_pairing_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    print("  ✓ Comprehensive ion pairing plot saved as: ion_pairing_comprehensive_analysis.png")
    plt.show()

else:
    print("  No ion pairing data available.")

# 4. Statistical summary
print("\n4. Ion Pairing Statistical Summary:")

if hasattr(z_analysis.results, 'ion_pairing'):
    pairing_data = z_analysis.results.ion_pairing
    
    print(f"\n  Overall Statistics:")
    for pair in pairing_data['contact_pair_fraction'].keys():
        fractions = pairing_data['contact_pair_fraction'][pair]
        distances = pairing_data['average_pair_distance'][pair]
        coord_nums = pairing_data['coordination_numbers'][pair]
        
        print(f"\n  {pair}:")
        print(f"    Contact pair fraction:")
        print(f"      Mean: {np.mean(fractions):.4f}")
        print(f"      Max: {np.max(fractions):.4f}")
        print(f"      Std: {np.std(fractions):.4f}")
        
        print(f"    Average pair distance:")
        print(f"      Mean: {np.mean(distances):.3f} Å")
        print(f"      Min: {np.min(distances):.3f} Å")
        print(f"      Max: {np.max(distances):.3f} Å")
        
        print(f"    Coordination number:")
        print(f"      Mean: {np.mean(coord_nums):.2f}")
        print(f"      Max: {np.max(coord_nums):.2f}")
        
        # Identify regions of high pairing
        high_pairing_mask = fractions > (np.mean(fractions) + np.std(fractions))
        if np.any(high_pairing_mask):
            high_pairing_z = z_centers[high_pairing_mask]
            print(f"    High pairing regions: z = {high_pairing_z[0]:.2f} to {high_pairing_z[-1]:.2f} Å")

# 5. Compare pairing at different regions
print("\n5. Regional Pairing Analysis:")

if hasattr(z_analysis.results, 'ion_pairing'):
    pairing_data = z_analysis.results.ion_pairing
    z_centers = pairing_data['z_centers']
    
    # Define regions
    regions = {
        'Center': (np.abs(z_centers) < 5.0),
        'Interface': ((np.abs(z_centers) >= 5.0) & (np.abs(z_centers) < 15.0)),
        'Bulk': (np.abs(z_centers) >= 15.0)
    }
    
    for region_name, mask in regions.items():
        if np.any(mask):
            print(f"\n  {region_name} Region:")
            
            for pair in pairing_data['contact_pair_fraction'].keys():
                fractions = pairing_data['contact_pair_fraction'][pair]
                region_fraction = np.mean(fractions[mask])
                
                print(f"    {pair}: {region_fraction:.4f}")

# 6. Ion association strength classification
print("\n6. Ion Association Strength Classification:")

if hasattr(z_analysis.results, 'ion_pairing'):
    for pair in pairing_data['contact_pair_fraction'].keys():
        fractions = pairing_data['contact_pair_fraction'][pair]
        mean_fraction = np.mean(fractions)
        max_fraction = np.max(fractions)
        
        print(f"\n  {pair}:")
        print(f"    Mean fraction: {mean_fraction:.4f}")
        print(f"    Max fraction: {max_fraction:.4f}")
        
        # Classify association strength
        if mean_fraction > 0.3:
            strength = "Strong association"
        elif mean_fraction > 0.1:
            strength = "Moderate association"
        elif mean_fraction > 0.05:
            strength = "Weak association"
        else:
            strength = "Minimal association"
        
        print(f"    Classification: {strength}")

print("\n=== Analysis 10 Complete ===")
print("\n🎉 ALL 10 comprehensive analyses are now complete!")

In [ ]:
print("=== ANALYSIS 12: Radial Distribution Function (RDF) Analysis ===")

import numpy as np
import matplotlib.pyplot as plt

# 1. Calculate RDF vs z
print("\n1. Calculating RDF vs Z-Position:")

# Check the actual method signature
import inspect
sig = inspect.signature(z_analysis.calculate_rdf_vs_z_detailed)
print(f"  calculate_rdf_vs_z_detailed parameters: {sig}")

# Call the RDF calculation
try:
    z_analysis.calculate_rdf_vs_z_detailed()
    print("  ✓ RDF vs z calculation complete")
except Exception as e:
    print(f"  Error during calculation: {e}")

# 2. Plot RDF results using the plotter
print("\n2. Plotting RDF Results:")

# Access the plotter
if hasattr(z_analysis, 'plotter'):
    z_analysis.plotter.plot_rdf_vs_z(save_plots=True)
else:
    # Create a plotter if it doesn't exist
    from ZDirectionalPlotter import ZDirectionalPlotter
    plotter = ZDirectionalPlotter(z_analysis)
    plotter.plot_rdf_vs_z(save_plots=True)

# 3. Detailed RDF analysis
print("\n3. Detailed RDF Analysis:")

if hasattr(z_analysis.results, 'rdf_vs_z'):
    rdf_data = z_analysis.results.rdf_vs_z
    
    # Get ion pairs
    ion_pairs = list(rdf_data.keys())
    
    if len(ion_pairs) == 0:
        print("  No RDF data available.")
    else:
        fig = plt.figure(figsize=(20, 16))
        gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)
        
        colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
        
        # Plot 1: 2D heatmap of first ion pair
        if len(ion_pairs) > 0:
            pair = ion_pairs[0]
            pair_data = rdf_data[pair]
            
            ax1 = fig.add_subplot(gs[0, :])
            
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            im1 = ax1.contourf(r_centers, z_centers, rdf_matrix, 
                              levels=20, cmap='hot')
            ax1.set_xlabel('r (Å)')
            ax1.set_ylabel('z (Å)')
            ax1.set_title(f'RDF Heatmap: {pair}', fontweight='bold', fontsize=14)
            ax1.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=2)
            
            cbar1 = plt.colorbar(im1, ax=ax1)
            cbar1.set_label('g(r)')
        
        # Plot 2: RDF at specific z-positions for first pair
        if len(ion_pairs) > 0:
            ax2 = fig.add_subplot(gs[1, 0])
            
            pair = ion_pairs[0]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Select specific z-positions to show
            z_positions_to_show = [0, z_centers.min()/2, z_centers.max()/2]
            
            for z_pos in z_positions_to_show:
                z_idx = np.argmin(np.abs(z_centers - z_pos))
                ax2.plot(r_centers, rdf_matrix[z_idx, :], 
                        linewidth=2, label=f'z = {z_centers[z_idx]:.1f} Å')
            
            ax2.set_xlabel('r (Å)')
            ax2.set_ylabel('g(r)')
            ax2.set_title(f'RDF at Different Z-Positions: {pair}', fontweight='bold')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            ax2.axhline(1, color='k', linestyle='--', alpha=0.5, label='Bulk')
        
        # Plot 3: First peak position vs z
        ax3 = fig.add_subplot(gs[1, 1])
        
        for idx, pair in enumerate(ion_pairs):
            color = colors[idx % len(colors)]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Find first peak position for each z
            first_peak_positions = []
            for z_idx in range(len(z_centers)):
                rdf_slice = rdf_matrix[z_idx, :]
                # Find peaks
                from scipy.signal import find_peaks
                peaks, _ = find_peaks(rdf_slice, height=1.0)
                if len(peaks) > 0:
                    first_peak_positions.append(r_centers[peaks[0]])
                else:
                    first_peak_positions.append(np.nan)
            
            # Plot, filtering out NaNs
            valid_mask = ~np.isnan(first_peak_positions)
            if np.any(valid_mask):
                ax3.plot(z_centers[valid_mask], np.array(first_peak_positions)[valid_mask], 
                        color=color, linewidth=2, marker='o', markersize=4, label=pair)
        
        ax3.set_xlabel('z (Å)')
        ax3.set_ylabel('First Peak Position (Å)')
        ax3.set_title('First Shell Radius vs Z', fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
        
        # Plot 4: First peak height vs z
        ax4 = fig.add_subplot(gs[1, 2])
        
        for idx, pair in enumerate(ion_pairs):
            color = colors[idx % len(colors)]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Find first peak height for each z
            first_peak_heights = []
            for z_idx in range(len(z_centers)):
                rdf_slice = rdf_matrix[z_idx, :]
                from scipy.signal import find_peaks
                peaks, properties = find_peaks(rdf_slice, height=1.0)
                if len(peaks) > 0:
                    first_peak_heights.append(properties['peak_heights'][0])
                else:
                    first_peak_heights.append(np.nan)
            
            valid_mask = ~np.isnan(first_peak_heights)
            if np.any(valid_mask):
                ax4.plot(z_centers[valid_mask], np.array(first_peak_heights)[valid_mask], 
                        color=color, linewidth=2, marker='s', markersize=4, label=pair)
        
        ax4.set_xlabel('z (Å)')
        ax4.set_ylabel('First Peak Height')
        ax4.set_title('Structure Strength vs Z', fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
        ax4.axhline(1, color='k', linestyle='--', alpha=0.5)
        
        # Plot 5: Coordination number vs z (from RDF integration)
        ax5 = fig.add_subplot(gs[2, 0])
        
        for idx, pair in enumerate(ion_pairs):
            color = colors[idx % len(colors)]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Calculate coordination number up to first minimum
            coord_numbers = []
            for z_idx in range(len(z_centers)):
                rdf_slice = rdf_matrix[z_idx, :]
                
                # Find first minimum after first peak
                from scipy.signal import find_peaks
                peaks, _ = find_peaks(rdf_slice, height=1.0)
                valleys, _ = find_peaks(-rdf_slice)
                
                if len(peaks) > 0 and len(valleys) > 0:
                    # Find first valley after first peak
                    first_peak_idx = peaks[0]
                    valleys_after_peak = valleys[valleys > first_peak_idx]
                    
                    if len(valleys_after_peak) > 0:
                        first_min_idx = valleys_after_peak[0]
                        
                        # Integrate g(r) up to first minimum
                        # CN = 4π ρ ∫ r² g(r) dr
                        # Simplified: just integrate rdf
                        dr = r_centers[1] - r_centers[0] if len(r_centers) > 1 else 1.0
                        coord_num = np.trapz(rdf_slice[:first_min_idx] * r_centers[:first_min_idx]**2,
                                            dx=dr)
                        coord_numbers.append(coord_num)
                    else:
                        coord_numbers.append(np.nan)
                else:
                    coord_numbers.append(np.nan)
            
            valid_mask = ~np.isnan(coord_numbers)
            if np.any(valid_mask):
                ax5.plot(z_centers[valid_mask], np.array(coord_numbers)[valid_mask], 
                        color=color, linewidth=2, marker='^', markersize=4, label=pair)
        
        ax5.set_xlabel('z (Å)')
        ax5.set_ylabel('Coordination Number (from RDF)')
        ax5.set_title('Coordination from RDF Integration', fontweight='bold')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
        ax5.axvline(0, color='k', linestyle='--', alpha=0.5)
        
        # Plot 6: RDF comparison at center vs edges
        ax6 = fig.add_subplot(gs[2, 1])
        
        for idx, pair in enumerate(ion_pairs):
            color = colors[idx % len(colors)]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            z_centers = pair_data['z_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Get center and edge RDFs
            center_idx = np.argmin(np.abs(z_centers))
            edge_idx = np.argmax(np.abs(z_centers))
            
            ax6.plot(r_centers, rdf_matrix[center_idx, :], 
                    color=color, linewidth=2, linestyle='-', 
                    label=f'{pair} (center)')
            ax6.plot(r_centers, rdf_matrix[edge_idx, :], 
                    color=color, linewidth=2, linestyle='--', alpha=0.5,
                    label=f'{pair} (edge)')
        
        ax6.set_xlabel('r (Å)')
        ax6.set_ylabel('g(r)')
        ax6.set_title('RDF: Center vs Edge', fontweight='bold')
        ax6.legend()
        ax6.grid(True, alpha=0.3)
        ax6.axhline(1, color='k', linestyle='--', alpha=0.5)
        
        # Plot 7: Average RDF across all z
        ax7 = fig.add_subplot(gs[2, 2])
        
        for idx, pair in enumerate(ion_pairs):
            color = colors[idx % len(colors)]
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Average over all z
            avg_rdf = np.mean(rdf_matrix, axis=0)
            std_rdf = np.std(rdf_matrix, axis=0)
            
            ax7.plot(r_centers, avg_rdf, color=color, linewidth=2, label=pair)
            ax7.fill_between(r_centers, avg_rdf - std_rdf, avg_rdf + std_rdf,
                            alpha=0.3, color=color)
        
        ax7.set_xlabel('r (Å)')
        ax7.set_ylabel('g(r)')
        ax7.set_title('Average RDF (All Z-Positions)', fontweight='bold')
        ax7.legend()
        ax7.grid(True, alpha=0.3)
        ax7.axhline(1, color='k', linestyle='--', alpha=0.5)
        
        # Plot 8: RDF structural metrics summary
        ax8 = fig.add_subplot(gs[3, 0])
        
        metrics = {}
        for pair in ion_pairs:
            pair_data = rdf_data[pair]
            rdf_matrix = pair_data['rdf_matrix']
            
            # Calculate average first peak height
            first_peak_heights = []
            for z_idx in range(len(z_centers)):
                rdf_slice = rdf_matrix[z_idx, :]
                from scipy.signal import find_peaks
                peaks, properties = find_peaks(rdf_slice, height=1.0)
                if len(peaks) > 0:
                    first_peak_heights.append(properties['peak_heights'][0])
            
            if first_peak_heights:
                metrics[pair] = np.mean(first_peak_heights)
        
        if metrics:
            pairs_list = list(metrics.keys())
            values_list = list(metrics.values())
            
            bars = ax8.bar(range(len(pairs_list)), values_list,
                          color=[colors[i % len(colors)] for i in range(len(pairs_list))],
                          alpha=0.7, edgecolor='black')
            
            ax8.set_ylabel('Average First Peak Height')
            ax8.set_title('Overall Structure Strength', fontweight='bold')
            ax8.set_xticks(range(len(pairs_list)))
            ax8.set_xticklabels(pairs_list, rotation=45, ha='right')
            ax8.grid(True, alpha=0.3, axis='y')
            ax8.axhline(1, color='k', linestyle='--', alpha=0.5)
            
            # Add value labels
            for i, (bar, value) in enumerate(zip(bars, values_list)):
                height = bar.get_height()
                ax8.text(bar.get_x() + bar.get_width()/2., height,
                        f'{value:.2f}', ha='center', va='bottom', 
                        fontweight='bold', fontsize=9)
        
        # Plot 9: First shell radius summary
        ax9 = fig.add_subplot(gs[3, 1])
        
        shell_radii = {}
        for pair in ion_pairs:
            pair_data = rdf_data[pair]
            r_centers = pair_data['r_centers']
            rdf_matrix = pair_data['rdf_matrix']
            
            # Calculate average first peak position
            first_peak_positions = []
            for z_idx in range(len(z_centers)):
                rdf_slice = rdf_matrix[z_idx, :]
                from scipy.signal import find_peaks
                peaks, _ = find_peaks(rdf_slice, height=1.0)
                if len(peaks) > 0:
                    first_peak_positions.append(r_centers[peaks[0]])
            
            if first_peak_positions:
                shell_radii[pair] = np.mean(first_peak_positions)
        
        if shell_radii:
            pairs_list = list(shell_radii.keys())
            radii_list = list(shell_radii.values())
            
            bars = ax9.barh(range(len(pairs_list)), radii_list,
                           color=[colors[i % len(colors)] for i in range(len(pairs_list))],
                           alpha=0.7, edgecolor='black')
            
            ax9.set_xlabel('First Shell Radius (Å)')
            ax9.set_title('Average First Shell Radii', fontweight='bold')
            ax9.set_yticks(range(len(pairs_list)))
            ax9.set_yticklabels(pairs_list)
            ax9.grid(True, alpha=0.3, axis='x')
            
            # Add value labels
            for i, (bar, value) in enumerate(zip(bars, radii_list)):
                width = bar.get_width()
                ax9.text(width, bar.get_y() + bar.get_height()/2.,
                        f'{value:.2f} Å', ha='left', va='center', 
                        fontweight='bold', fontsize=9)
        
        # Plot 10: Structure evolution heatmap
        ax10 = fig.add_subplot(gs[3, 2])
        
        if len(ion_pairs) > 0:
            # Create matrix of first peak heights for all pairs
            peak_height_matrix = []
            
            for pair in ion_pairs:
                pair_data = rdf_data[pair]
                rdf_matrix = pair_data['rdf_matrix']
                
                first_peak_heights = []
                for z_idx in range(len(z_centers)):
                    rdf_slice = rdf_matrix[z_idx, :]
                    from scipy.signal import find_peaks
                    peaks, properties = find_peaks(rdf_slice, height=1.0)
                    if len(peaks) > 0:
                        first_peak_heights.append(properties['peak_heights'][0])
                    else:
                        first_peak_heights.append(0)
                
                peak_height_matrix.append(first_peak_heights)
            
            peak_height_array = np.array(peak_height_matrix)
            
            im = ax10.imshow(peak_height_array, aspect='auto', cmap='viridis',
                            extent=[z_centers.min(), z_centers.max(), -0.5, len(ion_pairs)-0.5])
            ax10.set_xlabel('z (Å)')
            ax10.set_ylabel('Ion Pairs')
            ax10.set_title('Structure Strength Evolution', fontweight='bold')
            ax10.set_yticks(range(len(ion_pairs)))
            ax10.set_yticklabels(ion_pairs)
            ax10.axvline(0, color='white', linestyle='--', alpha=0.7)
            
            cbar = plt.colorbar(im, ax=ax10)
            cbar.set_label('First Peak Height')
        
        plt.savefig('rdf_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
        print("  ✓ Comprehensive RDF plot saved as: rdf_comprehensive_analysis.png")
        plt.show()

else:
    print("  No RDF data available.")

# 4. Statistical summary
print("\n4. RDF Statistical Summary:")

if hasattr(z_analysis.results, 'rdf_vs_z'):
    rdf_data = z_analysis.results.rdf_vs_z
    
    print(f"\n  Overall Statistics:")
    for pair in rdf_data.keys():
        pair_data = rdf_data[pair]
        r_centers = pair_data['r_centers']
        z_centers = pair_data['z_centers']
        rdf_matrix = pair_data['rdf_matrix']
        
        print(f"\n  {pair}:")
        print(f"    RDF matrix shape: {rdf_matrix.shape}")
        print(f"    r range: {r_centers.min():.2f} - {r_centers.max():.2f} Å")
        print(f"    z range: {z_centers.min():.2f} - {z_centers.max():.2f} Å")
        print(f"    g(r) range: {rdf_matrix.min():.3f} - {rdf_matrix.max():.3f}")
        
        # Calculate first peak statistics
        from scipy.signal import find_peaks
        
        first_peak_positions = []
        first_peak_heights = []
        
        for z_idx in range(len(z_centers)):
            rdf_slice = rdf_matrix[z_idx, :]
            peaks, properties = find_peaks(rdf_slice, height=1.0)
            if len(peaks) > 0:
                first_peak_positions.append(r_centers[peaks[0]])
                first_peak_heights.append(properties['peak_heights'][0])
        
        if first_peak_positions:
            print(f"    First peak position: {np.mean(first_peak_positions):.2f} ± {np.std(first_peak_positions):.2f} Å")
            print(f"    First peak height: {np.mean(first_peak_heights):.2f} ± {np.std(first_peak_heights):.2f}")

# 5. Structural classification
print("\n5. Structural Classification from RDF:")

if hasattr(z_analysis.results, 'rdf_vs_z'):
    for pair in rdf_data.keys():
        pair_data = rdf_data[pair]
        rdf_matrix = pair_data['rdf_matrix']
        
        # Average RDF
        avg_rdf = np.mean(rdf_matrix, axis=0)
        
        # Find first peak
        from scipy.signal import find_peaks
        peaks, properties = find_peaks(avg_rdf, height=1.0)
        
        if len(peaks) > 0:
            first_peak_height = properties['peak_heights'][0]
            
            print(f"\n  {pair}:")
            print(f"    First peak height: {first_peak_height:.2f}")
            
            # Classify structure
            if first_peak_height > 3.0:
                structure = "Highly structured (strong ordering)"
            elif first_peak_height > 2.0:
                structure = "Well structured (moderate ordering)"
            elif first_peak_height > 1.5:
                structure = "Weakly structured (weak ordering)"
            else:
                structure = "Poorly structured (near random)"
            
            print(f"    Classification: {structure}")

print("\n=== Analysis 12 Complete ===")
print("\n🎉 ALL 12 comprehensive analyses are now complete!")
print("\nComplete Analysis Suite:")
print("  1. Basic density profiles")
print("  2. Ion coordination and pairing")
print("  3. Water structure and clustering")
print("  4. Symmetrization analysis")
print("  5. Interface detection")
print("  6. Layer-based analysis")
print("  7. Time-resolved analysis")
print("  8. Advanced structural/dynamic analysis")
print("  9. Electrostatic potential analysis")
print(" 10. Ion pairing analysis")
print(" 11. Ion clustering analysis")
print(" 12. Radial distribution function (RDF) analysis")

In [ ]:
print("=== ANALYSIS 14: Mobility and Diffusion Analysis ===")

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# 1. Check for trajectory/universe object
print("\n1. Checking for trajectory object:")

if hasattr(z_analysis, 'u'):
    universe = z_analysis.u
    print("  Found trajectory as 'u'")
elif hasattr(z_analysis, 'universe'):
    universe = z_analysis.universe
    print("  Found trajectory as 'universe'")
elif hasattr(z_analysis, 'trajectory'):
    universe = z_analysis.trajectory
    print("  Found trajectory as 'trajectory'")
else:
    print("  ERROR: Cannot find trajectory/universe object")
    universe = None

# 2. Calculate MSD vs z
print("\n2. Calculating Mean Squared Displacement (MSD):")

def calculate_msd_vs_z(trajectory, z_bins, species_selection, max_lag_frames=50):
    """Calculate MSD for particles in different z-regions"""
    z_bin_centers = (z_bins[:-1] + z_bins[1:]) / 2
    msd_data = {}
    
    print(f"  Analyzing {len(species_selection)} particles...")
    
    for z_idx, z_center in enumerate(z_bin_centers):
        z_min, z_max = z_bins[z_idx], z_bins[z_idx + 1]
        
        particle_trajectories = []
        
        for ts in trajectory.trajectory[:max_lag_frames * 2]:
            z_positions = species_selection.positions[:, 2]
            in_bin = (z_positions >= z_min) & (z_positions < z_max)
            
            if np.any(in_bin):
                particle_trajectories.append(species_selection.positions[in_bin].copy())
        
        if len(particle_trajectories) < 2:
            continue
        
        msd_values = []
        lag_times = []
        
        for lag in range(1, min(max_lag_frames, len(particle_trajectories) - 1)):
            squared_displacements = []
            
            for i in range(len(particle_trajectories) - lag):
                if len(particle_trajectories[i]) > 0 and len(particle_trajectories[i + lag]) > 0:
                    n_particles = min(len(particle_trajectories[i]), 
                                     len(particle_trajectories[i + lag]))
                    
                    if n_particles > 0:
                        r0 = particle_trajectories[i][:n_particles]
                        rt = particle_trajectories[i + lag][:n_particles]
                        
                        displacements = rt - r0
                        sq_disp = np.sum(displacements**2, axis=1)
                        squared_displacements.extend(sq_disp)
            
            if squared_displacements:
                msd_values.append(np.mean(squared_displacements))
                lag_times.append(lag)
        
        if lag_times:
            msd_data[z_center] = {
                'lag_times': np.array(lag_times),
                'msd': np.array(msd_values)
            }
    
    return msd_data

if universe is not None:
    # Calculate MSD for different species
    print("  Calculating MSD for water molecules...")
    msd_water = calculate_msd_vs_z(universe, 
                                    np.linspace(z_analysis.z_min, z_analysis.z_max, 20),
                                    z_analysis.waters,
                                    max_lag_frames=30)
    
    print("  Calculating MSD for cations...")
    msd_cations = calculate_msd_vs_z(universe,
                                      np.linspace(z_analysis.z_min, z_analysis.z_max, 20),
                                      z_analysis.cations,
                                      max_lag_frames=30)
    
    print("  Calculating MSD for anions...")
    msd_anions = calculate_msd_vs_z(universe,
                                     np.linspace(z_analysis.z_min, z_analysis.z_max, 20),
                                     z_analysis.anions,
                                     max_lag_frames=30)
    
    print("  ✓ MSD calculations complete")
    
    # 3. Calculate diffusion coefficients from MSD
    print("\n3. Calculating Diffusion Coefficients:")
    
    def calculate_diffusion_coefficient(msd_data, dt):
        """Calculate diffusion coefficient from MSD using Einstein relation"""
        diffusion_coeffs = {}
        
        for z_center, data in msd_data.items():
            lag_times = data['lag_times'] * dt  # Convert to ps
            msd = data['msd']  # in Å²
            
            # Fit linear region (usually first 1/4 to 1/2 of data)
            fit_end = len(lag_times) // 2
            if fit_end > 3:
                slope, intercept, r_value, p_value, std_err = linregress(
                    lag_times[:fit_end], msd[:fit_end]
                )
                
                # D = slope / 6 (for 3D), convert to cm²/s
                # 1 Å²/ps = 10⁻³ cm²/s
                D = (slope / 6.0) * 1e-3  # cm²/s
                
                diffusion_coeffs[z_center] = {
                    'D': D,
                    'slope': slope,
                    'r_squared': r_value**2,
                    'std_err': std_err
                }
        
        return diffusion_coeffs
    
    # Get timestep from trajectory
    dt = universe.trajectory.dt  # in ps
    
    D_water = calculate_diffusion_coefficient(msd_water, dt)
    D_cations = calculate_diffusion_coefficient(msd_cations, dt)
    D_anions = calculate_diffusion_coefficient(msd_anions, dt)
    
    print(f"  ✓ Diffusion coefficients calculated for {len(D_water)} z-bins (water)")
    print(f"  ✓ Diffusion coefficients calculated for {len(D_cations)} z-bins (cations)")
    print(f"  ✓ Diffusion coefficients calculated for {len(D_anions)} z-bins (anions)")
    
    # 4. Comprehensive visualization
    print("\n4. Creating Comprehensive Mobility and Diffusion Plots:")
    
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)
    
    colors = ['blue', 'red', 'green']
    
    # Plot 1: Diffusion coefficient vs z
    ax1 = fig.add_subplot(gs[0, :])
    
    if D_water:
        z_centers_water = sorted(D_water.keys())
        D_values_water = [D_water[z]['D'] * 1e5 for z in z_centers_water]  # Convert to 10^-5 cm²/s
        ax1.plot(z_centers_water, D_values_water, color='blue', linewidth=2, 
                marker='o', markersize=6, label='Water')
    
    if D_cations:
        z_centers_cations = sorted(D_cations.keys())
        D_values_cations = [D_cations[z]['D'] * 1e5 for z in z_centers_cations]
        ax1.plot(z_centers_cations, D_values_cations, color='red', linewidth=2,
                marker='s', markersize=6, label='Cations')
    
    if D_anions:
        z_centers_anions = sorted(D_anions.keys())
        D_values_anions = [D_anions[z]['D'] * 1e5 for z in z_centers_anions]
        ax1.plot(z_centers_anions, D_values_anions, color='green', linewidth=2,
                marker='^', markersize=6, label='Anions')
    
    ax1.set_xlabel('z (Å)', fontsize=12)
    ax1.set_ylabel('Diffusion Coefficient (×10⁻⁵ cm²/s)', fontsize=12)
    ax1.set_title('Diffusion Coefficient vs Z-Position', fontweight='bold', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 2: Example MSD curves for different z-positions (water)
    ax2 = fig.add_subplot(gs[1, 0])
    
    if msd_water:
        z_positions = sorted(msd_water.keys())
        selected_z = [z_positions[0], z_positions[len(z_positions)//2], z_positions[-1]]
        
        for z_pos in selected_z:
            if z_pos in msd_water:
                data = msd_water[z_pos]
                time = data['lag_times'] * dt
                ax2.plot(time, data['msd'], linewidth=2, marker='o', 
                        markersize=4, label=f'z = {z_pos:.1f} Å')
        
        ax2.set_xlabel('Time (ps)', fontsize=11)
        ax2.set_ylabel('MSD (Ų)', fontsize=11)
        ax2.set_title('Water MSD at Different Z-Positions', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # Plot 3: Relative diffusion (normalized to bulk)
    ax3 = fig.add_subplot(gs[1, 1])
    
    if D_water:
        bulk_mask_z = 20.0  # Å
        
        water_z = np.array(list(D_water.keys()))
        water_D = np.array([D_water[z]['D'] for z in water_z])
        bulk_water = np.mean(water_D[np.abs(water_z) > bulk_mask_z]) if np.any(np.abs(water_z) > bulk_mask_z) else np.mean(water_D)
        
        rel_D_water = water_D / bulk_water if bulk_water > 0 else water_D
        
        ax3.plot(water_z, rel_D_water, color='blue', linewidth=2, 
                marker='o', markersize=6, label='Water')
        
        if D_cations:
            cation_z = np.array(list(D_cations.keys()))
            cation_D = np.array([D_cations[z]['D'] for z in cation_z])
            bulk_cation = np.mean(cation_D[np.abs(cation_z) > bulk_mask_z]) if np.any(np.abs(cation_z) > bulk_mask_z) else np.mean(cation_D)
            rel_D_cation = cation_D / bulk_cation if bulk_cation > 0 else cation_D
            ax3.plot(cation_z, rel_D_cation, color='red', linewidth=2,
                    marker='s', markersize=6, label='Cations')
        
        if D_anions:
            anion_z = np.array(list(D_anions.keys()))
            anion_D = np.array([D_anions[z]['D'] for z in anion_z])
            bulk_anion = np.mean(anion_D[np.abs(anion_z) > bulk_mask_z]) if np.any(np.abs(anion_z) > bulk_mask_z) else np.mean(anion_D)
            rel_D_anion = anion_D / bulk_anion if bulk_anion > 0 else anion_D
            ax3.plot(anion_z, rel_D_anion, color='green', linewidth=2,
                    marker='^', markersize=6, label='Anions')
        
        ax3.set_xlabel('z (Å)', fontsize=11)
        ax3.set_ylabel('Relative Diffusion (D/D_bulk)', fontsize=11)
        ax3.set_title('Normalized Diffusion Coefficient', fontweight='bold')
        ax3.axhline(1, color='k', linestyle='--', alpha=0.5, label='Bulk')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 4: Fit quality (R²) vs z
    ax4 = fig.add_subplot(gs[1, 2])
    
    if D_water:
        r_squared_water = [D_water[z]['r_squared'] for z in z_centers_water]
        ax4.plot(z_centers_water, r_squared_water, color='blue', linewidth=2,
                marker='o', markersize=4, label='Water')
    
    if D_cations:
        r_squared_cations = [D_cations[z]['r_squared'] for z in z_centers_cations]
        ax4.plot(z_centers_cations, r_squared_cations, color='red', linewidth=2,
                marker='s', markersize=4, label='Cations')
    
    if D_anions:
        r_squared_anions = [D_anions[z]['r_squared'] for z in z_centers_anions]
        ax4.plot(z_centers_anions, r_squared_anions, color='green', linewidth=2,
                marker='^', markersize=4, label='Anions')
    
    ax4.set_xlabel('z (Å)', fontsize=11)
    ax4.set_ylabel('R² (Linear Fit Quality)', fontsize=11)
    ax4.set_title('MSD Fit Quality', fontweight='bold')
    ax4.axhline(0.95, color='orange', linestyle='--', alpha=0.5, label='Good fit (>0.95)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax4.set_ylim([0, 1.05])
    
    # Plot 5: Regional comparison
    ax5 = fig.add_subplot(gs[2, 0])
    
    regions = {
        'Center': lambda z: np.abs(z) < 5.0,
        'Interface': lambda z: (np.abs(z) >= 5.0) & (np.abs(z) < 15.0),
        'Bulk': lambda z: np.abs(z) >= 15.0
    }
    
    region_names = list(regions.keys())
    water_regional = []
    cation_regional = []
    anion_regional = []
    
    for region_name, region_func in regions.items():
        if D_water:
            water_z = np.array(list(D_water.keys()))
            water_D = np.array([D_water[z]['D'] * 1e5 for z in water_z])
            mask = region_func(water_z)
            water_regional.append(np.mean(water_D[mask]) if np.any(mask) else 0)
        
        if D_cations:
            cation_z = np.array(list(D_cations.keys()))
            cation_D = np.array([D_cations[z]['D'] * 1e5 for z in cation_z])
            mask = region_func(cation_z)
            cation_regional.append(np.mean(cation_D[mask]) if np.any(mask) else 0)
        
        if D_anions:
            anion_z = np.array(list(D_anions.keys()))
            anion_D = np.array([D_anions[z]['D'] * 1e5 for z in anion_z])
            mask = region_func(anion_z)
            anion_regional.append(np.mean(anion_D[mask]) if np.any(mask) else 0)
    
    x_pos = np.arange(len(region_names))
    width = 0.25
    
    if water_regional:
        ax5.bar(x_pos - width, water_regional, width, label='Water', 
               color='blue', alpha=0.7, edgecolor='black')
    if cation_regional:
        ax5.bar(x_pos, cation_regional, width, label='Cations',
               color='red', alpha=0.7, edgecolor='black')
    if anion_regional:
        ax5.bar(x_pos + width, anion_regional, width, label='Anions',
               color='green', alpha=0.7, edgecolor='black')
    
    ax5.set_ylabel('Diffusion Coefficient (×10⁻⁵ cm²/s)', fontsize=11)
    ax5.set_title('Regional Diffusion Comparison', fontweight='bold')
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(region_names)
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Diffusion slowdown factor
    ax6 = fig.add_subplot(gs[2, 1])
    
    if D_water and bulk_water > 0:
        slowdown = []
        for z in z_centers_water:
            D_local = D_water[z]['D']
            slowdown_factor = bulk_water / D_local if D_local > 0 else 0
            slowdown.append(slowdown_factor)
        
        ax6.plot(z_centers_water, slowdown, color='blue', linewidth=2,
                marker='o', markersize=6, label='Water')
    
    ax6.set_xlabel('z (Å)', fontsize=11)
    ax6.set_ylabel('Slowdown Factor (D_bulk / D_local)', fontsize=11)
    ax6.set_title('Diffusion Slowdown vs Z', fontweight='bold')
    ax6.axhline(1, color='k', linestyle='--', alpha=0.5, label='No slowdown')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    ax6.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 7: Residence time proxy (inverse of diffusion)
    ax7 = fig.add_subplot(gs[2, 2])
    
    if D_water:
        residence_proxy = [1.0/D_water[z]['D'] if D_water[z]['D'] > 0 else 0 
                          for z in z_centers_water]
        ax7.plot(z_centers_water, residence_proxy, color='blue', linewidth=2,
                marker='o', markersize=6)
    
    ax7.set_xlabel('z (Å)', fontsize=11)
    ax7.set_ylabel('Residence Time Proxy (1/D)', fontsize=11)
    ax7.set_title('Relative Residence Time', fontweight='bold')
    ax7.grid(True, alpha=0.3)
    ax7.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 8: Overall statistics
    ax8 = fig.add_subplot(gs[3, 0])
    
    stats_text = "Diffusion Coefficient Statistics:\n\n"
    
    if D_water:
        water_D_all = [D_water[z]['D'] * 1e5 for z in D_water.keys()]
        stats_text += f"Water:\n"
        stats_text += f"  Mean: {np.mean(water_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Std:  {np.std(water_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Range: {np.min(water_D_all):.3f} - {np.max(water_D_all):.3f}\n\n"
    
    if D_cations:
        cation_D_all = [D_cations[z]['D'] * 1e5 for z in D_cations.keys()]
        stats_text += f"Cations:\n"
        stats_text += f"  Mean: {np.mean(cation_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Std:  {np.std(cation_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Range: {np.min(cation_D_all):.3f} - {np.max(cation_D_all):.3f}\n\n"
    
    if D_anions:
        anion_D_all = [D_anions[z]['D'] * 1e5 for z in D_anions.keys()]
        stats_text += f"Anions:\n"
        stats_text += f"  Mean: {np.mean(anion_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Std:  {np.std(anion_D_all):.3f} ×10⁻⁵ cm²/s\n"
        stats_text += f"  Range: {np.min(anion_D_all):.3f} - {np.max(anion_D_all):.3f}\n"
    
    ax8.text(0.1, 0.9, stats_text, transform=ax8.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    ax8.axis('off')
    
    # Plot 9: Mobility ratio (cation/anion)
    ax9 = fig.add_subplot(gs[3, 1])
    
    if D_cations and D_anions:
        common_z = sorted(set(D_cations.keys()) & set(D_anions.keys()))
        
        if common_z:
            mobility_ratio = []
            for z in common_z:
                ratio = D_cations[z]['D'] / D_anions[z]['D'] if D_anions[z]['D'] > 0 else 0
                mobility_ratio.append(ratio)
            
            ax9.plot(common_z, mobility_ratio, color='purple', linewidth=2,
                    marker='o', markersize=6)
            ax9.axhline(1, color='k', linestyle='--', alpha=0.5, label='Equal mobility')
            
            ax9.set_xlabel('z (Å)', fontsize=11)
            ax9.set_ylabel('Mobility Ratio (D_cation / D_anion)', fontsize=11)
            ax9.set_title('Relative Ion Mobility', fontweight='bold')
            ax9.legend()
            ax9.grid(True, alpha=0.3)
            ax9.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 10: Summary comparison
    ax10 = fig.add_subplot(gs[3, 2])
    
    species_names = []
    mean_D_values = []
    
    if D_water:
        species_names.append('Water')
        mean_D_values.append(np.mean([D_water[z]['D'] * 1e5 for z in D_water.keys()]))
    if D_cations:
        species_names.append('Cations')
        mean_D_values.append(np.mean([D_cations[z]['D'] * 1e5 for z in D_cations.keys()]))
    if D_anions:
        species_names.append('Anions')
        mean_D_values.append(np.mean([D_anions[z]['D'] * 1e5 for z in D_anions.keys()]))
    
    if species_names:
        bars = ax10.barh(species_names, mean_D_values, 
                        color=['blue', 'red', 'green'][:len(species_names)],
                        alpha=0.7, edgecolor='black')
        
        ax10.set_xlabel('Mean D (×10⁻⁵ cm²/s)', fontsize=11)
        ax10.set_title('Overall Mean Diffusion', fontweight='bold')
        ax10.grid(True, alpha=0.3, axis='x')
        
        # Add value labels
        for bar, value in zip(bars, mean_D_values):
            width = bar.get_width()
            ax10.text(width, bar.get_y() + bar.get_height()/2.,
                    f'{value:.3f}', ha='left', va='center', 
                    fontweight='bold', fontsize=10)
    
    plt.savefig('mobility_diffusion_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    print("  ✓ Comprehensive mobility and diffusion plot saved as: mobility_diffusion_comprehensive_analysis.png")
    plt.show()
    
    # 5. Summary statistics
    print("\n5. Mobility and Diffusion Summary:")
    
    if D_water:
        print(f"\n  Water Diffusion:")
        water_D_values = [D_water[z]['D'] * 1e5 for z in D_water.keys()]
        print(f"    Mean D: {np.mean(water_D_values):.3f} ×10⁻⁵ cm²/s")
        print(f"    Bulk D (literature): ~2.3 ×10⁻⁵ cm²/s at 298K")
        print(f"    Ratio (calculated/literature): {np.mean(water_D_values)/2.3:.2f}")
    
    if D_cations:
        print(f"\n  Cation Diffusion:")
        cation_D_values = [D_cations[z]['D'] * 1e5 for z in D_cations.keys()]
        print(f"    Mean D: {np.mean(cation_D_values):.3f} ×10⁻⁵ cm²/s")
        if D_water:
            print(f"    Relative to water: {np.mean(cation_D_values)/np.mean(water_D_values):.2f}x")
    
    if D_anions:
        print(f"\n  Anion Diffusion:")
        anion_D_values = [D_anions[z]['D'] * 1e5 for z in D_anions.keys()]
        print(f"    Mean D: {np.mean(anion_D_values):.3f} ×10⁻⁵ cm²/s")
        if D_water:
            print(f"    Relative to water: {np.mean(anion_D_values)/np.mean(water_D_values):.2f}x")

else:
    print("\n  ERROR: No trajectory object available for MSD calculation")

print("\n=== Analysis 14 Complete ===")
print("\n🎉 ALL 14 comprehensive analyses are now complete!")
print("\nComplete Analysis Suite:")
print("  1. Basic density profiles")
print("  2. Ion coordination and pairing")
print("  3. Water structure and clustering")
print("  4. Symmetrization analysis")
print("  5. Interface detection")
print("  6. Layer-based analysis")
print("  7. Time-resolved analysis")
print("  8. Advanced structural/dynamic analysis")
print("  9. Electrostatic potential analysis")
print(" 10. Ion pairing analysis")
print(" 11. Ion clustering analysis")
print(" 12. Radial distribution function (RDF) analysis")
print(" 13. Solvation shell analysis")
print(" 14. Mobility and diffusion analysis ✓")

In [ ]:
print("=== ANALYSIS 15: Water Structure and Dipole Orientation Analysis ===")

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# 1. Calculate water structure if not already done
print("\n1. Calculating Water Structure vs Z-Position:")

if hasattr(z_analysis, 'calculate_water_structure_vs_z_detailed'):
    try:
        z_analysis.calculate_water_structure_vs_z_detailed()
        print("  ✓ Water structure calculation complete")
    except Exception as e:
        print(f"  Error during calculation: {e}")
else:
    print("  Note: calculate_water_structure_vs_z_detailed method not found")

# 2. Plot water structure using the plotter
print("\n2. Plotting Water Structure Results:")

if hasattr(z_analysis, 'plotter'):
    z_analysis.plotter.plot_water_structure_analysis(save_plots=True)
else:
    from ZDirectionalPlotter import ZDirectionalPlotter
    plotter = ZDirectionalPlotter(z_analysis)
    plotter.plot_water_structure_analysis(save_plots=True)

# 3. Plot dipole orientation using the plotter
print("\n3. Plotting Dipole Orientation Results:")

if hasattr(z_analysis, 'plotter'):
    z_analysis.plotter.plot_dipole_orientation_vs_z(save_plots=True, bin_size=2.0)
else:
    from ZDirectionalPlotter import ZDirectionalPlotter
    plotter = ZDirectionalPlotter(z_analysis)
    plotter.plot_dipole_orientation_vs_z(save_plots=True, bin_size=2.0)

# 4. Comprehensive water structure and orientation analysis
print("\n4. Creating Comprehensive Water Analysis:")

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

# Check if water structure data is available
if hasattr(z_analysis.results, 'water_structure'):
    struct_data = z_analysis.results.water_structure
    z_centers = struct_data['z_centers']
    
    # Plot 1: Tetrahedral order parameter
    ax1 = fig.add_subplot(gs[0, 0])
    tetrahedral_order = struct_data['tetrahedral_order']
    water_counts = struct_data['water_counts']
    valid_mask = (water_counts > 0) & (tetrahedral_order > 0)
    
    if np.any(valid_mask):
        ax1.plot(z_centers[valid_mask], tetrahedral_order[valid_mask], 
                'b-', linewidth=2, marker='o', markersize=4)
        ax1.axhline(1.0, color='r', linestyle='--', alpha=0.5, label='Perfect tetrahedral')
        ax1.axhline(0.0, color='gray', linestyle='--', alpha=0.3, label='Random')
    
    ax1.set_xlabel('z (Å)', fontsize=11)
    ax1.set_ylabel('Tetrahedral Order (Q)', fontsize=11)
    ax1.set_title('Tetrahedral Order Parameter', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 2: Hydrogen bonds per water
    ax2 = fig.add_subplot(gs[0, 1])
    hbonds = struct_data['hydrogen_bonds_per_water']
    
    if np.any(valid_mask):
        ax2.plot(z_centers[valid_mask], hbonds[valid_mask], 
                'r-', linewidth=2, marker='s', markersize=4)
        ax2.axhline(4.0, color='g', linestyle='--', alpha=0.5, label='Bulk water (~4)')
    
    ax2.set_xlabel('z (Å)', fontsize=11)
    ax2.set_ylabel('H-bonds per Water', fontsize=11)
    ax2.set_title('Hydrogen Bonding Network', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 3: Water coordination number
    ax3 = fig.add_subplot(gs[0, 2])
    coord_num = struct_data['water_coordination_number']
    
    if np.any(valid_mask):
        ax3.plot(z_centers[valid_mask], coord_num[valid_mask], 
                'g-', linewidth=2, marker='^', markersize=4)
    
    ax3.set_xlabel('z (Å)', fontsize=11)
    ax3.set_ylabel('Coordination Number', fontsize=11)
    ax3.set_title('Water-Water Coordination', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 4: Structure correlation (Q vs H-bonds)
    ax4 = fig.add_subplot(gs[1, 0])
    
    if np.any(valid_mask):
        scatter = ax4.scatter(tetrahedral_order[valid_mask], hbonds[valid_mask], 
                c=z_centers[valid_mask], cmap='viridis', s=50, alpha=0.7)
        cbar = plt.colorbar(scatter, ax=ax4)
        cbar.set_label('z (Å)')
    
    ax4.set_xlabel('Tetrahedral Order (Q)', fontsize=11)
    ax4.set_ylabel('H-bonds per Water', fontsize=11)
    ax4.set_title('Structure Correlation', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Regional comparison (structure)
    ax5 = fig.add_subplot(gs[1, 1])
    
    regions = {
        'Center': np.abs(z_centers) < 5.0,
        'Interface': (np.abs(z_centers) >= 5.0) & (np.abs(z_centers) < 15.0),
        'Bulk': np.abs(z_centers) >= 15.0
    }
    
    region_names = []
    q_values = []
    hbond_values = []
    
    for region_name, mask in regions.items():
        combined_mask = mask & valid_mask
        if np.any(combined_mask):
            region_names.append(region_name)
            q_values.append(np.mean(tetrahedral_order[combined_mask]))
            hbond_values.append(np.mean(hbonds[combined_mask]))
    
    x_pos = np.arange(len(region_names))
    width = 0.35
    
    ax5.bar(x_pos - width/2, q_values, width, label='Q parameter', 
           color='blue', alpha=0.7, edgecolor='black')
    ax5_twin = ax5.twinx()
    ax5_twin.bar(x_pos + width/2, hbond_values, width, label='H-bonds', 
                color='red', alpha=0.7, edgecolor='black')
    
    ax5.set_ylabel('Tetrahedral Order', color='blue', fontsize=11)
    ax5_twin.set_ylabel('H-bonds per Water', color='red', fontsize=11)
    ax5.set_title('Regional Structure Comparison', fontweight='bold')
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(region_names)
    ax5.legend(loc='upper left')
    ax5_twin.legend(loc='upper right')
    
    # Plot 6: Structure strength heatmap
    ax6 = fig.add_subplot(gs[1, 2])
    
    # Create structure strength metric
    structure_strength = tetrahedral_order * hbonds / 4.0  # Normalized
    
    if np.any(valid_mask):
        ax6.fill_between(z_centers[valid_mask], 0, structure_strength[valid_mask], 
                        alpha=0.6, color='purple')
        ax6.plot(z_centers[valid_mask], structure_strength[valid_mask], 
                'purple', linewidth=2)
    
    ax6.set_xlabel('z (Å)', fontsize=11)
    ax6.set_ylabel('Structure Strength', fontsize=11)
    ax6.set_title('Combined Structure Metric', fontweight='bold')
    ax6.grid(True, alpha=0.3)
    ax6.axvline(0, color='k', linestyle='--', alpha=0.5)

else:
    for i in range(6):
        row = i // 3
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        ax.text(0.5, 0.5, 'No water structure\ndata available', 
               ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

# Check if dipole orientation data is available
if hasattr(z_analysis.results, 'dipole_vs_z') and z_analysis.results.dipole_vs_z['z_positions']:
    z_pos = np.array(z_analysis.results.dipole_vs_z['z_positions'])
    angles = np.array(z_analysis.results.dipole_vs_z['dipole_angles'])
    
    # Plot 7: Dipole orientation distribution
    ax7 = fig.add_subplot(gs[2, 0])
    
    # Create 2D histogram
    bin_size = 2.0
    z_bins = np.arange(z_analysis.z_min, z_analysis.z_max + bin_size, bin_size)
    angle_bins = np.arange(0, 181, 5)
    
    H, xedges, yedges = np.histogram2d(z_pos, angles, bins=[z_bins, angle_bins])
    
    im = ax7.imshow(H.T, origin='lower', aspect='auto', cmap='hot',
                    extent=[z_bins[0], z_bins[-1], angle_bins[0], angle_bins[-1]])
    ax7.axhline(90, color='cyan', linestyle='--', linewidth=2, alpha=0.7, label='Random (90°)')
    ax7.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.5)
    
    ax7.set_xlabel('z (Å)', fontsize=11)
    ax7.set_ylabel('Dipole Angle (degrees)', fontsize=11)
    ax7.set_title('Dipole Orientation Heatmap', fontweight='bold')
    
    cbar = plt.colorbar(im, ax=ax7)
    cbar.set_label('Count')
    ax7.legend()
    
    # Plot 8: Average dipole angle vs z
    ax8 = fig.add_subplot(gs[2, 1])
    
    z_bin_centers = (z_bins[:-1] + z_bins[1:]) / 2
    binned_angles = []
    binned_stds = []
    
    for i in range(len(z_bins)-1):
        mask = (z_pos >= z_bins[i]) & (z_pos < z_bins[i+1])
        if np.any(mask):
            binned_angles.append(np.mean(angles[mask]))
            binned_stds.append(np.std(angles[mask]))
        else:
            binned_angles.append(np.nan)
            binned_stds.append(np.nan)
    
    binned_angles = np.array(binned_angles)
    binned_stds = np.array(binned_stds)
    valid = ~np.isnan(binned_angles)
    
    ax8.plot(z_bin_centers[valid], binned_angles[valid], 
            'b-', linewidth=2, marker='o', markersize=4)
    ax8.fill_between(z_bin_centers[valid], 
                    binned_angles[valid] - binned_stds[valid],
                    binned_angles[valid] + binned_stds[valid],
                    alpha=0.3, color='blue')
    ax8.axhline(90, color='r', linestyle='--', alpha=0.5, label='Random (90°)')
    
    ax8.set_xlabel('z (Å)', fontsize=11)
    ax8.set_ylabel('Average Dipole Angle (°)', fontsize=11)
    ax8.set_title('Mean Dipole Orientation', fontweight='bold')
    ax8.legend()
    ax8.grid(True, alpha=0.3)
    ax8.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 9: Orientation order parameter
    ax9 = fig.add_subplot(gs[2, 2])
    
    # Calculate order parameter: S = <3cos²θ - 1>/2
    # Perfect alignment: S=1, random: S=0, perfect anti-alignment: S=-0.5
    order_params = []
    
    for i in range(len(z_bins)-1):
        mask = (z_pos >= z_bins[i]) & (z_pos < z_bins[i+1])
        if np.any(mask):
            cos_theta = np.cos(np.radians(angles[mask]))
            order_param = np.mean(1.5 * cos_theta**2 - 0.5)
            order_params.append(order_param)
        else:
            order_params.append(np.nan)
    
    order_params = np.array(order_params)
    valid = ~np.isnan(order_params)
    
    ax9.plot(z_bin_centers[valid], order_params[valid], 
            'g-', linewidth=2, marker='s', markersize=4)
    ax9.axhline(0, color='r', linestyle='--', alpha=0.5, label='Random (S=0)')
    ax9.axhline(1, color='blue', linestyle='--', alpha=0.3, label='Perfect alignment')
    
    ax9.set_xlabel('z (Å)', fontsize=11)
    ax9.set_ylabel('Order Parameter (S)', fontsize=11)
    ax9.set_title('Orientational Order', fontweight='bold')
    ax9.legend()
    ax9.grid(True, alpha=0.3)
    ax9.axvline(0, color='k', linestyle='--', alpha=0.5)
    
    # Plot 10: Regional dipole statistics
    ax10 = fig.add_subplot(gs[3, 0])
    
    regions_dipole = {
        'Center': np.abs(z_pos) < 5.0,
        'Interface': (np.abs(z_pos) >= 5.0) & (np.abs(z_pos) < 15.0),
        'Bulk': np.abs(z_pos) >= 15.0
    }
    
    region_names = []
    mean_angles = []
    
    for region_name, mask in regions_dipole.items():
        if np.any(mask):
            region_names.append(region_name)
            mean_angles.append(np.mean(angles[mask]))
    
    bars = ax10.bar(region_names, mean_angles, 
                   color=['red', 'orange', 'green'][:len(region_names)],
                   alpha=0.7, edgecolor='black')
    ax10.axhline(90, color='k', linestyle='--', alpha=0.5, label='Random')
    
    ax10.set_ylabel('Mean Dipole Angle (°)', fontsize=11)
    ax10.set_title('Regional Orientation', fontweight='bold')
    ax10.legend()
    ax10.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, value in zip(bars, mean_angles):
        height = bar.get_height()
        ax10.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.1f}°', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    # Plot 11: Orientation histogram by region
    ax11 = fig.add_subplot(gs[3, 1])
    
    angle_bins_hist = np.arange(0, 181, 10)
    
    for region_name, mask in regions_dipole.items():
        if np.any(mask):
            hist, _ = np.histogram(angles[mask], bins=angle_bins_hist, density=True)
            bin_centers = (angle_bins_hist[:-1] + angle_bins_hist[1:]) / 2
            ax11.plot(bin_centers, hist, linewidth=2, label=region_name, alpha=0.7)
    
    ax11.axvline(90, color='k', linestyle='--', alpha=0.5, label='Random')
    ax11.set_xlabel('Dipole Angle (°)', fontsize=11)
    ax11.set_ylabel('Probability Density', fontsize=11)
    ax11.set_title('Angle Distribution by Region', fontweight='bold')
    ax11.legend()
    ax11.grid(True, alpha=0.3)

else:
    for i in range(7, 12):
        if i < 10:
            row = i // 3
            col = i % 3
        else:
            row = 3
            col = i - 10
        ax = fig.add_subplot(gs[row, col])
        ax.text(0.5, 0.5, 'No dipole orientation\ndata available', 
               ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

# Plot 12: Combined structure-orientation correlation
ax12 = fig.add_subplot(gs[3, 2])

if (hasattr(z_analysis.results, 'water_structure') and 
    hasattr(z_analysis.results, 'dipole_vs_z') and 
    z_analysis.results.dipole_vs_z['z_positions']):
    
    # Bin dipole data to match structure data
    struct_data = z_analysis.results.water_structure
    z_centers_struct = struct_data['z_centers']
    
    binned_angles_struct = []
    for z_center in z_centers_struct:
        mask = np.abs(z_pos - z_center) < 1.0  # 1Å tolerance
        if np.any(mask):
            binned_angles_struct.append(np.mean(angles[mask]))
        else:
            binned_angles_struct.append(np.nan)
    
    binned_angles_struct = np.array(binned_angles_struct)
    valid_both = ~np.isnan(binned_angles_struct) & (struct_data['water_counts'] > 0)
    
    if np.any(valid_both):
        scatter = ax12.scatter(struct_data['tetrahedral_order'][valid_both], 
                             binned_angles_struct[valid_both],
                             c=z_centers_struct[valid_both], 
                             cmap='viridis', s=50, alpha=0.7)
        cbar = plt.colorbar(scatter, ax=ax12)
        cbar.set_label('z (Å)')
        
        ax12.axhline(90, color='r', linestyle='--', alpha=0.5, label='Random angle')
        ax12.axvline(1.0, color='b', linestyle='--', alpha=0.5, label='Perfect tetrahedral')
    
    ax12.set_xlabel('Tetrahedral Order (Q)', fontsize=11)
    ax12.set_ylabel('Dipole Angle (°)', fontsize=11)
    ax12.set_title('Structure-Orientation Coupling', fontweight='bold')
    ax12.legend()
    ax12.grid(True, alpha=0.3)
else:
    ax12.text(0.5, 0.5, 'Insufficient data for\nstructure-orientation\ncorrelation', 
             ha='center', va='center', transform=ax12.transAxes)
    ax12.axis('off')

plt.savefig('water_structure_orientation_comprehensive.png', dpi=300, bbox_inches='tight')
print("  ✓ Comprehensive water structure and orientation plot saved as: water_structure_orientation_comprehensive.png")
plt.show()

# 5. Statistical summary
print("\n5. Water Structure and Orientation Summary:")

if hasattr(z_analysis.results, 'water_structure'):
    struct_data = z_analysis.results.water_structure
    valid_mask = (struct_data['water_counts'] > 0) & (struct_data['tetrahedral_order'] > 0)
    
    if np.any(valid_mask):
        print(f"\n  Water Structure:")
        print(f"    Mean tetrahedral order: {np.mean(struct_data['tetrahedral_order'][valid_mask]):.3f}")
        print(f"    Mean H-bonds per water: {np.mean(struct_data['hydrogen_bonds_per_water'][valid_mask]):.2f}")
        print(f"    Mean coordination number: {np.mean(struct_data['water_coordination_number'][valid_mask]):.2f}")

if hasattr(z_analysis.results, 'dipole_vs_z') and z_analysis.results.dipole_vs_z['z_positions']:
    angles = np.array(z_analysis.results.dipole_vs_z['dipole_angles'])
    
    print(f"\n  Dipole Orientation:")
    print(f"    Mean dipole angle: {np.mean(angles):.1f}°")
    print(f"    Standard deviation: {np.std(angles):.1f}°")
    print(f"    Median dipole angle: {np.median(angles):.1f}°")
    
    # Calculate overall order parameter
    cos_theta = np.cos(np.radians(angles))
    overall_order = np.mean(1.5 * cos_theta**2 - 0.5)
    print(f"    Overall order parameter: {overall_order:.3f}")
    
    if overall_order > 0.1:
        print(f"      → Significant preferential alignment")
    elif overall_order < -0.1:
        print(f"      → Significant preferential anti-alignment")
    else:
        print(f"      → Nearly random orientation")

# 6. Classification
print("\n6. Water Classification:")

if hasattr(z_analysis.results, 'water_structure'):
    struct_data = z_analysis.results.water_structure
    valid_mask = (struct_data['water_counts'] > 0) & (struct_data['tetrahedral_order'] > 0)
    
    if np.any(valid_mask):
        mean_q = np.mean(struct_data['tetrahedral_order'][valid_mask])
        mean_hbonds = np.mean(struct_data['hydrogen_bonds_per_water'][valid_mask])
        
        print(f"\n  Based on structure metrics:")
        
        # Tetrahedral order classification
        if mean_q > 0.8:
            q_class = "Highly structured (ice-like)"
        elif mean_q > 0.5:
            q_class = "Well structured (liquid water)"
        elif mean_q > 0.3:
            q_class = "Moderately structured"
        else:
            q_class = "Poorly structured (disrupted)"
        print(f"    Tetrahedral order: {q_class}")
        
        # H-bonding classification
        if mean_hbonds > 3.5:
            hbond_class = "Strong H-bond network"
        elif mean_hbonds > 3.0:
            hbond_class = "Normal H-bond network"
        elif mean_hbonds > 2.0:
            hbond_class = "Weakened H-bond network"
        else:
            hbond_class = "Disrupted H-bond network"
        print(f"    H-bonding: {hbond_class}")

print("\n=== Analysis 15 Complete ===")
print("\n🎉 ALL 15 comprehensive analyses are now complete!")
print("\nComplete Analysis Suite:")
print("  1. Basic density profiles")
print("  2. Ion coordination and pairing")
print("  3. Water structure and clustering")
print("  4. Symmetrization analysis")
print("  5. Interface detection")
print("  6. Layer-based analysis")
print("  7. Time-resolved analysis")
print("  8. Advanced structural/dynamic analysis")
print("  9. Electrostatic potential analysis")
print(" 10. Ion pairing analysis")
print(" 11. Ion clustering analysis")
print(" 12. Radial distribution function (RDF) analysis")
print(" 13. Solvation shell analysis")
print(" 14. Mobility and diffusion analysis")
print(" 15. Water structure and dipole orientation analysis ✓")

In [ ]:
z_analysis.plot_dipole_orientation_vs_z()


In [ ]:
z_positions = water_structure['z_centers']
tetrahedral_order = water_structure['tetrahedral_order']
hbonds_per_water = water_structure['hydrogen_bonds_per_water']
coordination_numbers = water_structure['water_coordination_number']

In [ ]:
# Plot water structure parameters
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Tetrahedral order
axes[0,0].plot(z_positions, tetrahedral_order)
axes[0,0].set_title('Tetrahedral Order Parameter')
axes[0,0].set_ylabel('Q parameter')

# Hydrogen bonds
axes[0,1].plot(z_positions, hbonds_per_water)
axes[0,1].set_title('H-bonds per Water')
axes[0,1].set_ylabel('H-bonds')

# Coordination number
axes[1,0].plot(z_positions, coordination_numbers)
axes[1,0].set_title('Water Coordination')
axes[1,0].set_ylabel('Neighbors')

# Water counts
axes[1,1].plot(z_positions, water_structure['water_counts'])
axes[1,1].set_title('Water Count per Layer')
axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
z_analysis.export_results('my_z_analysis.pkl')